# Agent 1 — Module 3: Topic Extraction, Syllabus Mapping, Memory-First Resolution, and Human Approval

This notebook keeps the existing Module 3 mapping logic in-notebook while using the shared **SyllabusStore** as the only syllabus storage/lookup gateway.

It does **not** rerun Module 1 or Module 2 and does **not** recreate their PDFs.
It reads the structured chunk output produced by Module 2:

```text
Agent_1/OUTPUT/<transcript>/02_chunking.json
```

For each of the 12 production transcript folders, it writes only these three files:

```text
Agent_1/OUTPUT/<transcript>/03_topics_readable.pdf
Agent_1/OUTPUT/<transcript>/04_llm_mapping.pdf
Agent_1/OUTPUT/<transcript>/05_final_topic_summary.pdf
```

## Final production flow

```text
02_chunking.json
    -> PostgreSQL SyllabusStore + MiniLM/Qdrant official AQA candidate retrieval
    -> lexical/context/salience/evidence-quality rules
    -> continuation and non-CS handling
    -> existing topic merging and primary/supporting assignment
    -> every genuine unmapped-CS signal enters the resolver
    -> PostgreSQL topic_mapping_memory lookup
    -> memory miss: Qdrant shortlist + one batched Groq call
    -> proposal saved to topic_human_review as pending
    -> user approval triggers promotion to topic_mapping_memory
    -> future matching runs return a PostgreSQL memory hit
    -> three Module 3/4 PDF reports
```

## Restored routing rule

All genuine unmapped-CS signals use the memory-first resolver. This routing
change does not alter official-topic extraction, confidence scoring, evidence
quality, merging, or primary/supporting role assignment.


## 0. Install notebook dependencies


In [ ]:
%pip install -q numpy pandas pydantic sentence-transformers qdrant-client groq python-dotenv reportlab "SQLAlchemy>=2.0" "psycopg[binary]>=3.1"


## 1. Environment and notebook configuration


In [ ]:
from __future__ import annotations

import csv
import hashlib
import html
import json
import os
import re
import sys
import time
import unicodedata
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Literal

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from groq import Groq


def find_project_root(start: Path | None = None) -> Path:
    """Find the Agent_1 folder from the notebook or current working folder."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "OUTPUT").is_dir() and (
            (candidate / "Notebooks").exists()
            or (candidate / "Test Data").exists()
            or (candidate / ".env").exists()
        ):
            return candidate

    return start


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
load_dotenv(PROJECT_ROOT / ".env")

# The normal user run keeps both flags True. During offline notebook validation,
# set MODULE3_SKIP_EXTERNAL_SERVICES=1 before starting the kernel.
SKIP_EXTERNAL_SERVICES = os.getenv(
    "MODULE3_SKIP_EXTERNAL_SERVICES", "0"
).strip() == "1"

RUN_QDRANT_SETUP = not SKIP_EXTERNAL_SERVICES
RUN_MODULE3_BATCH = not SKIP_EXTERNAL_SERVICES
RUN_REGRESSION_TESTS = True

AUTO_INDEX_QDRANT = True
RECREATE_QDRANT = False
EXPECTED_TRANSCRIPT_COUNT = 13
EXCLUDED_TRANSCRIPT_FOLDERS = {"testing"}
OVERWRITE_MODULE3_PDFS = True

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)
print("Python:", sys.executable)
print("Skip external services:", SKIP_EXTERNAL_SERVICES)

# -----------------------------------------------------------------
# Optional Streamlit frontend single-file mode.
#
# The original notebook configuration and batch behaviour above remain
# unchanged when AGENT1_FRONTEND_MODE is not set to "1".
# -----------------------------------------------------------------
FRONTEND_SINGLE_FILE_MODE = (
    os.getenv("AGENT1_FRONTEND_MODE", "0").strip() == "1"
)
FRONTEND_TRANSCRIPT_NAME = os.getenv(
    "AGENT1_TRANSCRIPT_NAME", ""
).strip()
FRONTEND_OUTPUT_ROOT_RAW = os.getenv(
    "AGENT1_OUTPUT_ROOT", ""
).strip()

if FRONTEND_SINGLE_FILE_MODE:
    if not FRONTEND_TRANSCRIPT_NAME:
        raise ValueError(
            "AGENT1_TRANSCRIPT_NAME is required in frontend mode."
        )
    if not FRONTEND_OUTPUT_ROOT_RAW:
        raise ValueError(
            "AGENT1_OUTPUT_ROOT is required in frontend mode."
        )

    # Module 1 and Module 2 write into this per-job output root.
    OUTPUT_DIR = Path(FRONTEND_OUTPUT_ROOT_RAW).expanduser().resolve()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # The frontend sends exactly one transcript through the pipeline.
    EXPECTED_TRANSCRIPT_COUNT = 1
    EXCLUDED_TRANSCRIPT_FOLDERS = set()

    # Regression tests are development checks and need not run on every
    # frontend upload. This does not alter production topic-mapping logic.
    RUN_REGRESSION_TESTS = False

print({
    "frontend_single_file_mode": FRONTEND_SINGLE_FILE_MODE,
    "frontend_transcript_name": FRONTEND_TRANSCRIPT_NAME or None,
    "effective_output_dir": str(OUTPUT_DIR),
    "effective_expected_count": EXPECTED_TRANSCRIPT_COUNT,
})


## 2. Module 3 schemas


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field

ExtractionMethod = Literal["keyword", "embedding", "keyword_embedding"]
ChunkClassification = Literal["official_aqa_topic", "mixed_official_and_unmapped", "cs_related_unmapped", "continuation_no_new_topic", "no_topic"]
UnmappedDetectionMethod = Literal["lexical", "semantic", "lexical_semantic"]
TopicRole = Literal["primary", "supporting"]
TeachingDepthLabel = Literal["mention_only", "definition", "explanation", "worked_example", "sustained_teaching"]

class RawTopicCandidate(BaseModel):
    concept_id: str = Field(min_length=1)
    topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)
    official_reference: str = Field(min_length=1)
    chapter_reference: str = Field(min_length=1)
    official_title: str = Field(min_length=1)
    paper: str = Field(min_length=1)
    source_pages: list[int] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    keyword_score: float = Field(ge=0.0, le=1.0)
    semantic_score: float = Field(ge=-1.0, le=1.0)
    salience_score: float = Field(ge=0.0, le=1.0)
    extraction_method: ExtractionMethod
    matched_aliases: list[str] = Field(default_factory=list)
    total_alias_hits: int = Field(default=0, ge=0)
    evidence_sentence_count: int = Field(default=0, ge=0)
    single_word_alias_only: bool = False
    ambiguous_alias_only: bool = False
    matched_context_terms: list[str] = Field(default_factory=list)
    matched_conflicting_context_terms: list[str] = Field(default_factory=list)
    minimum_context_hits: int = Field(default=1, ge=1)
    context_collision: bool = False
    teaching_depth_level: int = Field(default=0, ge=0, le=4)
    teaching_depth_label: TeachingDepthLabel = "mention_only"
    evidence_quality_score: float = Field(default=0.0, ge=0.0, le=1.0)
    shared_evidence_penalty: float = Field(default=0.0, ge=0.0, le=1.0)
    evidence_quality_notes: list[str] = Field(default_factory=list)
    recap_evidence_only: bool = False
    recap_evidence_count: int = Field(default=0, ge=0)
    substantive_evidence_count: int = Field(default=0, ge=0)
    comparison_evidence_only: bool = False
    comparison_evidence_count: int = Field(default=0, ge=0)
    independent_evidence_count: int = Field(default=0, ge=0)
    evidence: list[str] = Field(default_factory=list)
    parent_concept_id: str | None = None

class TopicCandidate(RawTopicCandidate):
    cs_relevance_score: float = Field(ge=0.0, le=1.0)
    cs_relevant: bool

class UnmappedCSSignal(BaseModel):
    rough_topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)
    score: float = Field(ge=-1.0, le=1.0)
    evidence: str = Field(min_length=1)
    matched_aliases: list[str] = Field(default_factory=list)
    detection_method: UnmappedDetectionMethod

class ChunkTopicResult(BaseModel):
    chunk_id: int = Field(ge=1)
    source_word_count: int = Field(default=0, ge=0)
    classification: ChunkClassification = "no_topic"
    is_cs_relevant: bool
    creates_new_topic: bool = False
    cs_relevance_score: float = Field(ge=0.0, le=1.0)
    topic_candidates: list[TopicCandidate] = Field(default_factory=list)
    rejected_candidates: list[TopicCandidate] = Field(default_factory=list)
    has_unmapped_cs_content: bool = False
    unmapped_cs_signals: list[UnmappedCSSignal] = Field(default_factory=list)
    continuation_of_chunk_id: int | None = Field(default=None, ge=1)
    requires_llm_fallback: bool = False
    notes: list[str] = Field(default_factory=list)

class MergedTopic(BaseModel):
    concept_id: str = Field(min_length=1)
    topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)
    official_reference: str = Field(min_length=1)
    chapter_reference: str = Field(min_length=1)
    official_title: str = Field(min_length=1)
    paper: str = Field(min_length=1)
    source_pages: list[int] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    ranking_score: float = Field(ge=0.0, le=1.0)
    topic_role: TopicRole
    source_chunk_ids: list[int] = Field(default_factory=list)
    support_span_count: int = Field(default=1, ge=1)
    mean_semantic_score: float = Field(ge=-1.0, le=1.0)
    mean_keyword_score: float = Field(ge=0.0, le=1.0)
    mean_salience_score: float = Field(ge=0.0, le=1.0)
    coverage_score: float = Field(ge=0.0, le=1.0)
    evidence: list[str] = Field(default_factory=list)
    supporting_candidate_count: int = Field(ge=1)

class Module3Result(BaseModel):
    chunk_results: list[ChunkTopicResult]
    merged_topics: list[MergedTopic]
    total_chunks: int = Field(ge=0)
    cs_relevant_chunks: int = Field(ge=0)
    non_cs_chunks: int = Field(ge=0)
    official_topic_chunks: int = Field(default=0, ge=0)
    mixed_official_unmapped_chunks: int = Field(default=0, ge=0)
    unmapped_cs_chunks: int = Field(default=0, ge=0)
    continuation_chunks: int = Field(default=0, ge=0)
    no_topic_chunks: int = Field(default=0, ge=0)
    llm_fallback_chunk_ids: list[int] = Field(default_factory=list)
    embedding_model: str
    candidate_keep_threshold: float = Field(ge=0.0, le=1.0)


## 3. Official AQA syllabus via SyllabusStore


In [ ]:
# The syllabus is no longer embedded in this notebook.
# PostgreSQL is authoritative and Qdrant is the derived semantic index.
# SyllabusStore is the single gateway for both.

AGENT1_CODE_ROOT: Path | None = None

for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "app" / "services" / "syllabus_store.py").is_file():
        AGENT1_CODE_ROOT = candidate
        break

if AGENT1_CODE_ROOT is None:
    raise RuntimeError(
        "Could not locate Agent_1/app/services/syllabus_store.py "
        "from the current Module 3 project path."
    )

if str(AGENT1_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT1_CODE_ROOT))

from app.services.syllabus_store import (
    FlexiblePattern,
    SemanticConceptMatch,
    SyllabusConcept,
    SyllabusStore,
)

syllabus_store = SyllabusStore()
syllabus_concepts = tuple(syllabus_store.get_all_concepts())

if not syllabus_concepts:
    raise RuntimeError(
        "PostgreSQL returned zero active syllabus concepts."
    )

concept_ids = [concept.concept_id for concept in syllabus_concepts]
if len(concept_ids) != len(set(concept_ids)):
    raise RuntimeError(
        "PostgreSQL returned duplicate active syllabus concept IDs."
    )

missing_parents = {
    concept.parent_concept_id
    for concept in syllabus_concepts
    if (
        concept.parent_concept_id is not None
        and concept.parent_concept_id not in set(concept_ids)
    )
}

if missing_parents:
    raise RuntimeError(
        "PostgreSQL syllabus concepts reference missing parent concepts: "
        + ", ".join(sorted(missing_parents))
    )

print("Syllabus source: PostgreSQL via SyllabusStore")
print("Active syllabus concepts:", len(syllabus_concepts))
print("Agent 1 code root:", AGENT1_CODE_ROOT)


## 4. Embedding service and retained model decision


In [ ]:
import os
from collections.abc import Sequence
from functools import lru_cache

import numpy as np
from sentence_transformers import SentenceTransformer


# Module 2 and Module 3 both use MiniLM.
DEFAULT_EMBEDDING_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

CHUNKING_EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
TOPIC_EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL


@lru_cache(maxsize=4)
def get_embedding_model(
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> SentenceTransformer:
    """
    Load and cache a SentenceTransformer model.

    The same MiniLM model is currently used for:
    - Module 2 semantic chunking
    - Module 3 syllabus topic retrieval
    - Qdrant syllabus indexing

    The model is cached so it is loaded only once per Python process.
    """

    device = os.getenv("EMBEDDING_DEVICE")

    kwargs: dict[str, str] = {}

    if device:
        kwargs["device"] = device

    return SentenceTransformer(
        model_name,
        **kwargs,
    )


def embed_texts(
    texts: Sequence[str],
    model_name: str = DEFAULT_EMBEDDING_MODEL,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Generate normalized embeddings for multiple text values.

    Normalized vectors allow cosine similarity to be represented by
    a dot product. Qdrant also stores these vectors using cosine distance.
    """

    cleaned_texts = [
        str(text).strip()
        for text in texts
        if str(text).strip()
    ]

    if not cleaned_texts:
        return np.empty(
            (0, 0),
            dtype=np.float32,
        )

    if batch_size < 1:
        raise ValueError(
            "batch_size must be at least 1."
        )

    model = get_embedding_model(
        model_name
    )

    embeddings = model.encode(
        cleaned_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    return np.asarray(
        embeddings,
        dtype=np.float32,
    )


def get_embedding_dimension(
    model_name: str = DEFAULT_EMBEDDING_MODEL,
) -> int:
    """
    Return the output vector dimension of an embedding model.

    MiniLM-L6-v2 currently produces 384-dimensional embeddings.
    The value is obtained from the loaded model rather than hardcoded.
    """

    model = get_embedding_model(
        model_name
    )

    dimension = (
        model.get_sentence_embedding_dimension()
    )

    if dimension is None or dimension < 1:
        raise RuntimeError(
            "Could not determine embedding dimension for "
            f"{model_name}."
        )

    return int(dimension)

# Retained only for optional historical comparison.
QWEN_EXPERIMENT_MODEL = "Qwen/Qwen3-Embedding-0.6B"

print("Production topic model:", TOPIC_EMBEDDING_MODEL)
print("Optional discarded comparison model:", QWEN_EXPERIMENT_MODEL)


### Optional historical Qwen versus MiniLM comparison


In [ ]:
RUN_MODEL_COMPARISON = False
RUN_QWEN_COMPARISON = False

MODEL_COMPARISON_TEXTS = [
    "Trace tables follow a program line by line and record changing variable values.",
    "Bubble sort compares adjacent elements and swaps them over repeated passes.",
    "Binary values can be converted into decimal and hexadecimal.",
]

if RUN_MODEL_COMPARISON:
    model_comparison_rows = []
    models_to_test = [("MiniLM", TOPIC_EMBEDDING_MODEL)]

    if RUN_QWEN_COMPARISON:
        models_to_test.append(("Qwen", QWEN_EXPERIMENT_MODEL))

    for label, model_name in models_to_test:
        started = time.perf_counter()
        vectors = embed_texts(MODEL_COMPARISON_TEXTS, model_name, 8)
        elapsed = time.perf_counter() - started
        model_comparison_rows.append({
            "model": label,
            "model_name": model_name,
            "texts": len(MODEL_COMPARISON_TEXTS),
            "dimension": int(vectors.shape[1]) if vectors.ndim == 2 and vectors.size else 0,
            "seconds": round(elapsed, 4),
        })

    model_comparison_df = pd.DataFrame(model_comparison_rows)
    display(model_comparison_df)
else:
    print("Optional MiniLM/Qwen comparison skipped. Set RUN_MODEL_COMPARISON=True to run it.")

print("Final production decision: MiniLM is used for Module 3.")


## 5. SyllabusStore-backed Qdrant access


In [ ]:
# Qdrant access is provided by the shared SyllabusStore loaded above.
# The previous embedded Qdrant implementation has been removed.
# No topic-extraction/scoring logic is changed in this section.

print(
    "Qdrant gateway:",
    f"SyllabusStore -> {syllabus_store.config.qdrant_collection}",
)


### Prepare and validate PostgreSQL → Qdrant syllabus sync


In [ ]:
qdrant_store: SyllabusStore | None = None

if RUN_QDRANT_SETUP:
    qdrant_store = syllabus_store
    collection_name = qdrant_store.config.qdrant_collection

    postgres_count = qdrant_store.count_concepts()
    collection_exists = qdrant_store.qdrant_collection_exists()
    collection_count = qdrant_store.count_qdrant_points()

    needs_index = (
        not collection_exists
        or collection_count != postgres_count
    )

    if needs_index:
        if not AUTO_INDEX_QDRANT:
            raise RuntimeError(
                "The Qdrant syllabus collection is missing or stale. "
                "Set AUTO_INDEX_QDRANT=True or sync it manually."
            )

        indexed = qdrant_store.sync_qdrant(
            recreate=(RECREATE_QDRANT or collection_exists),
            batch_size=32,
        )
        collection_count = indexed

    print("Qdrant collection:", collection_name)
    print("Stored Qdrant syllabus concepts:", collection_count)
    print("PostgreSQL syllabus concepts:", postgres_count)

    if collection_count != postgres_count:
        raise RuntimeError(
            "PostgreSQL and Qdrant syllabus counts still do not match."
        )
else:
    print("Qdrant setup skipped for offline notebook validation.")


## 6. Official topic candidate extractor


In [ ]:
import os
import re
from collections.abc import Callable, Sequence
from dataclasses import dataclass

import numpy as np



EmbeddingFunction = Callable[[Sequence[str], str, int], np.ndarray]


@dataclass(frozen=True)
class TopicExtractionConfig:
    """
    Configuration for official AQA topic candidate extraction.

    The salience rules are generic. They do not contain transcript-specific
    words. Their purpose is to stop an isolated ordinary word such as
    "integer", "function" or "bit" from becoming a lesson topic unless the
    surrounding semantic or repeated evidence supports it.
    """

    semantic_unit_words: int = 60

    raw_candidate_floor: float = 0.30
    semantic_only_threshold: float = 0.50

    # A topic supported only by one-word aliases requires stronger context.
    single_word_semantic_floor: float = 0.45
    single_word_min_evidence_sentences: int = 2
    single_word_min_distinct_aliases: int = 2

    # Ambiguous aliases require contextual confirmation. This prevents a
    # valid but overloaded word such as "binary" from mapping to number
    # bases when the phrase actually refers to binary search.
    ambiguous_alias_semantic_floor: float = 0.50
    ambiguous_alias_min_context_terms: int = 1

    max_raw_candidates: int = 12
    max_final_candidates: int = 6
    max_evidence_per_candidate: int = 3

    suppress_redundant_parents: bool = True
    parent_suppression_margin: float = 0.08

    # Suppress two labels that are driven by effectively the same sentence
    # and the same technical phrase.
    duplicate_evidence_overlap: float = 0.80
    duplicate_alias_token_overlap: float = 0.50

    # Module 3 uses the same MiniLM model used when indexing Qdrant.
    embedding_model: str = TOPIC_EMBEDDING_MODEL


    qdrant_top_k_per_unit: int = int(
        os.getenv(
            "QDRANT_TOP_K_PER_UNIT",
            "20",
        )
    )

    def __post_init__(self) -> None:
        if self.semantic_unit_words <= 0:
            raise ValueError("semantic_unit_words must be positive.")

        for value_name in (
            "raw_candidate_floor",
            "semantic_only_threshold",
            "single_word_semantic_floor",
            "ambiguous_alias_semantic_floor",
            "parent_suppression_margin",
            "duplicate_evidence_overlap",
            "duplicate_alias_token_overlap",
        ):
            value = getattr(self, value_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{value_name} must be between 0 and 1.")

        if self.max_raw_candidates < 1 or self.max_final_candidates < 1:
            raise ValueError("Candidate limits must be at least 1.")

        if self.ambiguous_alias_min_context_terms < 1:
            raise ValueError(
                "ambiguous_alias_min_context_terms must be at least 1."
            )


        if self.qdrant_top_k_per_unit < 1:
            raise ValueError(
                "qdrant_top_k_per_unit must be at least 1."
            )


@dataclass(frozen=True)
class KeywordEvidence:
    score: float
    matched_aliases: list[str]
    evidence_sentences: list[str]
    total_hits: int
    excluded_hits: int
    single_word_alias_only: bool
    ambiguous_alias_only: bool
    matched_context_terms: list[str]
    matched_conflicting_context_terms: list[str]
    minimum_context_hits: int


class TopicCandidateExtractor:
    """
    Extract official AQA topic candidates from one transcript chunk.

    Evidence sources:
    - exact transcript-friendly catalogue aliases
    - Qdrant semantic retrieval using MiniLM embeddings
    - evidence repetition/diversity (topic salience)
    """

    def __init__(
        self,
        config: TopicExtractionConfig | None = None,
        embedding_function: EmbeddingFunction = embed_texts,
        qdrant_store: SyllabusStore | None = None,
    ) -> None:
        self.config = config or TopicExtractionConfig()
        self._embedding_function = embedding_function
        self._qdrant_store = (
            qdrant_store
            if qdrant_store is not None
            else syllabus_store
        )

    def extract(
        self,
        chunk_id: int,
        text: str,
    ) -> list[RawTopicCandidate]:
        if chunk_id < 1:
            raise ValueError("chunk_id must be at least 1.")

        if not isinstance(text, str):
            raise TypeError("text must be a string.")

        text = text.strip()
        if not text:
            return []

        sentences = self._split_sentences(text)
        semantic_units = self._build_semantic_units(sentences) or [text]

        unit_embeddings = self._embedding_function(
            semantic_units,
            self.config.embedding_model,
            32,
        )

        if unit_embeddings.size == 0:
            return []

        semantic_scores = self._semantic_scores(
            unit_embeddings=unit_embeddings,
            sentences=sentences,
            full_text=text,
        )

        raw_candidates: list[RawTopicCandidate] = []

        for concept in syllabus_concepts:
            semantic_result = semantic_scores.get(concept.concept_id)

            if semantic_result is None:
                semantic_score = 0.0
                best_unit_index = 0
            else:
                semantic_score, best_unit_index = semantic_result

            keyword = self._keyword_evidence(
                concept=concept,
                sentences=sentences,
                full_text=text,
            )

            has_keyword_support = keyword.score > 0.0
            # A known longer compound term can contain a shorter alias.
            # When every lexical hit was blocked by catalogue exclusions,
            # semantic similarity alone must not remap that compound back to
            # the shorter official concept.
            has_semantic_only_support = (
                semantic_score >= self.config.semantic_only_threshold
                and keyword.excluded_hits == 0
            )

            if not (has_keyword_support or has_semantic_only_support):
                continue

            if has_keyword_support and not self._passes_salience_gate(
                keyword=keyword,
                semantic_score=semantic_score,
            ):
                continue

            salience_score = self._calculate_salience_score(
                keyword=keyword,
                semantic_score=semantic_score,
            )

            confidence = self._calculate_confidence(
                keyword_score=keyword.score,
                semantic_score=semantic_score,
                single_word_alias_only=keyword.single_word_alias_only,
                salience_score=salience_score,
            )

            if confidence < self.config.raw_candidate_floor:
                continue

            semantic_evidence = semantic_units[best_unit_index].strip()
            evidence = (
                keyword.evidence_sentences
                if keyword.evidence_sentences
                else [semantic_evidence]
            )
            evidence = self._unique_strings(evidence)[
                : self.config.max_evidence_per_candidate
            ]

            if keyword.score > 0.0 and semantic_score >= 0.20:
                method = "keyword_embedding"
            elif keyword.score > 0.0:
                method = "keyword"
            else:
                method = "embedding"

            raw_candidates.append(
                RawTopicCandidate(
                    concept_id=concept.concept_id,
                    topic=concept.label,
                    domain=concept.domain,
                    official_reference=concept.official_reference,
                    chapter_reference=concept.chapter_reference,
                    official_title=concept.official_title,
                    paper=concept.paper,
                    source_pages=list(concept.source_pages),
                    confidence=round(confidence, 4),
                    keyword_score=round(keyword.score, 4),
                    semantic_score=round(semantic_score, 4),
                    salience_score=round(salience_score, 4),
                    extraction_method=method,
                    matched_aliases=keyword.matched_aliases,
                    total_alias_hits=keyword.total_hits,
                    evidence_sentence_count=len(keyword.evidence_sentences),
                    single_word_alias_only=keyword.single_word_alias_only,
                    ambiguous_alias_only=keyword.ambiguous_alias_only,
                    matched_context_terms=keyword.matched_context_terms,
                    matched_conflicting_context_terms=(
                        keyword.matched_conflicting_context_terms
                    ),
                    minimum_context_hits=keyword.minimum_context_hits,
                    evidence=evidence,
                    parent_concept_id=concept.parent_concept_id,
                )
            )

        raw_candidates.sort(
            key=lambda candidate: (
                candidate.confidence,
                candidate.salience_score,
                candidate.semantic_score,
                candidate.keyword_score,
            ),
            reverse=True,
        )

        raw_candidates = raw_candidates[: self.config.max_raw_candidates]

        if self.config.suppress_redundant_parents:
            raw_candidates = self._suppress_redundant_candidates(
                raw_candidates
            )

        return raw_candidates[: self.config.max_final_candidates]

    def _semantic_scores(
        self,
        unit_embeddings: np.ndarray,
        sentences: list[str],
        full_text: str,
    ) -> dict[str, tuple[float, int]]:
        """
        Retrieve the strongest semantic score and semantic-unit index for each
        candidate concept using Qdrant.

        Qdrant performs server-side nearest-neighbour search for every
        semantic unit. Exact lexical candidates that fall outside the semantic
        top-k are still scored by retrieving only their stored vectors from
        Qdrant. The full syllabus vector matrix is never loaded into memory.
        """

        result_sets = self._qdrant_store.search_by_vectors(
            unit_embeddings,
            top_k=self.config.qdrant_top_k_per_unit,
        )

        scores: dict[str, tuple[float, int]] = {}

        for unit_index, matches in enumerate(result_sets):
            for match in matches:
                previous = scores.get(match.concept_id)

                if previous is None or match.score > previous[0]:
                    scores[match.concept_id] = (
                        float(match.score),
                        unit_index,
                    )

        # Lexical matching still scans catalogue metadata, not catalogue
        # vectors. When a lexical candidate is outside Qdrant's top-k, fetch
        # only that concept's stored vector so the existing confidence formula
        # receives a semantic score.
        lexical_concept_ids: list[str] = []

        for concept in syllabus_concepts:
            if concept.concept_id in scores:
                continue

            keyword = self._keyword_evidence(
                concept=concept,
                sentences=sentences,
                full_text=full_text,
            )

            if keyword.score > 0.0:
                lexical_concept_ids.append(concept.concept_id)

        stored_vectors = self._qdrant_store.retrieve_concept_vectors(
            lexical_concept_ids
        )

        for concept_id, concept_vector in stored_vectors.items():
            similarities = unit_embeddings @ concept_vector
            best_unit_index = int(np.argmax(similarities))

            scores[concept_id] = (
                float(similarities[best_unit_index]),
                best_unit_index,
            )

        return scores

    @staticmethod
    def _split_sentences(text: str) -> list[str]:
        text = re.sub(r"\s+", " ", text).strip()
        parts = re.split(r"(?<=[.!?])\s+", text)
        return [part.strip() for part in parts if part.strip()]

    def _build_semantic_units(self, sentences: list[str]) -> list[str]:
        units: list[str] = []
        buffer: list[str] = []
        buffer_words = 0

        for sentence in sentences:
            buffer.append(sentence)
            buffer_words += self._word_count(sentence)

            if buffer_words >= self.config.semantic_unit_words:
                units.append(" ".join(buffer))
                buffer = []
                buffer_words = 0

        if buffer:
            units.append(" ".join(buffer))

        return units

    def _keyword_evidence(
        self,
        concept: SyllabusConcept,
        sentences: list[str],
        full_text: str,
    ) -> KeywordEvidence:
        matched_aliases: list[str] = []
        evidence: list[str] = []
        alias_weights: list[float] = []
        total_hits = 0
        excluded_hits = 0
        matched_word_counts: list[int] = []

        for alias in concept.aliases:
            normalized_alias = self._normalize_for_match(alias)
            if not normalized_alias:
                continue

            pattern = re.compile(
                r"(?<!\w)"
                + re.escape(normalized_alias).replace(r"\ ", r"\s+")
                + r"(?!\w)",
                re.IGNORECASE,
            )

            valid_alias_hits = 0

            for sentence in sentences:
                normalized_sentence = self._normalize_for_match(sentence)
                blocked_ranges = self._excluded_ranges(
                    text=normalized_sentence,
                    excluded_phrases=concept.excluded_phrases,
                )

                sentence_valid_hits = 0

                for match in pattern.finditer(normalized_sentence):
                    if self._overlaps_any(match.span(), blocked_ranges):
                        excluded_hits += 1
                        continue

                    sentence_valid_hits += 1

                if sentence_valid_hits:
                    valid_alias_hits += sentence_valid_hits
                    evidence.append(sentence.strip())

            if not valid_alias_hits:
                continue

            total_hits += valid_alias_hits
            matched_aliases.append(alias)

            alias_word_count = len(normalized_alias.split())
            matched_word_counts.append(alias_word_count)

            if alias_word_count == 1:
                weight = 0.50
            elif alias_word_count == 2:
                weight = 0.72
            else:
                weight = 0.82

            alias_weights.append(weight)

        # Flexible patterns support natural classroom phrasing such as
        # passive voice, inserted variable names and ASR variation. The
        # mechanism is catalogue-driven and can be reused by any concept.
        for flexible_pattern in concept.match_patterns:
            pattern = re.compile(
                flexible_pattern.regex,
                re.IGNORECASE,
            )

            pattern_hits = 0

            for sentence in sentences:
                normalized_sentence = self._normalize_for_match(sentence)
                blocked_ranges = self._excluded_ranges(
                    text=normalized_sentence,
                    excluded_phrases=concept.excluded_phrases,
                )

                sentence_hits = 0

                for match in pattern.finditer(normalized_sentence):
                    if self._overlaps_any(match.span(), blocked_ranges):
                        excluded_hits += 1
                        continue

                    sentence_hits += 1

                if sentence_hits:
                    pattern_hits += sentence_hits
                    evidence.append(sentence.strip())

            if not pattern_hits:
                continue

            total_hits += pattern_hits
            matched_aliases.append(flexible_pattern.label)
            matched_word_counts.append(
                max(2, len(flexible_pattern.label.split()))
            )
            alias_weights.append(flexible_pattern.weight)

        if not alias_weights:
            return KeywordEvidence(
                score=0.0,
                matched_aliases=[],
                evidence_sentences=[],
                total_hits=0,
                excluded_hits=excluded_hits,
                single_word_alias_only=False,
                ambiguous_alias_only=False,
                matched_context_terms=[],
                matched_conflicting_context_terms=[],
                minimum_context_hits=concept.minimum_context_hits,
            )

        evidence = self._unique_strings(evidence)
        matched_aliases = self._unique_strings(matched_aliases)

        normalized_ambiguous_aliases = {
            self._normalize_for_match(alias)
            for alias in concept.ambiguous_aliases
            if self._normalize_for_match(alias)
        }

        normalized_matched_aliases = {
            self._normalize_for_match(alias)
            for alias in matched_aliases
            if self._normalize_for_match(alias)
        }

        ambiguous_alias_only = bool(
            normalized_matched_aliases
            and normalized_matched_aliases.issubset(
                normalized_ambiguous_aliases
            )
        )

        matched_context_terms = self._matched_context_terms(
            full_text=full_text,
            context_terms=concept.supporting_context_terms,
            excluded_phrases=concept.excluded_phrases,
        )
        matched_conflicting_context_terms = self._matched_context_terms(
            full_text=full_text,
            context_terms=concept.conflicting_context_terms,
            excluded_phrases=concept.excluded_phrases,
        )

        strongest_alias = max(alias_weights)
        distinct_bonus = 0.08 * max(0, len(matched_aliases) - 1)
        evidence_bonus = 0.05 * min(3, max(0, len(evidence) - 1))
        repetition_bonus = 0.02 * min(4, max(0, total_hits - 1))

        keyword_score = min(
            1.0,
            strongest_alias
            + distinct_bonus
            + evidence_bonus
            + repetition_bonus,
        )

        return KeywordEvidence(
            score=keyword_score,
            matched_aliases=matched_aliases,
            evidence_sentences=evidence,
            total_hits=total_hits,
            excluded_hits=excluded_hits,
            single_word_alias_only=all(
                word_count == 1 for word_count in matched_word_counts
            ),
            ambiguous_alias_only=ambiguous_alias_only,
            matched_context_terms=matched_context_terms,
            matched_conflicting_context_terms=(
                matched_conflicting_context_terms
            ),
            minimum_context_hits=concept.minimum_context_hits,
        )

    @classmethod
    def _matched_context_terms(
        cls,
        full_text: str,
        context_terms: tuple[str, ...],
        excluded_phrases: tuple[str, ...],
    ) -> list[str]:
        """
        Return catalogue-provided context terms found outside blocked phrases.

        A context term cannot confirm an ambiguous alias when its occurrence
        exists only inside a known confusable compound phrase.
        """

        normalized_text = cls._normalize_for_match(full_text)
        blocked_ranges = cls._excluded_ranges(
            text=normalized_text,
            excluded_phrases=excluded_phrases,
        )

        matched: list[str] = []

        for term in context_terms:
            normalized_term = cls._normalize_for_match(term)
            if not normalized_term:
                continue

            pattern = re.compile(
                r"(?<!\w)"
                + re.escape(normalized_term).replace(r"\ ", r"\s+")
                + r"(?!\w)",
                re.IGNORECASE,
            )

            if any(
                not cls._overlaps_any(match.span(), blocked_ranges)
                for match in pattern.finditer(normalized_text)
            ):
                matched.append(term)

        return cls._unique_strings(matched)

    @classmethod
    def _excluded_ranges(
        cls,
        text: str,
        excluded_phrases: tuple[str, ...],
    ) -> list[tuple[int, int]]:
        """
        Return character ranges occupied by longer confusable phrases.

        The rule is catalogue-driven rather than transcript-specific. Any
        concept can declare compound phrases which must not be consumed by a
        shorter alias.
        """

        ranges: list[tuple[int, int]] = []

        for phrase in excluded_phrases:
            normalized_phrase = cls._normalize_for_match(phrase)
            if not normalized_phrase:
                continue

            pattern = re.compile(
                r"(?<!\w)"
                + re.escape(normalized_phrase).replace(r"\ ", r"\s+")
                + r"(?!\w)",
                re.IGNORECASE,
            )

            ranges.extend(
                match.span()
                for match in pattern.finditer(text)
            )

        return ranges

    @staticmethod
    def _overlaps_any(
        span: tuple[int, int],
        blocked_ranges: list[tuple[int, int]],
    ) -> bool:
        start, end = span

        return any(
            start < blocked_end and end > blocked_start
            for blocked_start, blocked_end in blocked_ranges
        )

    def _passes_salience_gate(
        self,
        keyword: KeywordEvidence,
        semantic_score: float,
    ) -> bool:
        # Ambiguous aliases are stricter than ordinary single-word aliases.
        # They must be confirmed by catalogue-provided contextual language.
        # This is generic and applies to any future ambiguous catalogue term.
        if keyword.ambiguous_alias_only:
            return (
                semantic_score
                >= self.config.ambiguous_alias_semantic_floor
                and len(keyword.matched_context_terms)
                >= self.config.ambiguous_alias_min_context_terms
            )

        if not keyword.single_word_alias_only:
            return True

        return any(
            (
                semantic_score >= self.config.single_word_semantic_floor,
                len(keyword.evidence_sentences)
                >= self.config.single_word_min_evidence_sentences,
                len(keyword.matched_aliases)
                >= self.config.single_word_min_distinct_aliases,
            )
        )

    @staticmethod
    def _calculate_salience_score(
        keyword: KeywordEvidence,
        semantic_score: float,
    ) -> float:
        semantic_component = max(0.0, min(1.0, semantic_score))
        evidence_component = min(1.0, len(keyword.evidence_sentences) / 3.0)
        alias_component = min(1.0, len(keyword.matched_aliases) / 2.0)

        return min(
            1.0,
            0.55 * semantic_component
            + 0.25 * evidence_component
            + 0.20 * alias_component,
        )

    @staticmethod
    def _calculate_confidence(
        keyword_score: float,
        semantic_score: float,
        single_word_alias_only: bool,
        salience_score: float,
    ) -> float:
        semantic_component = max(0.0, min(1.0, semantic_score))

        combined = 0.58 * semantic_component + 0.42 * keyword_score

        # Multiword technical phrases are stronger direct evidence than an
        # isolated one-word match. Single-word matches therefore cannot win
        # through the keyword path alone.
        keyword_multiplier = 0.72 if single_word_alias_only else 0.88
        keyword_supported = keyword_multiplier * keyword_score
        semantic_only = 0.82 * semantic_component

        base_confidence = max(combined, keyword_supported, semantic_only)

        # Salience has a bounded effect: it can reduce an incidental mention,
        # but it cannot manufacture confidence without real evidence.
        adjusted = base_confidence * (0.85 + 0.15 * salience_score)

        return max(0.0, min(1.0, adjusted))

    def _suppress_redundant_candidates(
        self,
        candidates: list[RawTopicCandidate],
    ) -> list[RawTopicCandidate]:
        kept: list[RawTopicCandidate] = []

        for candidate in candidates:
            should_skip = False

            for existing in kept:
                if self._is_redundant_pair(candidate, existing):
                    should_skip = True
                    break

            if not should_skip:
                kept.append(candidate)

        return kept

    def _is_redundant_pair(
        self,
        candidate: RawTopicCandidate,
        existing: RawTopicCandidate,
    ) -> bool:
        # Parent/child suppression.
        if candidate.parent_concept_id == existing.concept_id:
            return candidate.confidence <= (
                existing.confidence + self.config.parent_suppression_margin
            )

        if existing.parent_concept_id == candidate.concept_id:
            return candidate.confidence <= existing.confidence

        evidence_overlap = self._jaccard(
            self._normalized_evidence(candidate.evidence),
            self._normalized_evidence(existing.evidence),
        )

        alias_overlap = self._jaccard(
            self._alias_tokens(candidate.matched_aliases),
            self._alias_tokens(existing.matched_aliases),
        )

        same_reference = (
            candidate.official_reference == existing.official_reference
        )

        return (
            evidence_overlap >= self.config.duplicate_evidence_overlap
            and alias_overlap >= self.config.duplicate_alias_token_overlap
            and (same_reference or self._topic_token_overlap(candidate, existing) >= 0.50)
        )

    def _topic_token_overlap(
        self,
        first: RawTopicCandidate,
        second: RawTopicCandidate,
    ) -> float:
        return self._jaccard(
            self._stemmed_tokens(first.topic),
            self._stemmed_tokens(second.topic),
        )

    @classmethod
    def _alias_tokens(cls, aliases: list[str]) -> set[str]:
        tokens: set[str] = set()
        for alias in aliases:
            tokens.update(cls._stemmed_tokens(alias))
        return tokens

    @classmethod
    def _normalized_evidence(cls, evidence: list[str]) -> set[str]:
        return {
            cls._normalize_for_match(value)
            for value in evidence
            if cls._normalize_for_match(value)
        }

    @classmethod
    def _stemmed_tokens(cls, text: str) -> set[str]:
        tokens = cls._normalize_for_match(text).split()
        return {cls._light_stem(token) for token in tokens if len(token) > 2}

    @staticmethod
    def _light_stem(token: str) -> str:
        for suffix in ("ing", "ed", "es", "s"):
            if token.endswith(suffix) and len(token) > len(suffix) + 3:
                return token[: -len(suffix)]
        return token

    @staticmethod
    def _jaccard(first: set[str], second: set[str]) -> float:
        if not first or not second:
            return 0.0
        return len(first & second) / len(first | second)

    @staticmethod
    def _normalize_for_match(text: str) -> str:
        text = text.lower()
        text = re.sub(r"[^a-z0-9]+", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _word_count(text: str) -> int:
        return len(re.findall(r"\S+", text))

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = re.sub(r"\s+", " ", value).strip().lower()
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique


## 7. Evidence-quality evaluator


In [ ]:
import re
from collections import defaultdict
from dataclasses import dataclass



@dataclass(frozen=True)
class EvidenceQualityConfig:
    """
    Generic evidence-quality rules used after confidence filtering.

    The evaluator:

    1. separates a brief mention from actual teaching;
    2. enforces catalogue-driven ambiguous-alias context;
    3. prevents generic multi-topic recaps from becoming strong evidence;
    4. reduces candidates whose evidence is owned more strongly by another
       candidate in the same chunk.
    """

    mention_only_max_score: float = 0.34
    definition_score: float = 0.50
    explanation_score: float = 0.68
    worked_example_score: float = 0.84
    sustained_teaching_score: float = 0.95

    isolated_mention_reject: bool = True
    ambiguous_alias_requires_context: bool = True

    # A sentence must be shared by at least this many official candidates
    # before it can be treated as a generic multi-topic recap.
    recap_minimum_shared_topics: int = 3
    recap_quality_penalty: float = 0.18

    # A brief comparison-only reference must not become a lesson topic.
    # More substantial comparisons are retained because they may genuinely
    # teach both concepts (for example, a dedicated merge-vs-bubble section).
    comparison_only_max_alias_hits: int = 2
    comparison_quality_penalty: float = 0.16
    comparison_window_words: int = 16

    # Minimal generic safeguards:
    # - explicit exclusion language is negative evidence for that candidate;
    # - a single contextual/example reference without an independent teaching
    #   signal receives only a small quality penalty.
    explicit_exclusion_quality_penalty: float = 0.22
    isolated_context_quality_penalty: float = 0.08

    evidence_overlap_threshold: float = 0.50
    competition_margin: float = 0.12
    maximum_shared_evidence_penalty: float = 0.24

    minimum_adjusted_score: float = 0.46

    def __post_init__(self) -> None:
        probability_fields = (
            "mention_only_max_score",
            "definition_score",
            "explanation_score",
            "worked_example_score",
            "sustained_teaching_score",
            "recap_quality_penalty",
            "comparison_quality_penalty",
            "explicit_exclusion_quality_penalty",
            "isolated_context_quality_penalty",
            "evidence_overlap_threshold",
            "competition_margin",
            "maximum_shared_evidence_penalty",
            "minimum_adjusted_score",
        )

        for field_name in probability_fields:
            value = getattr(self, field_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{field_name} must be between 0 and 1."
                )

        if self.recap_minimum_shared_topics < 2:
            raise ValueError(
                "recap_minimum_shared_topics must be at least 2."
            )

        if self.comparison_only_max_alias_hits < 1:
            raise ValueError(
                "comparison_only_max_alias_hits must be at least 1."
            )

        if self.comparison_window_words < 4:
            raise ValueError(
                "comparison_window_words must be at least 4."
            )


@dataclass(frozen=True)
class EvidenceQualityResult:
    retained: list[TopicCandidate]
    rejected: list[TopicCandidate]


@dataclass(frozen=True)
class ComparisonEvidenceProfile:
    total_hits: int
    comparison_hits: int
    independent_hits: int
    comparison_only: bool


class EvidenceQualityEvaluator:
    """
    Evaluate retained official candidates without changing topic discovery.

    Candidate extraction answers:
        "Which official topics may be present?"

    This evaluator answers:
        "Is the evidence deep and specific enough to keep the topic?"
    """

    _MENTION_PATTERNS = (
        r"\b(?:also|briefly|just)\s+(?:mention|mentioned|talked about)\b",
        r"\bby the way\b",
        r"\bin (?:the|our) syllabus\b",
        r"\bwe (?:also )?have\b",
        r"\bwe will (?:cover|do|study|look at)\b",
        r"\bnext (?:topic|chapter|lesson)\b",
        r"\bmove on to\b",
        r"\bremember (?:that )?we (?:did|covered|talked about)\b",
        r"\bwe are not (?:covering|doing|studying)\b",
    )

    _RECAP_PATTERNS = (
        r"\b(?:today|this lesson|last lesson|previously)\b.{0,80}"
        r"\b(?:covered|revised|reviewed|looked at|studied|discussed)\b",
        r"\bwe (?:have )?(?:covered|revised|reviewed|looked at|studied|discussed)\b",
        r"\btopics? (?:were|included|covered)\b",
        r"\bto recap\b",
        r"\bin summary\b",
        r"\bquick recap\b",
        r"\bwe learned about\b",
    )

    _DEFINITION_PATTERNS = (
        r"\b(?:is|are) (?:a|an|the)\b",
        r"\bmeans\b",
        r"\brefers to\b",
        r"\bdefined as\b",
        r"\bwe call\b",
    )

    _EXPLANATION_PATTERNS = (
        r"\bbecause\b",
        r"\btherefore\b",
        r"\bso that\b",
        r"\bworks by\b",
        r"\bthe reason\b",
        r"\bthis means\b",
        r"\bif\b.{0,100}\bthen\b",
        r"\bwhy\b",
        r"\bhow\b",
        r"\bcauses?\b",
        r"\ballows?\b",
        r"\bprevents?\b",
    )

    _WORKED_EXAMPLE_PATTERNS = (
        r"\bfor example\b",
        r"\bfor instance\b",
        r"\blet'?s (?:say|take|try|trace|calculate|work)\b",
        r"\bsuppose\b",
        r"\bgiven\b",
        r"\bstep by step\b",
        r"\btrace (?:this|the|through)\b",
        r"\bcalculate\b",
        r"\bwork(?:ed)? example\b",
    )

    _PRACTICE_PATTERNS = (
        r"\bwhat (?:is|happens|would|will|do you)\b",
        r"\bwhy (?:is|does|would|do you)\b",
        r"\bhow (?:many|does|would|do you)\b",
        r"\btry (?:this|it|the next|another)\b",
        r"\byour answer\b",
        r"\bcorrect answer\b",
        r"\bquestion\b",
        r"\bpractice\b",
        r"\bexercise\b",
        r"\bexam\b",
    )


    _COMPARISON_PATTERNS = (
        r"\bunlike\b",
        r"\bcompared? (?:with|to)\b",
        r"\bin comparison (?:with|to)\b",
        r"\bin contrast (?:with|to)\b",
        r"\bas opposed to\b",
        r"\bwhereas\b",
        r"\bon the other hand\b",
        r"\bversus\b",
        r"\bvs\b",
        r"\banother type of\b",
        r"\bdifferent from\b",
    )

    # Strict negative cues only. These are deliberately narrower than
    # "not the main focus", because a supporting topic may still be taught.
    _EXPLICIT_EXCLUSION_CUES = (
        "not being taught",
        "not being studied",
        "not being covered",
        "not learning",
        "not studying",
        "not covering",
        "not teaching",
        "haven t taught",
        "haven t actually taught",
        "have not taught",
        "different lesson",
        "separate lesson",
        "don t count",
        "do not count",
    )

    _LESSON_PURPOSE_CUES = (
        "today we are learning",
        "today we re learning",
        "today we are going to",
        "today we re going to",
        "main reason",
        "main focus",
        "main concept",
        "main concepts",
        "lesson is about",
        "lesson focuses on",
    )

    def __init__(
        self,
        config: EvidenceQualityConfig | None = None,
    ) -> None:
        self.config = config or EvidenceQualityConfig()

    def evaluate(
        self,
        *,
        text: str,
        candidates: list[TopicCandidate],
    ) -> EvidenceQualityResult:
        if not candidates:
            return EvidenceQualityResult(
                retained=[],
                rejected=[],
            )

        shared_recap_evidence = self._shared_recap_evidence(
            candidates
        )

        assessed = [
            self._assess_candidate(
                text=text,
                candidate=candidate,
                shared_recap_evidence=shared_recap_evidence,
            )
            for candidate in candidates
        ]

        retained: list[TopicCandidate] = []
        rejected: list[TopicCandidate] = []

        for candidate in assessed:
            rejection_reason = self._initial_rejection_reason(
                candidate
            )

            if rejection_reason is not None:
                rejected.append(
                    self._reject(
                        candidate,
                        rejection_reason,
                    )
                )
            else:
                retained.append(candidate)

        retained, competition_rejected = (
            self._apply_candidate_competition(
                retained
            )
        )
        rejected.extend(competition_rejected)

        retained.sort(
            key=lambda candidate: (
                candidate.cs_relevance_score,
                candidate.evidence_quality_score,
                candidate.teaching_depth_level,
                candidate.semantic_score,
            ),
            reverse=True,
        )

        rejected.sort(
            key=lambda candidate: (
                candidate.cs_relevance_score,
                candidate.evidence_quality_score,
            ),
            reverse=True,
        )

        return EvidenceQualityResult(
            retained=retained,
            rejected=rejected,
        )

    def _assess_candidate(
        self,
        *,
        text: str,
        candidate: TopicCandidate,
        shared_recap_evidence: set[str],
    ) -> TopicCandidate:
        evidence = candidate.evidence or [text]

        comparison_profile = self._comparison_profile(
            text=text,
            candidate=candidate,
        )

        recap_evidence = [
            sentence
            for sentence in evidence
            if self._normalize(sentence) in shared_recap_evidence
        ]

        explicit_exclusion_evidence = [
            sentence
            for sentence in evidence
            if self._is_candidate_explicitly_excluded(
                sentence=sentence,
                candidate=candidate,
            )
        ]
        explicit_exclusion_normalized = {
            self._normalize(sentence)
            for sentence in explicit_exclusion_evidence
        }

        substantive_evidence = [
            sentence
            for sentence in evidence
            if (
                self._normalize(sentence) not in shared_recap_evidence
                and self._normalize(sentence)
                not in explicit_exclusion_normalized
            )
        ]

        recap_only = (
            bool(recap_evidence)
            and not substantive_evidence
            and not explicit_exclusion_evidence
        )
        exclusion_only = (
            bool(explicit_exclusion_evidence)
            and not substantive_evidence
        )

        evidence_for_depth = (
            substantive_evidence
            if substantive_evidence
            else recap_evidence
        )

        depth_levels = [
            (
                0
                if self._normalize(sentence) in shared_recap_evidence
                else self._sentence_depth(sentence)
            )
            for sentence in evidence_for_depth
        ]

        # Independent teaching is candidate-specific evidence that actually
        # defines/explains/practises the concept or explicitly frames it as
        # a lesson purpose. Repetition remains valid teaching evidence.
        independent_teaching_count = sum(
            1
            for sentence in substantive_evidence
            if self._has_independent_teaching_signal(
                sentence=sentence,
                candidate=candidate,
            )
        )

        depth_level = max(depth_levels, default=0)

        non_mention_count = sum(
            1
            for level in depth_levels
            if level >= 1
        )
        explanatory_count = sum(
            1
            for level in depth_levels
            if level >= 2
        )

        # Only substantive evidence can upgrade a topic to sustained teaching.
        if (
            len(substantive_evidence) >= 3
            and explanatory_count >= 2
        ):
            depth_level = 4
        elif (
            len(substantive_evidence) >= 2
            and non_mention_count >= 2
            and depth_level >= 2
        ):
            depth_level = 4

        depth_label = self._depth_label(
            depth_level
        )
        depth_score = self._depth_score(
            depth_level
        )

        required_context_met = (
            len(candidate.matched_context_terms)
            >= candidate.minimum_context_hits
        )
        context_collision = bool(
            candidate.matched_conflicting_context_terms
            and candidate.ambiguous_alias_only
            and not required_context_met
        )

        context_component = (
            1.0
            if required_context_met
            else (
                0.20
                if context_collision
                else (0.35 if candidate.ambiguous_alias_only else 0.70)
            )
        )

        semantic_component = max(
            0.0,
            min(
                1.0,
                candidate.semantic_score,
            ),
        )

        quality_score = (
            0.45 * depth_score
            + 0.25 * candidate.salience_score
            + 0.20 * semantic_component
            + 0.10 * context_component
        )

        if depth_level == 0:
            quality_score = min(
                quality_score,
                self.config.mention_only_max_score,
            )

        if recap_evidence:
            recap_ratio = (
                len(recap_evidence)
                / max(1, len(evidence))
            )
            quality_score = max(
                0.0,
                quality_score
                - (
                    self.config.recap_quality_penalty
                    * recap_ratio
                ),
            )

        # A shallow comparison-only mention still receives the existing
        # quality penalty. If the evidence is explanatory or stronger,
        # the comparison itself is genuine teaching evidence and should
        # be scored normally.
        if (
            comparison_profile.comparison_only
            and depth_level < 2
        ):
            quality_score = max(
                0.0,
                quality_score
                - self.config.comparison_quality_penalty,
            )

        # Change 1: strict candidate-specific exclusion language is negative
        # evidence. Mixed evidence is penalised proportionally; exclusion-only
        # evidence is capped so it cannot survive as an independently taught
        # topic merely because semantic similarity is high.
        if explicit_exclusion_evidence:
            exclusion_ratio = (
                len(explicit_exclusion_evidence)
                / max(1, len(evidence))
            )
            quality_score = max(
                0.0,
                quality_score
                - (
                    self.config.explicit_exclusion_quality_penalty
                    * exclusion_ratio
                ),
            )

        # Change 2: a lone contextual/example reference without a direct
        # definition, explanation, practice signal, lesson-purpose cue, or
        # repeated evidence receives a small generic penalty. This does not
        # hard-reject single examples; it only stops mention/context breadth
        # from outweighing actual teaching depth.
        if (
            len(substantive_evidence) == 1
            and independent_teaching_count == 0
        ):
            quality_score = max(
                0.0,
                quality_score
                - self.config.isolated_context_quality_penalty,
            )

        adjusted_relevance = (
            0.72 * candidate.cs_relevance_score
            + 0.28 * quality_score
        )

        if exclusion_only:
            adjusted_relevance = min(
                adjusted_relevance,
                max(
                    0.0,
                    self.config.minimum_adjusted_score - 0.01,
                ),
            )
            depth_level = 0
            depth_label = self._depth_label(depth_level)

        notes = list(
            candidate.evidence_quality_notes
        )
        notes.append(
            f"Teaching depth: {depth_label} ({depth_level})."
        )

        if candidate.matched_context_terms:
            notes.append(
                "Supporting context: "
                + ", ".join(candidate.matched_context_terms)
                + "."
            )

        if candidate.matched_conflicting_context_terms:
            notes.append(
                "Conflicting context: "
                + ", ".join(
                    candidate.matched_conflicting_context_terms
                )
                + "."
            )

        if recap_evidence:
            notes.append(
                f"Generic recap evidence detected: "
                f"{len(recap_evidence)} sentence(s)."
            )

        if comparison_profile.comparison_hits:
            notes.append(
                "Comparison evidence detected: "
                f"{comparison_profile.comparison_hits} hit(s); "
                f"independent evidence: "
                f"{comparison_profile.independent_hits} hit(s)."
            )

        if comparison_profile.comparison_only:
            if depth_level < 2:
                notes.append(
                    "Candidate is supported only by a brief comparison or "
                    "contrast reference."
                )
            else:
                notes.append(
                    "Comparison-only evidence retained because teaching "
                    "depth is explanatory or stronger."
                )

        if explicit_exclusion_evidence:
            notes.append(
                "Candidate-specific explicit exclusion evidence detected: "
                f"{len(explicit_exclusion_evidence)} sentence(s)."
            )

        if (
            len(substantive_evidence) == 1
            and independent_teaching_count == 0
        ):
            notes.append(
                "Only one contextual/example reference was found without "
                "an independent teaching signal; a small quality penalty "
                "was applied."
            )

        return candidate.model_copy(
            update={
                "teaching_depth_level": depth_level,
                "teaching_depth_label": depth_label,
                "evidence_quality_score": round(
                    quality_score,
                    4,
                ),
                "cs_relevance_score": round(
                    adjusted_relevance,
                    4,
                ),
                "recap_evidence_only": recap_only,
                "recap_evidence_count": len(recap_evidence),
                "substantive_evidence_count": len(
                    substantive_evidence
                ),
                "context_collision": context_collision,
                "comparison_evidence_only": (
                    comparison_profile.comparison_only
                ),
                "comparison_evidence_count": (
                    comparison_profile.comparison_hits
                ),
                "independent_evidence_count": (
                    comparison_profile.independent_hits
                ),
                "evidence_quality_notes": notes,
            }
        )

    def _shared_recap_evidence(
        self,
        candidates: list[TopicCandidate],
    ) -> set[str]:
        evidence_topics: dict[str, set[str]] = defaultdict(set)
        original_evidence: dict[str, str] = {}

        for candidate in candidates:
            for sentence in candidate.evidence:
                normalized = self._normalize(sentence)
                if not normalized:
                    continue

                evidence_topics[normalized].add(
                    candidate.concept_id
                )
                original_evidence.setdefault(
                    normalized,
                    sentence,
                )

        shared_recap: set[str] = set()

        for normalized, concept_ids in evidence_topics.items():
            if (
                len(concept_ids)
                < self.config.recap_minimum_shared_topics
            ):
                continue

            original = original_evidence[normalized]
            if self._matches_any(
                self._normalize(original),
                self._RECAP_PATTERNS,
            ):
                shared_recap.add(normalized)

        return shared_recap

    def _initial_rejection_reason(
        self,
        candidate: TopicCandidate,
    ) -> str | None:
        if candidate.context_collision:
            return (
                "Rejected because ambiguous aliases were found in a "
                "conflicting conceptual context without enough "
                "catalogue-defined supporting context."
            )

        if (
            self.config.ambiguous_alias_requires_context
            and candidate.ambiguous_alias_only
            and not candidate.matched_context_terms
        ):
            return (
                "Rejected because the candidate is supported only by an "
                "ambiguous alias without catalogue-defined context."
            )

        if candidate.recap_evidence_only:
            return (
                "Rejected because the candidate is supported only by a "
                "generic multi-topic recap rather than topic-specific "
                "teaching evidence."
            )

        # Comparison-only evidence is unsafe when it is merely a mention
        # or definition. But a dedicated explanatory comparison can itself
        # be the lesson's teaching evidence, so do not hard-reject it.
        if (
            candidate.comparison_evidence_only
            and candidate.teaching_depth_level < 2
        ):
            return (
                "Rejected because the candidate is supported only by a "
                "brief comparison or contrast reference, without "
                "independent teaching evidence."
            )

        if (
            self.config.isolated_mention_reject
            and candidate.teaching_depth_level == 0
        ):
            return (
                "Rejected because the evidence is an isolated mention "
                "rather than definition, explanation, example or practice."
            )

        if (
            candidate.cs_relevance_score
            < self.config.minimum_adjusted_score
        ):
            return (
                "Rejected because evidence quality reduced the adjusted "
                "candidate score below the retention threshold."
            )

        return None

    def _apply_candidate_competition(
        self,
        candidates: list[TopicCandidate],
    ) -> tuple[
        list[TopicCandidate],
        list[TopicCandidate],
    ]:
        ordered = sorted(
            candidates,
            key=self._ownership_score,
            reverse=True,
        )

        kept: list[TopicCandidate] = []
        rejected: list[TopicCandidate] = []

        for candidate in ordered:
            competitor = self._stronger_competitor(
                candidate=candidate,
                kept=kept,
            )

            if competitor is None:
                kept.append(candidate)
                continue

            overlap = self._evidence_overlap(
                candidate,
                competitor,
            )

            penalty = min(
                self.config.maximum_shared_evidence_penalty,
                overlap
                * self.config.maximum_shared_evidence_penalty,
            )

            adjusted_score = max(
                0.0,
                candidate.cs_relevance_score - penalty,
            )

            notes = list(
                candidate.evidence_quality_notes
            )
            notes.append(
                "Shared evidence is owned more strongly by "
                f"'{competitor.topic}'."
            )

            updated = candidate.model_copy(
                update={
                    "shared_evidence_penalty": round(
                        penalty,
                        4,
                    ),
                    "cs_relevance_score": round(
                        adjusted_score,
                        4,
                    ),
                    "evidence_quality_notes": notes,
                }
            )

            if (
                adjusted_score
                < self.config.minimum_adjusted_score
            ):
                rejected.append(
                    self._reject(
                        updated,
                        "Rejected after candidate competition because "
                        "another topic explained the shared evidence more "
                        "strongly.",
                    )
                )
            else:
                kept.append(updated)

        return kept, rejected

    def _stronger_competitor(
        self,
        *,
        candidate: TopicCandidate,
        kept: list[TopicCandidate],
    ) -> TopicCandidate | None:
        candidate_ownership = self._ownership_score(
            candidate
        )

        for existing in kept:
            overlap = self._evidence_overlap(
                candidate,
                existing,
            )

            if (
                overlap
                < self.config.evidence_overlap_threshold
            ):
                continue

            existing_ownership = (
                self._ownership_score(
                    existing
                )
            )

            candidate_is_contextually_weaker = any(
                (
                    candidate.ambiguous_alias_only
                    and not existing.ambiguous_alias_only,
                    candidate.teaching_depth_level
                    < existing.teaching_depth_level,
                    (
                        not candidate.matched_context_terms
                        and bool(existing.matched_context_terms)
                    ),
                )
            )

            if (
                candidate_is_contextually_weaker
                and existing_ownership
                >= (
                    candidate_ownership
                    + self.config.competition_margin
                )
            ):
                return existing

        return None

    @staticmethod
    def _ownership_score(
        candidate: TopicCandidate,
    ) -> float:
        context_score = min(
            1.0,
            len(candidate.matched_context_terms)
            / 2.0,
        )

        ambiguity_penalty = (
            0.10
            if candidate.ambiguous_alias_only
            else 0.0
        )

        recap_penalty = (
            0.12
            if candidate.recap_evidence_count > 0
            else 0.0
        )

        comparison_penalty = (
            0.14
            if candidate.comparison_evidence_only
            else 0.0
        )

        context_collision_penalty = (
            0.18 if candidate.context_collision else 0.0
        )

        return (
            0.32 * candidate.cs_relevance_score
            + 0.28 * candidate.evidence_quality_score
            + 0.20 * candidate.salience_score
            + 0.12 * max(
                0.0,
                min(
                    1.0,
                    candidate.semantic_score,
                ),
            )
            + 0.08 * context_score
            - ambiguity_penalty
            - recap_penalty
            - comparison_penalty
            - context_collision_penalty
        )

    @classmethod
    def _evidence_overlap(
        cls,
        first: TopicCandidate,
        second: TopicCandidate,
    ) -> float:
        first_evidence = {
            cls._normalize(value)
            for value in first.evidence
            if cls._normalize(value)
        }
        second_evidence = {
            cls._normalize(value)
            for value in second.evidence
            if cls._normalize(value)
        }

        if not first_evidence or not second_evidence:
            return 0.0

        return (
            len(
                first_evidence
                & second_evidence
            )
            / len(
                first_evidence
                | second_evidence
            )
        )


    def _comparison_profile(
        self,
        *,
        text: str,
        candidate: TopicCandidate,
    ) -> ComparisonEvidenceProfile:
        """
        Distinguish a brief contrast reference from independent teaching.

        Example rejected:
            "Linear search works on unsorted data, unlike binary search."

        Example retained:
            A dedicated comparison section repeatedly explains merge sort,
            its behaviour, speed, suitability and memory use.

        The rule is generic: it uses catalogue aliases and comparison
        language, never transcript-specific concept pairs.
        """

        normalized_text = self._normalize(text)
        words = normalized_text.split()

        aliases = self._comparison_aliases(candidate)
        if not aliases or not words:
            return ComparisonEvidenceProfile(
                total_hits=0,
                comparison_hits=0,
                independent_hits=0,
                comparison_only=False,
            )

        hit_spans: list[tuple[int, int]] = []

        for alias in aliases:
            alias_words = self._normalize(alias).split()
            if not alias_words:
                continue

            width = len(alias_words)

            for index in range(0, len(words) - width + 1):
                if words[index:index + width] == alias_words:
                    hit_spans.append((index, index + width))

        # De-duplicate the same occurrence matched by overlapping aliases.
        hit_spans = sorted(set(hit_spans))

        comparison_hits = 0

        for start, end in hit_spans:
            left = max(
                0,
                start - self.config.comparison_window_words,
            )
            right = min(
                len(words),
                end + self.config.comparison_window_words,
            )
            window = " ".join(words[left:right])

            if self._matches_any(
                window,
                self._COMPARISON_PATTERNS,
            ):
                comparison_hits += 1

        total_hits = len(hit_spans)
        independent_hits = max(
            0,
            total_hits - comparison_hits,
        )

        comparison_only = (
            total_hits > 0
            and comparison_hits == total_hits
            and independent_hits == 0
            and total_hits
            <= self.config.comparison_only_max_alias_hits
        )

        return ComparisonEvidenceProfile(
            total_hits=total_hits,
            comparison_hits=comparison_hits,
            independent_hits=independent_hits,
            comparison_only=comparison_only,
        )

    @classmethod
    def _comparison_aliases(
        cls,
        candidate: TopicCandidate,
    ) -> list[str]:
        aliases = [
            *candidate.matched_aliases,
            candidate.topic,
        ]

        unique: list[str] = []
        seen: set[str] = set()

        for alias in aliases:
            normalized = cls._normalize(alias)

            if (
                not normalized
                or normalized in seen
                or len(normalized.split()) < 2
            ):
                continue

            seen.add(normalized)
            unique.append(alias)

        return unique

    def _is_candidate_explicitly_excluded(
        self,
        *,
        sentence: str,
        candidate: TopicCandidate,
    ) -> bool:
        """Return True only when a strict exclusion cue is near this topic."""

        normalized = self._normalize(sentence)
        if not normalized:
            return False

        anchors = self._candidate_anchors(candidate)
        if not anchors:
            return False

        for anchor in anchors:
            for match in re.finditer(
                rf"(?<![a-z0-9]){re.escape(anchor)}(?![a-z0-9])",
                normalized,
            ):
                window = normalized[
                    max(0, match.start() - 100):
                    min(len(normalized), match.end() + 100)
                ]
                if any(
                    cue in window
                    for cue in self._EXPLICIT_EXCLUSION_CUES
                ):
                    return True

        return False

    def _has_independent_teaching_signal(
        self,
        *,
        sentence: str,
        candidate: TopicCandidate,
    ) -> bool:
        """Detect generic evidence that the candidate itself is being taught."""

        normalized = self._normalize(sentence)
        if not normalized:
            return False

        direct_teaching = any(
            (
                self._matches_any(
                    normalized,
                    self._DEFINITION_PATTERNS,
                ),
                self._matches_any(
                    normalized,
                    self._EXPLANATION_PATTERNS,
                ),
                self._matches_any(
                    normalized,
                    self._PRACTICE_PATTERNS,
                ),
            )
        )

        if direct_teaching:
            return True

        # A worked-example cue on its own can describe another concept's
        # example. Treat it as independent teaching only when accompanied by
        # an explanation or when lesson-purpose language names this candidate.
        if (
            self._matches_any(
                normalized,
                self._WORKED_EXAMPLE_PATTERNS,
            )
            and self._matches_any(
                normalized,
                self._EXPLANATION_PATTERNS,
            )
        ):
            return True

        anchors = self._candidate_anchors(candidate)
        for cue in self._LESSON_PURPOSE_CUES:
            cue_index = normalized.find(cue)
            if cue_index < 0:
                continue

            purpose_window = normalized[
                max(0, cue_index - 80):
                min(len(normalized), cue_index + len(cue) + 140)
            ]
            if any(
                anchor in purpose_window
                for anchor in anchors
            ):
                return True

        return False

    @classmethod
    def _candidate_anchors(
        cls,
        candidate: TopicCandidate,
    ) -> set[str]:
        anchors = {
            cls._normalize(value)
            for value in (
                [candidate.topic]
                + list(candidate.matched_aliases)
            )
            if value
        }
        anchors.discard("")
        return anchors

    def _sentence_depth(
        self,
        sentence: str,
    ) -> int:
        normalized = self._normalize(
            sentence
        )

        if not normalized:
            return 0

        mention = self._matches_any(
            normalized,
            self._MENTION_PATTERNS,
        )
        definition = self._matches_any(
            normalized,
            self._DEFINITION_PATTERNS,
        )
        explanation = self._matches_any(
            normalized,
            self._EXPLANATION_PATTERNS,
        )
        worked_example = self._matches_any(
            normalized,
            self._WORKED_EXAMPLE_PATTERNS,
        )
        practice = self._matches_any(
            normalized,
            self._PRACTICE_PATTERNS,
        )

        if worked_example and practice:
            return 4
        if practice and explanation:
            return 4
        if worked_example:
            return 3
        if explanation:
            return 2
        if definition:
            return 1
        if mention:
            return 0

        return 1

    def _depth_score(
        self,
        level: int,
    ) -> float:
        return {
            0: self.config.mention_only_max_score,
            1: self.config.definition_score,
            2: self.config.explanation_score,
            3: self.config.worked_example_score,
            4: self.config.sustained_teaching_score,
        }[level]

    @staticmethod
    def _depth_label(
        level: int,
    ) -> str:
        return {
            0: "mention_only",
            1: "definition",
            2: "explanation",
            3: "worked_example",
            4: "sustained_teaching",
        }[level]

    @staticmethod
    def _matches_any(
        text: str,
        patterns: tuple[str, ...],
    ) -> bool:
        return any(
            re.search(
                pattern,
                text,
                re.IGNORECASE,
            )
            is not None
            for pattern in patterns
        )

    @staticmethod
    def _reject(
        candidate: TopicCandidate,
        reason: str,
    ) -> TopicCandidate:
        notes = list(
            candidate.evidence_quality_notes
        )
        notes.append(reason)

        return candidate.model_copy(
            update={
                "cs_relevant": False,
                "evidence_quality_notes": notes,
            }
        )

    @staticmethod
    def _normalize(
        text: str,
    ) -> str:
        text = re.sub(
            r"[^a-z0-9]+",
            " ",
            text.lower(),
        )
        return re.sub(
            r"\s+",
            " ",
            text,
        ).strip()


## 8. CS relevance filter


In [ ]:
from dataclasses import dataclass



@dataclass(frozen=True)
class CSRelevanceConfig:
    candidate_keep_threshold: float = 0.46
    uncertain_candidate_floor: float = 0.34
    llm_fallback_min_words: int = 80
    max_rejected_candidates: int = 3

    def __post_init__(self) -> None:
        if not 0.0 <= self.candidate_keep_threshold <= 1.0:
            raise ValueError("candidate_keep_threshold must be between 0 and 1.")
        if not 0.0 <= self.uncertain_candidate_floor <= 1.0:
            raise ValueError("uncertain_candidate_floor must be between 0 and 1.")
        if self.uncertain_candidate_floor > self.candidate_keep_threshold:
            raise ValueError(
                "uncertain_candidate_floor cannot exceed candidate_keep_threshold."
            )


class CSRelevanceFilter:
    def __init__(self, config: CSRelevanceConfig | None = None) -> None:
        self.config = config or CSRelevanceConfig()

    def filter(
        self,
        chunk_id: int,
        source_word_count: int,
        candidates: list[RawTopicCandidate],
    ) -> ChunkTopicResult:
        relevant: list[TopicCandidate] = []
        rejected: list[TopicCandidate] = []

        for candidate in candidates:
            relevance_score = candidate.confidence
            is_relevant = relevance_score >= self.config.candidate_keep_threshold
            filtered_candidate = TopicCandidate(
                **candidate.model_dump(),
                cs_relevance_score=round(relevance_score, 4),
                cs_relevant=is_relevant,
            )
            (relevant if is_relevant else rejected).append(filtered_candidate)

        relevant.sort(
            key=lambda candidate: (
                candidate.cs_relevance_score,
                candidate.salience_score,
            ),
            reverse=True,
        )
        rejected.sort(
            key=lambda candidate: candidate.cs_relevance_score,
            reverse=True,
        )

        if relevant:
            best_score = max(c.cs_relevance_score for c in relevant)
            support_bonus = min(0.06, 0.02 * max(0, len(relevant) - 1))
            chunk_relevance_score = min(0.95, best_score + support_bonus)
        elif rejected:
            best_score = max(c.cs_relevance_score for c in rejected)
            chunk_relevance_score = best_score
        else:
            best_score = 0.0
            chunk_relevance_score = 0.0

        requires_llm_fallback = (
            not relevant
            and source_word_count >= self.config.llm_fallback_min_words
            and best_score >= self.config.uncertain_candidate_floor
        )

        notes: list[str] = []
        if not candidates:
            notes.append("No official AQA topic candidate was detected.")
        elif not relevant:
            notes.append("Only low-confidence official AQA candidates were detected.")
        if requires_llm_fallback:
            notes.append("Chunk is borderline and may require GPT-OSS fallback.")

        return ChunkTopicResult(
            chunk_id=chunk_id,
            source_word_count=source_word_count,
            classification="official_aqa_topic" if relevant else "no_topic",
            is_cs_relevant=bool(relevant),
            creates_new_topic=bool(relevant),
            cs_relevance_score=round(chunk_relevance_score, 4),
            topic_candidates=relevant,
            rejected_candidates=rejected[: self.config.max_rejected_candidates],
            requires_llm_fallback=requires_llm_fallback,
            notes=notes,
        )


## 9. Unmapped-CS detector


In [ ]:
import re
from collections.abc import Callable, Sequence
from dataclasses import dataclass

import numpy as np



EmbeddingFunction = Callable[[Sequence[str], str, int], np.ndarray]


@dataclass(frozen=True)
class CSDomain:
    name: str
    description: str


@dataclass(frozen=True)
class UnmappedConceptFamily:
    """
    A generic CS concept family which is useful for detecting syllabus gaps.

    These entries are not official AQA mappings. They only provide a rough
    label when technical content remains outside the official catalogue.
    """

    rough_topic: str
    domain: str
    description: str
    aliases: tuple[str, ...]
    minimum_distinct_aliases: int = 1
    minimum_total_hits: int = 1


# Broad descriptions used only for residual semantic detection.
CS_DOMAINS: tuple[CSDomain, ...] = (
    CSDomain(
        name="Programming and software development",
        description=(
            "Programming source code, variables, control flow, methods, "
            "classes, objects, constructors, attributes, access control, "
            "data structures and software behaviour."
        ),
    ),
    CSDomain(
        name="Algorithms and computational thinking",
        description=(
            "Algorithms, problem solving, tracing execution, searching, "
            "sorting, decomposition, abstraction and efficiency."
        ),
    ),
    CSDomain(
        name="Data representation",
        description=(
            "Binary, hexadecimal, bits, bytes, text encoding, images, "
            "sound, file size and compression."
        ),
    ),
    CSDomain(
        name="Computer systems",
        description=(
            "Computer hardware, software, CPU architecture, memory, "
            "storage, operating systems and translators."
        ),
    ),
    CSDomain(
        name="Networks and cyber security",
        description=(
            "Computer networks, protocols, network devices, security, "
            "attacks, authentication, encryption and protective controls."
        ),
    ),
    CSDomain(
        name="Databases and data management",
        description=(
            "Relational databases, tables, records, fields, keys, SQL and "
            "database queries."
        ),
    ),
)


# Generic, transcript-independent families commonly encountered in lessons
# but not represented as explicit AQA 8525 catalogue topics.
UNMAPPED_CONCEPT_FAMILIES: tuple[UnmappedConceptFamily, ...] = (
    UnmappedConceptFamily(
        rough_topic="Dynamic arrays and list collections",
        domain="Programming and software development",
        description=(
            "Resizable sequence data structures such as ArrayList or a "
            "dynamic array whose size can change while a program runs."
        ),
        aliases=(
            "array list",
            "array lists",
            "arraylist",
            "arraylists",
            "dynamic array",
            "resizable array",
            "resizable list",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Object construction and initialisation",
        domain="Programming and software development",
        description=(
            "Constructors create or initialise objects and set their "
            "initial attributes or state."
        ),
        aliases=(
            "constructor",
            "constructors",
            "object constructor",
            "create an object",
            "initialise an object",
            "initialize an object",
            "object initialisation",
            "object initialization",
            "class instance",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Encapsulation and access modifiers",
        domain="Programming and software development",
        description=(
            "Private and public members, class attributes and controlled "
            "access to an object's internal state."
        ),
        aliases=(
            "private attribute",
            "private attributes",
            "public attribute",
            "public attributes",
            "private and public",
            "access modifier",
            "access modifiers",
            "outside the class",
            "encapsulation",
            "controlled access",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Inheritance and polymorphism",
        domain="Programming and software development",
        description=(
            "Object-oriented inheritance, subclasses, method overriding "
            "and polymorphic behaviour."
        ),
        aliases=(
            "inheritance",
            "subclass",
            "superclass",
            "base class",
            "derived class",
            "method overriding",
            "polymorphism",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Higher-dimensional arrays",
        domain="Programming and software development",
        description=(
            "Arrays with three or more dimensions, including 3D, 4D and "
            "higher-dimensional indexing and visualisation."
        ),
        aliases=(
            "three dimensional array",
            "3d array",
            "four dimensional array",
            "4d array",
            "five dimensional array",
            "5d array",
            "higher dimensional array",
            "multidimensional array beyond two dimensions",
        ),
        minimum_distinct_aliases=1,
        minimum_total_hits=1,
    ),
    UnmappedConceptFamily(
        rough_topic="Time complexity",
        domain="Algorithms and computational thinking",
        description=(
            "How an algorithm's running time grows as the input size grows, "
            "including constant, linear, polynomial and exponential time."
        ),
        aliases=(
            "time complexity",
            "constant time",
            "linear time",
            "polynomial time",
            "exponential time",
            "running time grows",
        ),
        minimum_distinct_aliases=1,
        minimum_total_hits=1,
    ),
    UnmappedConceptFamily(
        rough_topic="Space complexity",
        domain="Algorithms and computational thinking",
        description=(
            "How much memory or other storage an algorithm requires as its "
            "input size changes."
        ),
        aliases=(
            "space complexity",
            "memory complexity",
            "memory required",
            "amount of memory",
            "space efficient",
        ),
        minimum_distinct_aliases=1,
        minimum_total_hits=1,
    ),
    UnmappedConceptFamily(
        rough_topic="Big O notation",
        domain="Algorithms and computational thinking",
        description=(
            "Big O notation classifies how computational cost grows with "
            "input size."
        ),
        aliases=(
            "big o notation",
            "big o",
            "o of n",
            "o n",
            "complexity notation",
        ),
        minimum_distinct_aliases=1,
        minimum_total_hits=1,
    ),
    UnmappedConceptFamily(
        rough_topic="Tractable and intractable problems",
        domain="Algorithms and computational thinking",
        description=(
            "Problems classified by whether algorithms can solve them in a "
            "practical amount of time, including polynomial and worse-than-"
            "polynomial growth."
        ),
        aliases=(
            "tractable problem",
            "tractable problems",
            "intractable problem",
            "intractable problems",
            "reasonable amount of time",
            "polynomial time or better",
        ),
        minimum_distinct_aliases=1,
        minimum_total_hits=1,
    ),
)


@dataclass(frozen=True)
class UnmappedDetectionConfig:
    sentence_similarity_threshold: float = 0.48
    strong_sentence_threshold: float = 0.60
    minimum_evidence_sentences: int = 2
    max_signals: int = 4
    embedding_model: str = CHUNKING_EMBEDDING_MODEL


class CSUnmappedDetector:
    """
    Detect CS content which is not explained by retained official topics.

    Detection has two generic routes:
    1. lexical families for recognisable off-syllabus concepts;
    2. broad semantic residual detection for unknown CS material.

    The detector never fabricates an official AQA reference.
    """

    def __init__(
        self,
        config: UnmappedDetectionConfig | None = None,
        embedding_function: EmbeddingFunction = embed_texts,
    ) -> None:
        self.config = config or UnmappedDetectionConfig()
        self._embedding_function = embedding_function
        self._domain_embeddings: np.ndarray | None = None
        self._family_embeddings: np.ndarray | None = None

    def detect(
        self,
        text: str,
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        sentences = self._split_sentences(text)
        if not sentences:
            return []

        lexical_signals = self._detect_known_families(
            sentences=sentences,
            official_candidates=official_candidates,
        )

        semantic_signals = self._detect_semantic_residual(
            sentences=sentences,
            official_candidates=official_candidates,
        )

        # A specific lexical family already gives a clearer rough topic than
        # the generic semantic residual for the same sentence. Keeping both
        # would create duplicate evidence and could trigger an unnecessary
        # LLM fallback.
        specific_evidence = {
            self._normalize(signal.evidence)
            for signal in lexical_signals
        }
        semantic_signals = [
            signal
            for signal in semantic_signals
            if self._normalize(signal.evidence) not in specific_evidence
        ]

        combined = lexical_signals + semantic_signals
        combined.sort(key=lambda signal: signal.score, reverse=True)

        return self._deduplicate(combined)[: self.config.max_signals]

    def _detect_known_families(
        self,
        sentences: list[str],
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        official_terms = self._official_terms(official_candidates)
        family_embeddings = self._get_family_embeddings()
        signals: list[UnmappedCSSignal] = []

        for family_index, family in enumerate(UNMAPPED_CONCEPT_FAMILIES):
            matched_aliases: list[str] = []
            evidence_sentences: list[str] = []
            total_alias_hits = 0

            for sentence in sentences:
                normalized_sentence = self._normalize(sentence)
                sentence_aliases = [
                    alias
                    for alias in family.aliases
                    if self._contains_phrase(
                        normalized_sentence,
                        self._normalize(alias),
                    )
                ]

                if not sentence_aliases:
                    continue

                matched_aliases.extend(sentence_aliases)
                total_alias_hits += len(sentence_aliases)
                evidence_sentences.append(sentence)

            matched_aliases = self._unique_strings(matched_aliases)
            evidence_sentences = self._unique_strings(evidence_sentences)

            if (
                len(matched_aliases) < family.minimum_distinct_aliases
                or total_alias_hits < family.minimum_total_hits
            ):
                continue

            # Do not duplicate an official candidate that already uses the
            # same specific terminology. Partial overlap such as array versus
            # ArrayList is deliberately not treated as coverage.
            if self._family_is_officially_covered(
                family=family,
                matched_aliases=matched_aliases,
                official_terms=official_terms,
            ):
                continue

            evidence = evidence_sentences[0]
            evidence_embedding = self._embedding_function(
                [evidence],
                self.config.embedding_model,
                32,
            )
            semantic_score = float(
                evidence_embedding[0] @ family_embeddings[family_index]
            )

            longest_alias_words = max(
                len(self._normalize(alias).split())
                for alias in matched_aliases
            )

            lexical_base = 0.62 if longest_alias_words >= 2 else 0.56
            diversity_bonus = 0.05 * min(3, len(matched_aliases) - 1)
            evidence_bonus = 0.04 * min(2, len(evidence_sentences) - 1)

            score = min(
                0.95,
                lexical_base
                + diversity_bonus
                + evidence_bonus
                + 0.18 * max(0.0, semantic_score),
            )

            method = (
                "lexical_semantic"
                if semantic_score >= self.config.sentence_similarity_threshold
                else "lexical"
            )

            signals.append(
                UnmappedCSSignal(
                    rough_topic=family.rough_topic,
                    domain=family.domain,
                    score=round(score, 4),
                    evidence=evidence.strip(),
                    matched_aliases=matched_aliases,
                    detection_method=method,
                )
            )

        return signals

    def _detect_semantic_residual(
        self,
        sentences: list[str],
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        covered = {
            self._normalize(sentence)
            for candidate in official_candidates
            for sentence in candidate.evidence
            if self._normalize(sentence)
        }

        uncovered_sentences = [
            sentence
            for sentence in sentences
            if self._normalize(sentence) not in covered
        ]

        if not uncovered_sentences:
            return []

        sentence_embeddings = self._embedding_function(
            uncovered_sentences,
            self.config.embedding_model,
            32,
        )
        domain_embeddings = self._get_domain_embeddings()

        similarities = sentence_embeddings @ domain_embeddings.T
        candidates: list[UnmappedCSSignal] = []

        for sentence_index, sentence in enumerate(uncovered_sentences):
            row = similarities[sentence_index]
            domain_index = int(np.argmax(row))
            score = float(row[domain_index])

            if score < self.config.sentence_similarity_threshold:
                continue

            candidates.append(
                UnmappedCSSignal(
                    rough_topic="Unmapped Computer Science content",
                    domain=CS_DOMAINS[domain_index].name,
                    score=round(score, 4),
                    evidence=sentence.strip(),
                    matched_aliases=[],
                    detection_method="semantic",
                )
            )

        candidates.sort(key=lambda signal: signal.score, reverse=True)
        candidates = self._deduplicate(candidates)

        strong_exists = any(
            signal.score >= self.config.strong_sentence_threshold
            for signal in candidates
        )

        if (
            len(candidates) < self.config.minimum_evidence_sentences
            and not strong_exists
        ):
            return []

        return candidates

    def _get_domain_embeddings(self) -> np.ndarray:
        if self._domain_embeddings is None:
            self._domain_embeddings = self._embedding_function(
                [domain.description for domain in CS_DOMAINS],
                self.config.embedding_model,
                32,
            )
        return self._domain_embeddings

    def _get_family_embeddings(self) -> np.ndarray:
        if self._family_embeddings is None:
            self._family_embeddings = self._embedding_function(
                [family.description for family in UNMAPPED_CONCEPT_FAMILIES],
                self.config.embedding_model,
                32,
            )
        return self._family_embeddings

    @classmethod
    def _official_terms(
        cls,
        official_candidates: list[TopicCandidate],
    ) -> set[str]:
        terms: set[str] = set()

        for candidate in official_candidates:
            terms.add(cls._normalize(candidate.topic))
            terms.add(cls._normalize(candidate.official_title))
            terms.update(
                cls._normalize(alias)
                for alias in candidate.matched_aliases
            )

        return {term for term in terms if term}

    @classmethod
    def _family_is_officially_covered(
        cls,
        family: UnmappedConceptFamily,
        matched_aliases: list[str],
        official_terms: set[str],
    ) -> bool:
        family_terms = {
            cls._normalize(family.rough_topic),
            *(
                cls._normalize(alias)
                for alias in matched_aliases
            ),
        }

        return any(
            family_term == official_term
            for family_term in family_terms
            for official_term in official_terms
            if family_term and official_term
        )

    @staticmethod
    def _split_sentences(text: str) -> list[str]:
        text = re.sub(r"\s+", " ", text).strip()
        parts = re.split(r"(?<=[.!?])\s+", text)
        return [part.strip() for part in parts if part.strip()]

    @staticmethod
    def _normalize(text: str) -> str:
        text = re.sub(r"[^a-z0-9]+", " ", text.lower())
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _contains_phrase(text: str, phrase: str) -> bool:
        if not phrase:
            return False

        pattern = re.compile(
            r"(?<!\w)"
            + re.escape(phrase).replace(r"\ ", r"\s+")
            + r"(?!\w)",
            re.IGNORECASE,
        )
        return bool(pattern.search(text))

    @classmethod
    def _deduplicate(
        cls,
        signals: list[UnmappedCSSignal],
    ) -> list[UnmappedCSSignal]:
        output: list[UnmappedCSSignal] = []
        seen: set[tuple[str, str]] = set()

        for signal in signals:
            key = (
                cls._normalize(signal.rough_topic),
                cls._normalize(signal.evidence),
            )
            if not key[0] or not key[1] or key in seen:
                continue
            seen.add(key)
            output.append(signal)

        return output

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = " ".join(value.lower().split())
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique


## 10. Topic merger


In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from statistics import fmean



@dataclass(frozen=True)
class TopicMergeConfig:
    """
    Merge repeated official topics and rank their importance in the lesson.

    Score responsibilities are deliberately separate:

    confidence:
        certainty that a topic exists;

    coverage:
        breadth of the lesson containing the topic;

    ranking:
        estimated lesson importance;

    topic role:
        relative primary/supporting classification based mainly on coverage
        and ranking, never confidence alone.
    """

    max_evidence_per_topic: int = 5

    # Only a new non-adjacent support span adds confidence.
    non_adjacent_span_bonus: float = 0.02
    maximum_support_bonus: float = 0.06
    maximum_merged_confidence: float = 0.95

    # Coverage is weighted most strongly because primary/supporting is about
    # lesson importance, not merely certainty that a topic was mentioned.
    ranking_confidence_weight: float = 0.20
    ranking_semantic_weight: float = 0.20
    ranking_salience_weight: float = 0.20
    ranking_coverage_weight: float = 0.40

    # Relative promotion rules.
    minimum_primary_coverage: float = 0.35
    dominant_coverage_ratio: float = 0.75
    primary_ranking_margin: float = 0.12
    minimum_primary_ranking_score: float = 0.42

    # Role-only teaching signals. These do not change retrieval, candidate
    # extraction, evidence evaluation, ranking scores, or topic merging.
    minimum_primary_teaching_depth: int = 2
    strong_primary_teaching_depth: int = 3
    minimum_strong_teaching_quality: float = 0.68
    strong_teaching_ranking_margin: float = 0.22
    minimum_strong_teaching_ranking_score: float = 0.30

    # Change 4: a topic that opens the lesson and is restated in the recap
    # (present in both the first and last topic-bearing chunk, with the body
    # in between) is the lesson's framing/primary concept even when the middle
    # worked example uses different surface vocabulary. Reuses the existing
    # conservative teaching-depth and lesson-relative ranking gates; adds no
    # new score thresholds.
    bracketing_promotes_primary: bool = True

    def __post_init__(self) -> None:
        weights = (
            self.ranking_confidence_weight,
            self.ranking_semantic_weight,
            self.ranking_salience_weight,
            self.ranking_coverage_weight,
        )

        if abs(sum(weights) - 1.0) > 1e-9:
            raise ValueError(
                "Ranking weights must sum to 1.0."
            )

        probability_fields = (
            "non_adjacent_span_bonus",
            "maximum_support_bonus",
            "maximum_merged_confidence",
            "minimum_primary_coverage",
            "dominant_coverage_ratio",
            "primary_ranking_margin",
            "minimum_primary_ranking_score",
            "minimum_strong_teaching_quality",
            "strong_teaching_ranking_margin",
            "minimum_strong_teaching_ranking_score",
        )

        for field_name in probability_fields:
            value = getattr(self, field_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{field_name} must be between 0 and 1."
                )

        if not 0 <= self.minimum_primary_teaching_depth <= 4:
            raise ValueError(
                "minimum_primary_teaching_depth must be between 0 and 4."
            )

        if not 0 <= self.strong_primary_teaching_depth <= 4:
            raise ValueError(
                "strong_primary_teaching_depth must be between 0 and 4."
            )

        if (
            self.strong_primary_teaching_depth
            < self.minimum_primary_teaching_depth
        ):
            raise ValueError(
                "strong_primary_teaching_depth must be at least "
                "minimum_primary_teaching_depth."
            )


class TopicMerger:
    """
    Merge the same official concept across chunks.

    Consecutive chunk IDs are one support span because they may come from one
    long discussion split by Module 2's size guardrail.

    Roles are assigned only after every merged topic has been scored, allowing
    primary/supporting decisions to be lesson-relative rather than based on
    independent absolute thresholds.
    """

    def __init__(self, config: TopicMergeConfig | None = None) -> None:
        self.config = config or TopicMergeConfig()

    def merge(
        self,
        chunk_results: list[ChunkTopicResult],
    ) -> list[MergedTopic]:
        grouped: dict[
            str,
            list[tuple[int, TopicCandidate]],
        ] = defaultdict(list)

        topic_bearing_chunk_count = max(
            1,
            sum(
                1
                for chunk_result in chunk_results
                if chunk_result.topic_candidates
            ),
        )

        # Lesson-structure signal for role assignment only. The first and
        # last chunks that bear any topic bracket the lesson body.
        topic_bearing_chunk_ids = sorted(
            {
                chunk_result.chunk_id
                for chunk_result in chunk_results
                if chunk_result.topic_candidates
            }
        )
        lesson_first_chunk = (
            topic_bearing_chunk_ids[0]
            if topic_bearing_chunk_ids
            else None
        )
        lesson_last_chunk = (
            topic_bearing_chunk_ids[-1]
            if topic_bearing_chunk_ids
            else None
        )

        for chunk_result in chunk_results:
            for candidate in chunk_result.topic_candidates:
                grouped[candidate.concept_id].append(
                    (chunk_result.chunk_id, candidate)
                )

        provisional_topics: list[MergedTopic] = []
        role_signals: dict[str, dict[str, float | int | bool]] = {}

        for concept_id, occurrences in grouped.items():
            first_candidate = occurrences[0][1]
            source_chunk_ids = sorted(
                {chunk_id for chunk_id, _ in occurrences}
            )

            support_span_count = self._count_contiguous_spans(
                source_chunk_ids
            )

            base_confidence = max(
                candidate.cs_relevance_score
                for _, candidate in occurrences
            )

            support_bonus = min(
                self.config.maximum_support_bonus,
                self.config.non_adjacent_span_bonus
                * max(0, support_span_count - 1),
            )

            confidence = min(
                self.config.maximum_merged_confidence,
                base_confidence + support_bonus,
            )

            mean_semantic_score = fmean(
                candidate.semantic_score
                for _, candidate in occurrences
            )
            mean_keyword_score = fmean(
                candidate.keyword_score
                for _, candidate in occurrences
            )
            mean_salience_score = fmean(
                candidate.salience_score
                for _, candidate in occurrences
            )

            coverage_score = min(
                1.0,
                len(source_chunk_ids) / topic_bearing_chunk_count,
            )

            ranking_score = self._ranking_score(
                confidence=confidence,
                mean_semantic_score=mean_semantic_score,
                mean_salience_score=mean_salience_score,
                coverage_score=coverage_score,
            )

            evidence: list[str] = []
            for _, candidate in occurrences:
                evidence.extend(candidate.evidence)

            # Keep teaching depth/purpose separate from the existing ranking
            # calculation. It is used only when assigning primary/supporting.
            role_signals[concept_id] = {
                "max_teaching_depth": max(
                    candidate.teaching_depth_level
                    for _, candidate in occurrences
                ),
                "mean_evidence_quality": fmean(
                    candidate.evidence_quality_score
                    for _, candidate in occurrences
                ),
                "explicitly_primary": any(
                    self._has_primary_purpose_cue(candidate)
                    for _, candidate in occurrences
                ),
                "explicitly_secondary": any(
                    self._has_secondary_purpose_cue(candidate)
                    for _, candidate in occurrences
                ),
                "brackets_lesson": (
                    lesson_first_chunk is not None
                    and lesson_last_chunk is not None
                    and lesson_last_chunk > lesson_first_chunk
                    and lesson_first_chunk in source_chunk_ids
                    and lesson_last_chunk in source_chunk_ids
                ),
            }

            provisional_topics.append(
                MergedTopic(
                    concept_id=concept_id,
                    topic=first_candidate.topic,
                    domain=first_candidate.domain,
                    official_reference=first_candidate.official_reference,
                    chapter_reference=first_candidate.chapter_reference,
                    official_title=first_candidate.official_title,
                    paper=first_candidate.paper,
                    source_pages=first_candidate.source_pages,
                    confidence=round(confidence, 4),
                    ranking_score=round(ranking_score, 4),
                    topic_role="supporting",
                    source_chunk_ids=source_chunk_ids,
                    support_span_count=support_span_count,
                    mean_semantic_score=round(mean_semantic_score, 4),
                    mean_keyword_score=round(mean_keyword_score, 4),
                    mean_salience_score=round(mean_salience_score, 4),
                    coverage_score=round(coverage_score, 4),
                    evidence=self._unique_strings(evidence)[
                        : self.config.max_evidence_per_topic
                    ],
                    supporting_candidate_count=len(occurrences),
                )
            )

        merged_topics = self._assign_relative_roles(
            provisional_topics,
            role_signals=role_signals,
        )

        role_priority = {
            "primary": 1,
            "supporting": 0,
        }

        merged_topics.sort(
            key=lambda topic: (
                role_priority[topic.topic_role],
                topic.ranking_score,
                topic.coverage_score,
                topic.mean_semantic_score,
                topic.confidence,
            ),
            reverse=True,
        )

        return merged_topics

    def _ranking_score(
        self,
        confidence: float,
        mean_semantic_score: float,
        mean_salience_score: float,
        coverage_score: float,
    ) -> float:
        semantic_component = max(
            0.0,
            min(1.0, mean_semantic_score),
        )

        score = (
            self.config.ranking_confidence_weight * confidence
            + self.config.ranking_semantic_weight * semantic_component
            + self.config.ranking_salience_weight * mean_salience_score
            + self.config.ranking_coverage_weight * coverage_score
        )

        return max(0.0, min(1.0, score))

    def _assign_relative_roles(
        self,
        topics: list[MergedTopic],
        *,
        role_signals: dict[
            str,
            dict[str, float | int | bool],
        ] | None = None,
    ) -> list[MergedTopic]:
        if not topics:
            return []

        role_signals = role_signals or {}

        max_ranking = max(
            topic.ranking_score
            for topic in topics
        )
        max_coverage = max(
            topic.coverage_score
            for topic in topics
        )

        # Change 5: coverage only discriminates importance when topics differ
        # on it. In a single-chunk lesson (or any run where every topic shares
        # the same coverage) coverage is 1.0 for all topics and carries no
        # relative signal, so it must not be used to promote. Primary then has
        # to be earned by teaching depth or an explicit lesson-purpose cue.
        coverage_is_discriminative = (
            len({round(topic.coverage_score, 6) for topic in topics}) > 1
        )

        dominant_coverage_threshold = max(
            self.config.minimum_primary_coverage,
            max_coverage
            * self.config.dominant_coverage_ratio,
        )

        ranking_threshold = max(
            self.config.minimum_primary_ranking_score,
            max_ranking
            - self.config.primary_ranking_margin,
        )

        # Strong, topic-specific teaching may occur in fewer chunks than a
        # contextual/supporting concept. Give such topics a slightly wider
        # lesson-relative ranking window, without changing ranking itself.
        strong_teaching_ranking_threshold = max(
            self.config.minimum_strong_teaching_ranking_score,
            max_ranking
            - self.config.strong_teaching_ranking_margin,
        )

        promoted: list[MergedTopic] = []

        for topic in topics:
            signals = role_signals.get(
                topic.concept_id,
                {},
            )

            teaching_depth = int(
                signals.get("max_teaching_depth", 0)
            )
            teaching_quality = float(
                signals.get("mean_evidence_quality", 0.0)
            )
            explicitly_primary = bool(
                signals.get("explicitly_primary", False)
            )
            explicitly_secondary = bool(
                signals.get("explicitly_secondary", False)
            )
            brackets_lesson = bool(
                signals.get("brackets_lesson", False)
            )

            has_primary_depth = (
                teaching_depth
                >= self.config.minimum_primary_teaching_depth
            )

            has_strong_teaching = (
                teaching_depth
                >= self.config.strong_primary_teaching_depth
                and teaching_quality
                >= self.config.minimum_strong_teaching_quality
            )

            coverage_supported_primary = (
                coverage_is_discriminative
                and topic.coverage_score
                >= dominant_coverage_threshold
                and topic.ranking_score
                >= ranking_threshold
                and has_primary_depth
            )

            teaching_supported_primary = (
                has_strong_teaching
                and topic.ranking_score
                >= strong_teaching_ranking_threshold
            )

            # Role fix: an explicit candidate-specific lesson-purpose cue can
            # promote explanatory teaching even when incidental topics rank
            # higher. It still needs real teaching depth, but lesson intent
            # should not be blocked by a relative ranking threshold.
            purpose_supported_primary = (
                explicitly_primary
                and has_primary_depth
            )

            # Change 4: lesson-framing topics recovered by structure. A topic
            # that brackets the whole lesson (stated at the start and restated
            # in the recap) is a primary subject even if the worked-example
            # body renames it. Gated by real teaching depth (>= the primary
            # teaching-depth floor) and the existing lesson-relative ranking
            # window; introduces no new score thresholds.
            structure_supported_primary = (
                self.config.bracketing_promotes_primary
                and brackets_lesson
                and has_primary_depth
                and topic.ranking_score
                >= strong_teaching_ranking_threshold
            )

            # Explicit secondary-purpose cues still win: a concept described
            # as contextual/not the main focus cannot be promoted solely by
            # coverage, an unrelated lesson-purpose phrase, or lesson framing.
            is_primary = (
                not explicitly_secondary
                and (
                    coverage_supported_primary
                    or teaching_supported_primary
                    or purpose_supported_primary
                    or structure_supported_primary
                )
            )

            promoted.append(
                topic.model_copy(
                    update={
                        "topic_role": (
                            "primary"
                            if is_primary
                            else "supporting"
                        )
                    }
                )
            )

        # A lesson with valid official topics should always have at least one
        # primary topic. Keep the fallback inside role assignment only, but
        # prefer genuine teaching depth before coverage when possible.
        if not any(
            topic.topic_role == "primary"
            for topic in promoted
        ):
            non_secondary_topics = [
                topic
                for topic in promoted
                if not bool(
                    role_signals.get(
                        topic.concept_id,
                        {},
                    ).get("explicitly_secondary", False)
                )
            ]
            fallback_pool = (
                non_secondary_topics
                if non_secondary_topics
                else promoted
            )

            best_topic = max(
                fallback_pool,
                key=lambda topic: (
                    int(
                        bool(
                            role_signals.get(
                                topic.concept_id,
                                {},
                            ).get("explicitly_primary", False)
                        )
                    ),
                    int(
                        role_signals.get(
                            topic.concept_id,
                            {},
                        ).get("max_teaching_depth", 0)
                    ),
                    float(
                        role_signals.get(
                            topic.concept_id,
                            {},
                        ).get("mean_evidence_quality", 0.0)
                    ),
                    topic.ranking_score,
                    topic.coverage_score,
                    topic.mean_salience_score,
                    topic.mean_semantic_score,
                ),
            )

            promoted = [
                (
                    topic.model_copy(
                        update={
                            "topic_role": "primary"
                        }
                    )
                    if topic.concept_id == best_topic.concept_id
                    else topic
                )
                for topic in promoted
            ]

        return promoted

    @staticmethod
    def _has_primary_purpose_cue(
        candidate: TopicCandidate,
    ) -> bool:
        """
        Detect narrow, candidate-specific lesson-purpose language.

        The cue must occur close to this candidate's own topic/alias. This
        prevents a lesson objective for one concept from promoting another
        concept mentioned elsewhere in the same evidence sentence.
        """

        if not candidate.evidence:
            return False

        anchors = {
            " ".join(
                re.sub(
                    r"[^a-z0-9]+",
                    " ",
                    value.lower(),
                ).split()
            )
            for value in (
                [candidate.topic]
                + list(candidate.matched_aliases)
            )
            if value
        }
        anchors.discard("")

        primary_cues = (
            "today we are learning",
            "today we re learning",
            "today we are going to",
            "today we re going to",
            "main reason",
            "main focus",
            "main concept",
            "main concepts",
            "lesson is about",
            "lesson focuses on",
            # Common lecture intro framings (anchored to this candidate's
            # own topic/alias, so they cannot promote an unrelated concept).
            "in this video we are going to",
            "in this video we re going to",
            "in this video we are looking at",
            "in this video we re looking at",
            "in this video we are thinking about",
            "in this video we re thinking about",
            "in this video we will",
            "in this video we ll",
            "we are going to take a look at",
            "we re going to take a look at",
            "we are going to look at",
            "we re going to look at",
            "this video is about",
            "this video covers",
            # Common recap framings.
            "has the following features",
            "have the following features",
        )

        for sentence in candidate.evidence:
            normalized = " ".join(
                re.sub(
                    r"[^a-z0-9]+",
                    " ",
                    sentence.lower(),
                ).split()
            )

            for cue in primary_cues:
                cue_index = normalized.find(cue)
                if cue_index < 0:
                    continue

                window = normalized[
                    max(0, cue_index - 80):
                    min(
                        len(normalized),
                        cue_index + len(cue) + 140,
                    )
                ]

                if any(
                    anchor in window
                    for anchor in anchors
                ):
                    return True

        return False

    @staticmethod
    def _has_secondary_purpose_cue(
        candidate: TopicCandidate,
    ) -> bool:
        """
        Detect direct teacher intent that this specific topic is contextual
        rather than a main lesson target.

        The cue must be attached to one of the candidate's own aliases/topic
        immediately before phrases such as "not today's main target". This
        keeps the check narrow and prevents one topic's disclaimer from
        demoting another topic mentioned later in the same sentence.
        """

        if not candidate.evidence:
            return False

        anchors = {
            " ".join(
                re.sub(
                    r"[^a-z0-9]+",
                    " ",
                    value.lower(),
                ).split()
            )
            for value in (
                [candidate.topic]
                + list(candidate.matched_aliases)
            )
            if value
        }
        anchors.discard("")

        # "Before-type" cues: the teacher names the topic and then disclaims
        # it -> "<topic> is not today's main focus".
        secondary_before_cues = (
            "not today s main target",
            "not the main target",
            "not today s main topic",
            "not the main topic",
            "not today s main focus",
            "not the main focus",
            "not the focus of this lesson",
            "not today s topic",
        )

        # Change 6: "after-type" de-scoping cues, where the topic name follows
        # the cue -> "we are not learning <topic> ...". A short after-window
        # keeps this attached to the immediately named topic, so a topic that
        # is only *contrasted* later in the sentence (e.g. "... to contrast it
        # with <other topic>") is not demoted by another topic's disclaimer.
        secondary_after_cues = (
            "we are not learning",
            "we re not learning",
            "we are not covering",
            "we re not covering",
            "not learning the steps of",
        )

        for sentence in candidate.evidence:
            normalized = " ".join(
                re.sub(
                    r"[^a-z0-9]+",
                    " ",
                    sentence.lower(),
                ).split()
            )

            for cue in secondary_before_cues:
                cue_index = normalized.find(cue)
                if cue_index < 0:
                    continue

                # The teacher normally names the topic immediately before the
                # disclaimer: "<topic> is/are not today's main target".
                before_cue = normalized[
                    max(0, cue_index - 100):cue_index
                ]

                if any(
                    anchor in before_cue
                    for anchor in anchors
                ):
                    return True

            for cue in secondary_after_cues:
                cue_index = normalized.find(cue)
                if cue_index < 0:
                    continue

                after_start = cue_index + len(cue)
                after_cue = normalized[
                    after_start:after_start + 60
                ]

                if any(
                    anchor in after_cue
                    for anchor in anchors
                ):
                    return True

        return False

    @staticmethod
    def _count_contiguous_spans(chunk_ids: list[int]) -> int:
        if not chunk_ids:
            return 0

        spans = 1
        for previous, current in zip(chunk_ids, chunk_ids[1:]):
            if current > previous + 1:
                spans += 1
        return spans

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = " ".join(value.lower().split())
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique


## 11. Complete Module 3 pipeline and fallback rule


In [ ]:
import re
from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any



@dataclass(frozen=True)
class Module3PipelineConfig:
    """
    Decision rules for LLM fallback and continuation handling.

    Strong, specific lexical evidence can be retained directly. Semantic-only
    or ambiguous evidence is escalated because it requires interpretation.
    """

    strong_unmapped_lexical_score: float = 0.70
    strong_unmapped_semantic_score: float = 0.82
    ambiguous_signal_margin: float = 0.04

    # Rejected candidates may still provide continuation evidence for a
    # concept already established in the immediately preceding discussion.
    # They are never allowed to create a new topic instance.
    continuation_support_floor: float = 0.50

    def __post_init__(self) -> None:
        for field_name in (
            "strong_unmapped_lexical_score",
            "strong_unmapped_semantic_score",
            "ambiguous_signal_margin",
            "continuation_support_floor",
        ):
            value = getattr(self, field_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{field_name} must be between 0 and 1."
                )


class Module3TopicPipeline:
    """
    Complete Module 3 pipeline:

    Module 2 chunks
        → official AQA candidate extraction
        → salience-aware filtering
        → evidence-quality and recap evaluation
        → continuation/no-new-topic handling
        → unmapped CS detection
        → official topic merging and lesson-relative role assignment
    """

    def __init__(
        self,
        extractor: TopicCandidateExtractor | None = None,
        relevance_filter: CSRelevanceFilter | None = None,
        unmapped_detector: CSUnmappedDetector | None = None,
        evidence_evaluator: EvidenceQualityEvaluator | None = None,
        merger: TopicMerger | None = None,
        config: Module3PipelineConfig | None = None,
    ) -> None:
        self.extractor = extractor or TopicCandidateExtractor()
        self.relevance_filter = relevance_filter or CSRelevanceFilter()
        self.unmapped_detector = unmapped_detector or CSUnmappedDetector()
        self.evidence_evaluator = (
            evidence_evaluator
            or EvidenceQualityEvaluator()
        )
        self.merger = merger or TopicMerger()
        self.config = config or Module3PipelineConfig()

    def process_chunks(self, chunks: Sequence[Any]) -> Module3Result:
        chunk_results: list[ChunkTopicResult] = []

        for raw_chunk in chunks:
            chunk_id = int(self._get_value(raw_chunk, "chunk_id"))
            text = str(self._get_value(raw_chunk, "text")).strip()

            word_count_value = self._get_optional_value(
                raw_chunk,
                "word_count",
            )
            word_count = (
                int(word_count_value)
                if word_count_value is not None
                else len(re.findall(r"\S+", text))
            )

            overlap_word_count = int(
                self._get_optional_value(
                    raw_chunk,
                    "overlap_word_count",
                )
                or 0
            )

            raw_candidates = self.extractor.extract(
                chunk_id=chunk_id,
                text=text,
            )

            base_result = self.relevance_filter.filter(
                chunk_id=chunk_id,
                source_word_count=word_count,
                candidates=raw_candidates,
            )

            quality_result = self.evidence_evaluator.evaluate(
                text=text,
                candidates=base_result.topic_candidates,
            )

            combined_rejected = (
                list(base_result.rejected_candidates)
                + quality_result.rejected
            )

            retained = quality_result.retained

            adjusted_score = (
                max(
                    candidate.cs_relevance_score
                    for candidate in retained
                )
                if retained
                else 0.0
            )

            quality_notes = list(base_result.notes)
            if quality_result.rejected:
                quality_notes.append(
                    f"Evidence-quality evaluation rejected "
                    f"{len(quality_result.rejected)} candidate(s)."
                )

            base_result = base_result.model_copy(
                update={
                    "topic_candidates": retained,
                    "rejected_candidates": combined_rejected,
                    "is_cs_relevant": bool(retained),
                    "creates_new_topic": bool(retained),
                    "cs_relevance_score": round(adjusted_score, 4),
                    "notes": quality_notes,
                }
            )

            previous_result = chunk_results[-1] if chunk_results else None

            # A chunk created with overlap immediately after a retained CS
            # chunk may simply conclude the same question. It should not be
            # forced into a new topic or sent to the unmapped detector.
            current_topic_ids = {
                candidate.concept_id
                for candidate in base_result.topic_candidates
            }
            previous_topic_ids = self._effective_previous_topic_ids(
                previous_result=previous_result,
                completed_results=chunk_results,
            )

            continuation_only = (
                overlap_word_count > 0
                and previous_result is not None
                and bool(previous_topic_ids)
                and (
                    not current_topic_ids
                    or current_topic_ids.issubset(previous_topic_ids)
                )
            )

            if continuation_only:
                continuation_candidates = list(
                    base_result.topic_candidates
                )

                if not continuation_candidates:
                    continuation_candidates = [
                        candidate.model_copy(
                            update={
                                "cs_relevant": True,
                                "evidence_quality_notes": [
                                    *candidate.evidence_quality_notes,
                                    (
                                        "Retained only as continuation "
                                        "evidence for an already established "
                                        "topic; it cannot create a new topic."
                                    ),
                                ],
                            }
                        )
                        for candidate in combined_rejected
                        if (
                            candidate.concept_id in previous_topic_ids
                            and candidate.cs_relevance_score
                            >= self.config.continuation_support_floor
                            and not candidate.recap_evidence_only
                            and not candidate.comparison_evidence_only
                            and not candidate.context_collision
                        )
                    ]

                continuation_score = (
                    max(
                        candidate.cs_relevance_score
                        for candidate in continuation_candidates
                    )
                    if continuation_candidates
                    else 0.0
                )

                final_result = base_result.model_copy(
                    update={
                        "classification": "continuation_no_new_topic",
                        "is_cs_relevant": False,
                        "creates_new_topic": False,
                        "cs_relevance_score": round(
                            continuation_score, 4
                        ),
                        "topic_candidates": continuation_candidates,
                        "has_unmapped_cs_content": False,
                        "unmapped_cs_signals": [],
                        "continuation_of_chunk_id": (
                            previous_result.continuation_of_chunk_id
                            or previous_result.chunk_id
                        ),
                        "requires_llm_fallback": False,
                        "notes": [
                            (
                                "No new standalone topic was detected; the "
                                "chunk continues the previous overlapped "
                                "discussion. Matching evidence is preserved "
                                "for lesson-level merging."
                            )
                        ],
                    }
                )
                chunk_results.append(final_result)
                continue

            unmapped_signals = self.unmapped_detector.detect(
                text=text,
                official_candidates=base_result.topic_candidates,
            )
            has_unmapped = bool(unmapped_signals)
            has_official = bool(base_result.topic_candidates)

            if has_official and has_unmapped:
                classification = "mixed_official_and_unmapped"
            elif has_official:
                classification = "official_aqa_topic"
            elif has_unmapped:
                classification = "cs_related_unmapped"
            else:
                classification = "no_topic"

            unmapped_requires_llm = (
                self._unmapped_requires_llm_fallback(unmapped_signals)
            )

            requires_llm_fallback = (
                unmapped_requires_llm
                or (
                    base_result.requires_llm_fallback
                    and not has_unmapped
                )
            )

            notes = list(base_result.notes)
            if has_unmapped:
                notes.append(
                    "Every genuine unmapped-CS signal is routed to the "
                    "memory-first resolver: PostgreSQL memory lookup, then "
                    "Qdrant shortlist plus Groq on a memory miss."
                )

            final_score = base_result.cs_relevance_score
            if has_unmapped and not has_official:
                final_score = max(
                    signal.score for signal in unmapped_signals
                )

            final_result = base_result.model_copy(
                update={
                    "classification": classification,
                    "is_cs_relevant": has_official or has_unmapped,
                    "creates_new_topic": has_official or has_unmapped,
                    "cs_relevance_score": round(final_score, 4),
                    "has_unmapped_cs_content": has_unmapped,
                    "unmapped_cs_signals": unmapped_signals,
                    "requires_llm_fallback": requires_llm_fallback,
                    "notes": notes,
                }
            )

            chunk_results.append(final_result)

        merged_topics = self.merger.merge(chunk_results)

        classification_counts = {
            "official_aqa_topic": 0,
            "mixed_official_and_unmapped": 0,
            "cs_related_unmapped": 0,
            "continuation_no_new_topic": 0,
            "no_topic": 0,
        }

        for result in chunk_results:
            classification_counts[result.classification] += 1

        cs_relevant_chunks = sum(
            1 for result in chunk_results if result.is_cs_relevant
        )

        llm_fallback_ids = [
            result.chunk_id
            for result in chunk_results
            if result.requires_llm_fallback
        ]

        return Module3Result(
            chunk_results=chunk_results,
            merged_topics=merged_topics,
            total_chunks=len(chunk_results),
            cs_relevant_chunks=cs_relevant_chunks,
            non_cs_chunks=len(chunk_results) - cs_relevant_chunks,
            official_topic_chunks=classification_counts[
                "official_aqa_topic"
            ],
            mixed_official_unmapped_chunks=classification_counts[
                "mixed_official_and_unmapped"
            ],
            unmapped_cs_chunks=classification_counts[
                "cs_related_unmapped"
            ],
            continuation_chunks=classification_counts[
                "continuation_no_new_topic"
            ],
            no_topic_chunks=classification_counts["no_topic"],
            llm_fallback_chunk_ids=llm_fallback_ids,
            embedding_model=self.extractor.config.embedding_model,
            candidate_keep_threshold=(
                self.relevance_filter.config.candidate_keep_threshold
            ),
        )

    @staticmethod
    def _effective_previous_topic_ids(
        previous_result: ChunkTopicResult | None,
        completed_results: list[ChunkTopicResult],
    ) -> set[str]:
        if previous_result is None:
            return set()

        direct_ids = {
            candidate.concept_id
            for candidate in previous_result.topic_candidates
        }
        if direct_ids:
            return direct_ids

        source_id = previous_result.continuation_of_chunk_id
        if source_id is None:
            return set()

        for result in reversed(completed_results):
            if result.chunk_id != source_id:
                continue

            return {
                candidate.concept_id
                for candidate in result.topic_candidates
            }

        return set()

    def _unmapped_requires_llm_fallback(
        self,
        signals: list[UnmappedCSSignal],
    ) -> bool:
        """
        Route every genuine unmapped-CS signal into the memory-first
        resolution layer.

        This restores the production architecture without changing topic
        extraction, evidence scoring, official-candidate ranking, topic
        merging, or primary/supporting role assignment.
        """
        return bool(signals)

    @staticmethod
    def _normalise_for_decision(text: str) -> str:
        text = re.sub(r"[^a-z0-9]+", " ", text.lower())
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _get_value(item: Any, field_name: str) -> Any:
        if isinstance(item, dict):
            if field_name not in item:
                raise KeyError(f"Missing chunk field: {field_name}")
            return item[field_name]

        if not hasattr(item, field_name):
            raise AttributeError(f"Chunk has no field: {field_name}")
        return getattr(item, field_name)

    @staticmethod
    def _get_optional_value(item: Any, field_name: str) -> Any | None:
        if isinstance(item, dict):
            return item.get(field_name)
        return getattr(item, field_name, None)


## 12. Optimized Module 4 configuration, PostgreSQL mapping memory, and topic human review

The notebook contains the database and memory operations directly; it does not import `app/db/*.py`.

Production order:

```text
Every genuine unmapped CS topic
-> reusable PostgreSQL mapping-memory lookup
-> Qdrant shortlist on a memory miss
-> one batched Groq call for the remaining topic
-> pending row in topic_human_review
-> Approve promotes the row into topic_mapping_memory
```

The PostgreSQL trigger performs the approval-to-memory promotion with
`validation_status='human_corrected'`. Existing approved memory remains reusable.
When PostgreSQL is unavailable, report generation can still complete, but persistent
human approval requires PostgreSQL.


Database compatibility: if an older `topic_human_review` table has a different schema or incompatible constraints, it is preserved as a timestamped `topic_human_review_legacy_*` backup and a canonical review table is created automatically.


In [ ]:

from datetime import datetime

from sqlalchemy import (
    BigInteger,
    Boolean,
    DateTime,
    Float,
    Integer,
    String,
    Text,
    create_engine,
    func,
    select,
    text,
    update,
)
from sqlalchemy.dialects.postgresql import JSONB, insert as pg_insert
from sqlalchemy.orm import (
    DeclarativeBase,
    Mapped,
    Session,
    mapped_column,
    sessionmaker,
)


GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
LLM_QDRANT_TOP_K = int(os.getenv("LLM_UNMAPPED_QDRANT_TOP_K", "8"))
LLM_AUTO_MAP_CONFIDENCE = float(
    os.getenv("LLM_UNMAPPED_AUTO_MAP_CONFIDENCE", "0.70")
)
MODULE4_PROMPT_VERSION = os.getenv(
    "MODULE4_PROMPT_VERSION", "module4-batch-v2-one-call-strict-aqa"
)
MODULE3_LOCAL_MIN_SCORE = float(
    os.getenv("MODULE3_LOCAL_MIN_SCORE", "0.60")
)
MODULE3_LOCAL_MIN_MARGIN = float(
    os.getenv("MODULE3_LOCAL_MIN_MARGIN", "0.08")
)
MODULE4_CONTEXT_LIMIT = int(
    os.getenv("MODULE4_CONTEXT_LIMIT", "1800")
)

# Active official AQA specification used by this Agent 1 catalogue.
# Reviewer-approved memory from another specification version is never reused.
AQA_SPEC_VERSION = os.getenv(
    "AQA_SPEC_VERSION",
    "AQA-8525-v1.2-2022-11-29",
).strip()
if not AQA_SPEC_VERSION:
    raise ValueError("AQA_SPEC_VERSION cannot be empty.")

# Conservative evidence thresholds for persistent missing-topic-label memory.
# Ignore/suppression requires the strongest match because a false positive
# could hide a real future lesson topic.
TOPIC_LABEL_MEMORY_IGNORE_MIN_SIMILARITY = float(
    os.getenv("TOPIC_LABEL_MEMORY_IGNORE_MIN_SIMILARITY", "0.92")
)
TOPIC_LABEL_MEMORY_LABEL_MIN_SIMILARITY = float(
    os.getenv("TOPIC_LABEL_MEMORY_LABEL_MIN_SIMILARITY", "0.88")
)
TOPIC_LABEL_MEMORY_MIN_MARGIN = float(
    os.getenv("TOPIC_LABEL_MEMORY_MIN_MARGIN", "0.04")
)

# The restored production architecture sends a memory miss to the Qdrant
# shortlist + Groq resolver. The previous local Module 3 shortcut remains
# available only as an explicit opt-in compatibility switch.
MODULE4_ALLOW_LOCAL_MODULE3_RESOLUTION = (
    os.getenv(
        "MODULE4_ALLOW_LOCAL_MODULE3_RESOLUTION",
        "0",
    ).strip()
    == "1"
)

# Strict production optimisation:
# all memory-miss topics for one transcript are sent in one array and Groq
# returns one JSON object containing a results array. Missing response items
# remain pending human review; the notebook never makes a second Groq call.
GROQ_OMITTED_ITEM_MAX_RETRIES = 0


groq_api_key = os.getenv("GROQ_API_KEY", "").strip()
groq_client = Groq(api_key=groq_api_key) if groq_api_key else None
CONCEPT_BY_ID = {concept.concept_id: concept for concept in syllabus_concepts}


class Base(DeclarativeBase):
    pass


class TopicMappingMemory(Base):
    """Reusable validated topic-mapping decisions."""

    __tablename__ = "topic_mapping_memory"

    id: Mapped[int] = mapped_column(
        BigInteger, primary_key=True, autoincrement=True
    )
    cache_key: Mapped[str] = mapped_column(
        String(64), nullable=False, unique=True
    )
    normalized_topic: Mapped[str] = mapped_column(Text, nullable=False)
    original_topic: Mapped[str] = mapped_column(Text, nullable=False)
    evidence_hash: Mapped[str] = mapped_column(String(64), nullable=False)
    evidence_text: Mapped[str] = mapped_column(Text, nullable=False)
    candidate_concept_ids: Mapped[list[str]] = mapped_column(
        JSONB, nullable=False, default=list, server_default="[]"
    )
    module3_concept_ids: Mapped[list[str]] = mapped_column(
        JSONB, nullable=False, default=list, server_default="[]"
    )
    decision: Mapped[str] = mapped_column(String(40), nullable=False)
    mapped_concept_id: Mapped[str | None] = mapped_column(Text, nullable=True)
    confidence: Mapped[float] = mapped_column(Float, nullable=False)
    reason: Mapped[str] = mapped_column(Text, nullable=False)
    model_name: Mapped[str] = mapped_column(String(150), nullable=False)
    prompt_version: Mapped[str] = mapped_column(String(50), nullable=False)
    validation_status: Mapped[str] = mapped_column(
        String(30), nullable=False, default="validated", server_default="validated"
    )
    source_transcript: Mapped[str | None] = mapped_column(Text, nullable=True)
    source_chunk_ids: Mapped[list[int]] = mapped_column(
        JSONB, nullable=False, default=list, server_default="[]"
    )
    spec_version: Mapped[str] = mapped_column(
        String(80), nullable=False
    )
    reviewer_approved: Mapped[bool] = mapped_column(
        Boolean, nullable=False, default=False, server_default="false"
    )
    reviewer_reason: Mapped[str | None] = mapped_column(
        Text, nullable=True
    )
    reviewed_by: Mapped[str | None] = mapped_column(
        Text, nullable=True
    )
    reviewed_at: Mapped[datetime | None] = mapped_column(
        DateTime(timezone=True), nullable=True
    )
    hit_count: Mapped[int] = mapped_column(
        Integer, nullable=False, default=0, server_default="0"
    )
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, server_default=func.now()
    )
    updated_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False,
        server_default=func.now(), onupdate=func.now()
    )
    last_used_at: Mapped[datetime | None] = mapped_column(
        DateTime(timezone=True), nullable=True
    )



class TopicLabelMemory(Base):
    """Reviewer-validated evidence memory for generic unmapped CS residue."""

    __tablename__ = "topic_label_memory"

    id: Mapped[int] = mapped_column(
        BigInteger, primary_key=True, autoincrement=True
    )
    memory_key: Mapped[str] = mapped_column(
        String(64), nullable=False, unique=True
    )
    evidence_hash: Mapped[str] = mapped_column(String(64), nullable=False)
    evidence_text: Mapped[str] = mapped_column(Text, nullable=False)
    action: Mapped[str] = mapped_column(String(20), nullable=False)
    assigned_rough_topic: Mapped[str | None] = mapped_column(Text, nullable=True)
    reason: Mapped[str | None] = mapped_column(Text, nullable=True)
    source_transcript: Mapped[str | None] = mapped_column(Text, nullable=True)
    source_chunk_ids: Mapped[list[int]] = mapped_column(
        JSONB, nullable=False, default=list, server_default="[]"
    )
    spec_version: Mapped[str] = mapped_column(String(80), nullable=False)
    reviewer_approved: Mapped[bool] = mapped_column(
        Boolean, nullable=False, default=True, server_default="true"
    )
    validation_status: Mapped[str] = mapped_column(
        String(30), nullable=False, default="validated", server_default="validated"
    )
    reviewed_by: Mapped[str | None] = mapped_column(Text, nullable=True)
    reviewed_at: Mapped[datetime | None] = mapped_column(
        DateTime(timezone=True), nullable=True
    )
    hit_count: Mapped[int] = mapped_column(
        Integer, nullable=False, default=0, server_default="0"
    )
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False, server_default=func.now()
    )
    updated_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), nullable=False,
        server_default=func.now(), onupdate=func.now()
    )
    last_used_at: Mapped[datetime | None] = mapped_column(
        DateTime(timezone=True), nullable=True
    )



def _quote_pg_identifier(value: str) -> str:
    """Quote one PostgreSQL identifier safely."""
    return '"' + str(value).replace('"', '""') + '"'


def ensure_topic_review_schema_and_trigger(engine: Any) -> None:
    """
    Verify the updated topic-memory schema without renaming or dropping data.

    The PostgreSQL migration is intentionally owned outside the notebook. This
    function only verifies the tables/columns that the final Streamlit Module 3
    requires and installs a compatible review trigger. It never renames an
    existing table, deletes rows, or recreates the review queue.
    """

    required_columns = {
        "topic_mapping_memory": {
            "id", "cache_key", "normalized_topic", "original_topic",
            "evidence_hash", "evidence_text", "candidate_concept_ids",
            "module3_concept_ids", "decision", "mapped_concept_id",
            "confidence", "reason", "model_name", "prompt_version",
            "validation_status", "source_transcript", "source_chunk_ids",
            "spec_version", "reviewer_approved", "reviewer_reason",
            "reviewed_by", "reviewed_at", "hit_count", "created_at",
            "updated_at", "last_used_at",
        },
        "topic_label_memory": {
            "id", "memory_key", "evidence_hash", "evidence_text",
            "action", "assigned_rough_topic", "reason",
            "source_transcript", "source_chunk_ids", "spec_version",
            "reviewer_approved", "validation_status", "reviewed_by",
            "reviewed_at", "hit_count", "created_at", "updated_at",
            "last_used_at",
        },
        "topic_human_review": {
            "id", "cache_key", "normalized_topic", "original_topic",
            "evidence_hash", "evidence_text", "source_transcript",
            "source_chunk_ids", "memory_lookup_result", "memory_source_id",
            "candidate_concept_ids", "qdrant_candidates",
            "proposed_decision", "proposed_mapped_concept_id", "confidence",
            "confidence_band", "reason", "model_name", "prompt_version",
            "status", "corrected_decision", "corrected_mapped_concept_id",
            "correction_reason", "review_notes", "reviewed_by",
            "reviewed_at", "spec_version", "created_at", "updated_at",
        },
        "topic_mapping_decision_log": {
            "id", "review_id", "memory_id", "source_memory_id",
            "pipeline_run_id", "cache_key", "normalized_topic",
            "source_transcript", "source_chunk_ids", "decision_stage",
            "actor_type", "action", "decision", "mapped_concept_id",
            "confidence", "reason", "decided_by", "details",
            "spec_version", "created_at",
        },
    }

    with engine.begin() as connection:
        for table_name, expected in required_columns.items():
            rows = connection.execute(
                text(
                    """
                    SELECT column_name
                    FROM information_schema.columns
                    WHERE table_schema = current_schema()
                      AND table_name = :table_name
                    """
                ),
                {"table_name": table_name},
            ).scalars().all()

            existing = {str(value) for value in rows}
            if not existing:
                raise RuntimeError(
                    f"Required PostgreSQL table {table_name!r} is missing. "
                    "Run the Agent 1 topic-memory migration in pgAdmin first."
                )

            missing = sorted(expected - existing)
            if missing:
                raise RuntimeError(
                    f"PostgreSQL table {table_name!r} is missing updated "
                    f"columns: {missing}. Run the latest Agent 1 migration."
                )

        # The stable review identity includes spec_version so the same
        # transcript/topic can be reviewed again after a syllabus update.
        connection.execute(
            text(
                """
                DROP INDEX IF EXISTS uq_topic_human_review_transcript_topic
                """
            )
        )
        connection.execute(
            text(
                """
                CREATE UNIQUE INDEX IF NOT EXISTS
                uq_topic_human_review_transcript_topic_spec
                ON topic_human_review(
                    source_transcript,
                    normalized_topic,
                    spec_version
                )
                WHERE source_transcript IS NOT NULL
                """
            )
        )

        # Remove only the old incompatible trigger. No table/data is touched.
        connection.execute(
            text(
                """
                DROP TRIGGER IF EXISTS
                trg_promote_approved_topic_review_to_memory
                ON topic_human_review
                """
            )
        )

        # Human review remains DB-enforced. Approve/Correct promote only a
        # reviewer-approved row for the same AQA specification. Reject never
        # creates reusable memory. Human actions are appended to the audit log.
        connection.execute(
            text(
                r"""
                CREATE OR REPLACE FUNCTION
                apply_topic_review_decision_v2()
                RETURNS TRIGGER
                LANGUAGE plpgsql
                AS $$
                DECLARE
                    final_decision TEXT;
                    final_mapped_concept_id TEXT;
                    final_reason TEXT;
                    final_validation_status TEXT;
                    promoted_memory_id BIGINT;
                    human_action TEXT;
                BEGIN
                    IF TG_OP = 'UPDATE' THEN
                        IF NEW.status IS NOT DISTINCT FROM OLD.status THEN
                            NEW.updated_at := NOW();
                            RETURN NEW;
                        END IF;
                    END IF;

                    IF NEW.status NOT IN ('approved', 'corrected', 'rejected') THEN
                        NEW.updated_at := NOW();
                        RETURN NEW;
                    END IF;

                    NEW.reviewed_at := COALESCE(NEW.reviewed_at, NOW());
                    NEW.updated_at := NOW();

                    IF NEW.status = 'rejected' THEN
                        INSERT INTO topic_mapping_decision_log (
                            review_id,
                            memory_id,
                            source_memory_id,
                            pipeline_run_id,
                            cache_key,
                            normalized_topic,
                            source_transcript,
                            source_chunk_ids,
                            decision_stage,
                            actor_type,
                            action,
                            decision,
                            mapped_concept_id,
                            confidence,
                            reason,
                            decided_by,
                            details,
                            spec_version,
                            created_at
                        ) VALUES (
                            NEW.id,
                            NULL,
                            NULL,
                            NULL,
                            NEW.cache_key,
                            NEW.normalized_topic,
                            NEW.source_transcript,
                            COALESCE(NEW.source_chunk_ids, '[]'::jsonb),
                            'human_review',
                            'human',
                            'reject',
                            NEW.proposed_decision,
                            NEW.proposed_mapped_concept_id,
                            NEW.confidence,
                            COALESCE(NULLIF(NEW.review_notes, ''), NEW.reason),
                            NEW.reviewed_by,
                            jsonb_build_object('review_status', NEW.status),
                            NEW.spec_version,
                            NOW()
                        );
                        RETURN NEW;
                    END IF;

                    IF NEW.status = 'corrected' THEN
                        IF NEW.corrected_decision IS NULL
                           OR NEW.correction_reason IS NULL
                           OR LENGTH(TRIM(NEW.correction_reason)) = 0 THEN
                            RAISE EXCEPTION
                                'Corrected review % requires corrected_decision and correction_reason',
                                NEW.id;
                        END IF;

                        final_decision := NEW.corrected_decision;
                        final_mapped_concept_id := NEW.corrected_mapped_concept_id;
                        final_reason := NEW.correction_reason;
                        final_validation_status := 'human_corrected';
                        human_action := 'correct';
                    ELSE
                        final_decision := NEW.proposed_decision;
                        final_mapped_concept_id := NEW.proposed_mapped_concept_id;
                        final_reason := COALESCE(
                            NULLIF(NEW.review_notes, ''),
                            NEW.reason,
                            'Human-approved topic mapping.'
                        );
                        final_validation_status := 'validated';
                        human_action := 'approve';
                    END IF;

                    IF final_decision = 'out_of_syllabus' THEN
                        final_mapped_concept_id := NULL;
                    ELSIF final_decision IN ('mapped', 'resolved_by_module3') THEN
                        IF final_mapped_concept_id IS NULL THEN
                            RAISE EXCEPTION
                                'Review % cannot be promoted without a mapped concept',
                                NEW.id;
                        END IF;
                    ELSE
                        RAISE EXCEPTION
                            'Review % cannot promote decision %',
                            NEW.id,
                            final_decision;
                    END IF;

                    INSERT INTO topic_mapping_decision_log (
                        review_id,
                        memory_id,
                        source_memory_id,
                        pipeline_run_id,
                        cache_key,
                        normalized_topic,
                        source_transcript,
                        source_chunk_ids,
                        decision_stage,
                        actor_type,
                        action,
                        decision,
                        mapped_concept_id,
                        confidence,
                        reason,
                        decided_by,
                        details,
                        spec_version,
                        created_at
                    ) VALUES (
                        NEW.id,
                        NULL,
                        NULL,
                        NULL,
                        NEW.cache_key,
                        NEW.normalized_topic,
                        NEW.source_transcript,
                        COALESCE(NEW.source_chunk_ids, '[]'::jsonb),
                        'human_review',
                        'human',
                        human_action,
                        final_decision,
                        final_mapped_concept_id,
                        NEW.confidence,
                        final_reason,
                        NEW.reviewed_by,
                        jsonb_build_object('review_status', NEW.status),
                        NEW.spec_version,
                        NOW()
                    );

                    INSERT INTO topic_mapping_memory (
                        cache_key,
                        normalized_topic,
                        original_topic,
                        evidence_hash,
                        evidence_text,
                        candidate_concept_ids,
                        module3_concept_ids,
                        decision,
                        mapped_concept_id,
                        confidence,
                        reason,
                        model_name,
                        prompt_version,
                        validation_status,
                        source_transcript,
                        source_chunk_ids,
                        spec_version,
                        reviewer_approved,
                        reviewer_reason,
                        reviewed_by,
                        reviewed_at,
                        hit_count,
                        created_at,
                        updated_at
                    ) VALUES (
                        NEW.cache_key,
                        NEW.normalized_topic,
                        NEW.original_topic,
                        NEW.evidence_hash,
                        NEW.evidence_text,
                        COALESCE(NEW.candidate_concept_ids, '[]'::jsonb),
                        COALESCE(NEW.module3_concept_ids, '[]'::jsonb),
                        final_decision,
                        final_mapped_concept_id,
                        COALESCE(NEW.confidence, 0),
                        final_reason,
                        COALESCE(NEW.model_name, 'human_review'),
                        COALESCE(NEW.prompt_version, 'human-review-v2'),
                        final_validation_status,
                        NEW.source_transcript,
                        COALESCE(NEW.source_chunk_ids, '[]'::jsonb),
                        NEW.spec_version,
                        TRUE,
                        CASE
                            WHEN NEW.status = 'corrected' THEN final_reason
                            ELSE NULL
                        END,
                        COALESCE(NEW.reviewed_by, 'human_review'),
                        NEW.reviewed_at,
                        0,
                        NOW(),
                        NOW()
                    )
                    ON CONFLICT (cache_key)
                    DO UPDATE SET
                        normalized_topic = EXCLUDED.normalized_topic,
                        original_topic = EXCLUDED.original_topic,
                        evidence_hash = EXCLUDED.evidence_hash,
                        evidence_text = EXCLUDED.evidence_text,
                        candidate_concept_ids = EXCLUDED.candidate_concept_ids,
                        module3_concept_ids = EXCLUDED.module3_concept_ids,
                        decision = EXCLUDED.decision,
                        mapped_concept_id = EXCLUDED.mapped_concept_id,
                        confidence = EXCLUDED.confidence,
                        reason = EXCLUDED.reason,
                        model_name = EXCLUDED.model_name,
                        prompt_version = EXCLUDED.prompt_version,
                        validation_status = EXCLUDED.validation_status,
                        source_transcript = EXCLUDED.source_transcript,
                        source_chunk_ids = EXCLUDED.source_chunk_ids,
                        spec_version = EXCLUDED.spec_version,
                        reviewer_approved = TRUE,
                        reviewer_reason = EXCLUDED.reviewer_reason,
                        reviewed_by = EXCLUDED.reviewed_by,
                        reviewed_at = EXCLUDED.reviewed_at,
                        updated_at = NOW()
                    RETURNING id INTO promoted_memory_id;

                    INSERT INTO topic_mapping_decision_log (
                        review_id,
                        memory_id,
                        source_memory_id,
                        pipeline_run_id,
                        cache_key,
                        normalized_topic,
                        source_transcript,
                        source_chunk_ids,
                        decision_stage,
                        actor_type,
                        action,
                        decision,
                        mapped_concept_id,
                        confidence,
                        reason,
                        decided_by,
                        details,
                        spec_version,
                        created_at
                    ) VALUES (
                        NEW.id,
                        promoted_memory_id,
                        NULL,
                        NULL,
                        NEW.cache_key,
                        NEW.normalized_topic,
                        NEW.source_transcript,
                        COALESCE(NEW.source_chunk_ids, '[]'::jsonb),
                        'memory_promotion',
                        'system',
                        'promoted',
                        final_decision,
                        final_mapped_concept_id,
                        NEW.confidence,
                        'Human-reviewed mapping promoted to reusable memory.',
                        NEW.reviewed_by,
                        jsonb_build_object('review_status', NEW.status),
                        NEW.spec_version,
                        NOW()
                    );

                    RETURN NEW;
                END;
                $$
                """
            )
        )

        connection.execute(
            text(
                """
                DROP TRIGGER IF EXISTS trg_apply_topic_review_decision_v2
                ON topic_human_review
                """
            )
        )
        connection.execute(
            text(
                """
                CREATE TRIGGER trg_apply_topic_review_decision_v2
                BEFORE INSERT OR UPDATE OF status
                ON topic_human_review
                FOR EACH ROW
                EXECUTE FUNCTION apply_topic_review_decision_v2()
                """
            )
        )

    print(
        "PostgreSQL topic-memory schema verified non-destructively; "
        "review trigger v2 is ready."
    )

def normalize_database_url(url: str) -> str:
    url = url.strip()
    if url.startswith("postgres://"):
        return "postgresql+psycopg://" + url[len("postgres://"):]
    if url.startswith("postgresql://"):
        return "postgresql+psycopg://" + url[len("postgresql://"):]
    return url


DATABASE_URL = normalize_database_url(os.getenv("DATABASE_URL", ""))
POSTGRES_MEMORY_ENABLED = bool(DATABASE_URL) and not SKIP_EXTERNAL_SERVICES
POSTGRES_MEMORY_ERROR: str | None = None
_engine = None
_SessionFactory = None

if POSTGRES_MEMORY_ENABLED:
    try:
        _engine = create_engine(
            DATABASE_URL,
            pool_pre_ping=True,
            future=True,
        )
        Base.metadata.create_all(_engine)
        ensure_topic_review_schema_and_trigger(_engine)
        _SessionFactory = sessionmaker(
            bind=_engine,
            autoflush=False,
            expire_on_commit=False,
            class_=Session,
        )
        with _engine.connect() as connection:
            connection.execute(select(1))
        print("PostgreSQL topic_mapping_memory: connected")
    except Exception as error:
        POSTGRES_MEMORY_ENABLED = False
        POSTGRES_MEMORY_ERROR = f"{type(error).__name__}: {error}"
        print(
            "PostgreSQL topic memory unavailable; the existing Qdrant "
            "flow will be used."
        )
        print(POSTGRES_MEMORY_ERROR)
else:
    print(
        "PostgreSQL topic memory not configured; the existing Qdrant "
        "flow will be used."
    )


LOCAL_MEMORY_PATH = OUTPUT_DIR / "_module3_topic_mapping_memory.json"


def _load_local_memory() -> dict[str, dict[str, Any]]:
    if not LOCAL_MEMORY_PATH.exists():
        return {}
    try:
        data = json.loads(LOCAL_MEMORY_PATH.read_text(encoding="utf-8"))
        return data if isinstance(data, dict) else {}
    except (OSError, json.JSONDecodeError):
        return {}


def _save_local_memory(data: dict[str, dict[str, Any]]) -> None:
    LOCAL_MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCAL_MEMORY_PATH.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def _normalise_topic_label_memory_evidence(value: str) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip().casefold()


def _topic_label_memory_hash(value: str) -> str:
    return hashlib.sha256(
        _normalise_topic_label_memory_evidence(value).encode("utf-8")
    ).hexdigest()


def _topic_label_memory_outcome(record: Any) -> tuple[str, str | None]:
    return (
        str(record.action or "").casefold(),
        str(record.assigned_rough_topic or "").strip().casefold() or None,
    )


def topic_label_memory_lookup(
    *,
    evidence_text: str,
    transcript_name: str | None = None,
    source_chunk_ids: list[int] | None = None,
) -> dict[str, Any] | None:
    """Reuse only strongly compatible reviewer-approved label/ignore memory."""

    evidence_text = str(evidence_text or "").strip()
    if (
        not evidence_text
        or not POSTGRES_MEMORY_ENABLED
        or _SessionFactory is None
    ):
        return None

    evidence_hash = _topic_label_memory_hash(evidence_text)

    with _SessionFactory() as session:
        exact = session.execute(
            select(TopicLabelMemory)
            .where(
                TopicLabelMemory.evidence_hash == evidence_hash,
                TopicLabelMemory.spec_version == AQA_SPEC_VERSION,
                TopicLabelMemory.reviewer_approved.is_(True),
                TopicLabelMemory.validation_status == "validated",
            )
            .order_by(
                TopicLabelMemory.updated_at.desc(),
                TopicLabelMemory.id.desc(),
            )
            .limit(1)
        ).scalar_one_or_none()

        similarity = 1.0
        record = exact

        if record is None:
            records = session.execute(
                select(TopicLabelMemory)
                .where(
                    TopicLabelMemory.spec_version == AQA_SPEC_VERSION,
                    TopicLabelMemory.reviewer_approved.is_(True),
                    TopicLabelMemory.validation_status == "validated",
                )
                .order_by(
                    TopicLabelMemory.updated_at.desc(),
                    TopicLabelMemory.id.desc(),
                )
                .limit(250)
            ).scalars().all()

            if not records:
                return None

            vectors = embed_texts(
                [evidence_text, *[str(item.evidence_text) for item in records]]
            )
            if vectors.ndim != 2 or vectors.shape[0] != len(records) + 1:
                return None

            scores = np.asarray(vectors[1:] @ vectors[0], dtype=float)
            ranked = sorted(
                zip(records, scores.tolist()),
                key=lambda item: item[1],
                reverse=True,
            )
            record, similarity = ranked[0]

            threshold = (
                TOPIC_LABEL_MEMORY_IGNORE_MIN_SIMILARITY
                if str(record.action).casefold() == "ignore"
                else TOPIC_LABEL_MEMORY_LABEL_MIN_SIMILARITY
            )
            if float(similarity) < threshold:
                return None

            if len(ranked) > 1:
                second_record, second_score = ranked[1]
                if (
                    float(similarity) - float(second_score)
                    < TOPIC_LABEL_MEMORY_MIN_MARGIN
                    and _topic_label_memory_outcome(record)
                    != _topic_label_memory_outcome(second_record)
                ):
                    # Conflicting human memories are too close: ask again
                    # rather than suppressing or relabelling incorrectly.
                    return None

        record.hit_count += 1
        record.last_used_at = datetime.now().astimezone()
        session.commit()
        session.refresh(record)

        result = {
            "id": int(record.id),
            "action": str(record.action).casefold(),
            "assigned_rough_topic": record.assigned_rough_topic,
            "reason": record.reason,
            "similarity": round(float(similarity), 4),
            "spec_version": record.spec_version,
        }

    if transcript_name and _engine is not None:
        try:
            final_decision = (
                "no_additional_topic"
                if result["action"] == "ignore"
                else "assigned_topic_label"
            )
            with _engine.begin() as connection:
                connection.execute(
                    text(
                        """
                        INSERT INTO topic_mapping_decision_log (
                            pipeline_run_id,
                            normalized_topic,
                            source_transcript,
                            source_chunk_ids,
                            decision_stage,
                            actor_type,
                            action,
                            decision,
                            reason,
                            decided_by,
                            details,
                            spec_version
                        )
                        VALUES (
                            NULL,
                            :normalized_topic,
                            :source_transcript,
                            CAST(:source_chunk_ids AS jsonb),
                            'topic_label_memory',
                            'system',
                            'reuse',
                            :decision,
                            :reason,
                            'system',
                            CAST(:details AS jsonb),
                            :spec_version
                        )
                        """
                    ),
                    {
                        "normalized_topic": (
                            str(result.get("assigned_rough_topic") or "unmapped computer science content")
                            .strip()
                            .casefold()
                        ),
                        "source_transcript": transcript_name,
                        "source_chunk_ids": json.dumps(source_chunk_ids or []),
                        "decision": final_decision,
                        "reason": result.get("reason"),
                        "details": json.dumps(
                            {
                                "topic_label_memory_id": result["id"],
                                "similarity": result["similarity"],
                                "memory_action": result["action"],
                                "assigned_rough_topic": result.get("assigned_rough_topic"),
                            },
                            ensure_ascii=False,
                        ),
                        "spec_version": AQA_SPEC_VERSION,
                    },
                )
        except Exception as error:
            print(
                "Topic-label memory reuse audit failed without blocking Module 3: "
                f"{type(error).__name__}: {error}"
            )

    return result


def memory_get_and_mark_used(cache_key: str) -> Any | None:
    """Return a reusable record, preferring PostgreSQL."""
    if POSTGRES_MEMORY_ENABLED and _SessionFactory is not None:
        with _SessionFactory() as session:
            record = session.execute(
                select(TopicMappingMemory).where(
                    TopicMappingMemory.cache_key == cache_key,
                    TopicMappingMemory.spec_version == AQA_SPEC_VERSION,
                    TopicMappingMemory.reviewer_approved.is_(True),
                    TopicMappingMemory.validation_status.in_(
                        ("validated", "human_corrected")
                    ),
                )
            ).scalar_one_or_none()

            if record is None:
                return None

            record.hit_count += 1
            record.last_used_at = datetime.now().astimezone()
            session.commit()
            session.refresh(record)
            return record

    # Updated architecture: do not reuse local JSON as trusted memory.
    return None



def memory_get_by_topic_evidence(
    *,
    normalized_topic: str,
    evidence_hash: str,
) -> Any | None:
    """
    Perform the architecture's first lookup before Qdrant or Groq.

    Only validated or human-corrected records are reusable. The newest
    matching decision is preferred when older cache identities also exist.
    """

    if POSTGRES_MEMORY_ENABLED and _SessionFactory is not None:
        with _SessionFactory() as session:
            record = session.execute(
                select(TopicMappingMemory)
                .where(
                    TopicMappingMemory.normalized_topic
                    == normalized_topic,
                    TopicMappingMemory.evidence_hash
                    == evidence_hash,
                    TopicMappingMemory.spec_version == AQA_SPEC_VERSION,
                    TopicMappingMemory.reviewer_approved.is_(True),
                    TopicMappingMemory.validation_status.in_(
                        ("validated", "human_corrected")
                    ),
                )
                .order_by(
                    TopicMappingMemory.updated_at.desc(),
                    TopicMappingMemory.id.desc(),
                )
                .limit(1)
            ).scalar_one_or_none()

            if record is None:
                return None

            record.hit_count += 1
            record.last_used_at = datetime.now().astimezone()
            session.commit()
            session.refresh(record)
            return record

    # Updated architecture: PostgreSQL is the only reusable mapping memory.
    return None



def memory_upsert_mapping(**values: Any) -> None:
    """Insert or update one validated mapping in the active memory backend."""
    if POSTGRES_MEMORY_ENABLED and _SessionFactory is not None:
        with _SessionFactory() as session:
            statement = pg_insert(TopicMappingMemory).values(**values)
            update_values = {
                key: getattr(statement.excluded, key)
                for key in values
                if key != "cache_key"
            }
            update_values["updated_at"] = func.now()
            statement = statement.on_conflict_do_update(
                index_elements=[TopicMappingMemory.cache_key],
                set_=update_values,
            )
            session.execute(statement)
            session.commit()
        return

    memory = _load_local_memory()
    existing = memory.get(values["cache_key"], {})
    serializable = {
        **existing,
        **values,
        "hit_count": int(existing.get("hit_count", 0)),
        "created_at": existing.get(
            "created_at", datetime.now().astimezone().isoformat()
        ),
        "updated_at": datetime.now().astimezone().isoformat(),
        "last_used_at": existing.get("last_used_at"),
    }
    memory[values["cache_key"]] = serializable
    _save_local_memory(memory)




def topic_review_upsert(
    *,
    cache_key: str,
    normalized_topic: str,
    original_topic: str,
    evidence_hash: str,
    evidence_text: str,
    candidate_concept_ids: list[str],
    qdrant_candidates: list[dict[str, Any]],
    module3_concept_ids: list[str],
    proposed_decision: str,
    proposed_mapped_concept_id: str | None,
    confidence: float,
    reason: str,
    model_name: str,
    prompt_version: str,
    source_transcript: str,
    source_chunk_ids: list[int],
) -> tuple[int | None, str]:
    """
    Insert one new topic-review proposal only when this transcript has not
    already produced the same normalized rough topic.

    Stable identity:
        (source_transcript, normalized_topic)

    Re-running the same transcript therefore returns the original pending,
    approved, or rejected review row without changing it. A genuinely new
    rough topic from that transcript still creates a new row.
    """

    if not POSTGRES_MEMORY_ENABLED or _engine is None:
        return None, "pending"

    values = {
        "cache_key": cache_key,
        "normalized_topic": normalized_topic,
        "original_topic": original_topic,
        "evidence_hash": evidence_hash,
        "evidence_text": evidence_text,
        "candidate_concept_ids": json.dumps(candidate_concept_ids),
        "qdrant_candidates": json.dumps(qdrant_candidates),
        "module3_concept_ids": json.dumps(module3_concept_ids),
        "memory_lookup_result": "miss",
        "confidence_band": (
            "high" if float(confidence) >= 0.80
            else "medium" if float(confidence) >= 0.60
            else "low"
        ),
        "spec_version": AQA_SPEC_VERSION,
        "proposed_decision": proposed_decision,
        "proposed_mapped_concept_id": proposed_mapped_concept_id,
        "confidence": float(confidence),
        "reason": reason,
        "model_name": model_name,
        "prompt_version": prompt_version,
        "source_transcript": source_transcript,
        "source_chunk_ids": json.dumps(source_chunk_ids),
    }

    insert_statement = text(
        """
        INSERT INTO topic_human_review (
            cache_key,
            normalized_topic,
            original_topic,
            evidence_hash,
            evidence_text,
            candidate_concept_ids,
            qdrant_candidates,
            module3_concept_ids,
            memory_lookup_result,
            proposed_decision,
            proposed_mapped_concept_id,
            confidence,
            confidence_band,
            reason,
            model_name,
            prompt_version,
            source_transcript,
            source_chunk_ids,
            spec_version,
            status,
            created_at,
            updated_at
        )
        VALUES (
            :cache_key,
            :normalized_topic,
            :original_topic,
            :evidence_hash,
            :evidence_text,
            CAST(:candidate_concept_ids AS JSONB),
            CAST(:qdrant_candidates AS JSONB),
            CAST(:module3_concept_ids AS JSONB),
            :memory_lookup_result,
            :proposed_decision,
            :proposed_mapped_concept_id,
            :confidence,
            :confidence_band,
            :reason,
            :model_name,
            :prompt_version,
            :source_transcript,
            CAST(:source_chunk_ids AS JSONB),
            :spec_version,
            'pending',
            NOW(),
            NOW()
        )
        ON CONFLICT DO NOTHING
        RETURNING id, status
        """
    )

    existing_statement = text(
        """
        SELECT id, status
        FROM topic_human_review
        WHERE source_transcript = :source_transcript
          AND normalized_topic = :normalized_topic
          AND spec_version = :spec_version
        ORDER BY
            CASE status
                WHEN 'corrected' THEN 0
                WHEN 'approved' THEN 1
                WHEN 'pending' THEN 2
                WHEN 'rejected' THEN 3
                ELSE 4
            END,
            id ASC
        LIMIT 1
        """
    )

    cache_fallback_statement = text(
        """
        SELECT id, status
        FROM topic_human_review
        WHERE cache_key = :cache_key
        ORDER BY id ASC
        LIMIT 1
        """
    )

    with _engine.begin() as connection:
        inserted = connection.execute(
            insert_statement,
            values,
        ).mappings().one_or_none()

        if inserted is not None:
            return int(inserted["id"]), str(inserted["status"])

        existing = connection.execute(
            existing_statement,
            {
                "source_transcript": source_transcript,
                "normalized_topic": normalized_topic,
                "spec_version": AQA_SPEC_VERSION,
            },
        ).mappings().one_or_none()

        if existing is None:
            # Defensive fallback for a cache-key collision originating from a
            # different transcript. Normal transcript reruns resolve through
            # the stable transcript/topic identity above.
            existing = connection.execute(
                cache_fallback_statement,
                {"cache_key": cache_key},
            ).mappings().one_or_none()

    if existing is None:
        raise RuntimeError(
            "Topic review was not inserted and no existing review row "
            "could be recovered."
        )

    return int(existing["id"]), str(existing["status"])


def topic_review_get_for_transcript_topic(
    *,
    transcript_name: str,
    normalized_topic: str,
) -> dict[str, Any] | None:
    """Return an existing review for this transcript/topic/spec identity."""

    if not POSTGRES_MEMORY_ENABLED or _engine is None:
        return None

    statement = text(
        """
        SELECT
            id,
            cache_key,
            normalized_topic,
            original_topic,
            proposed_decision,
            proposed_mapped_concept_id,
            corrected_decision,
            corrected_mapped_concept_id,
            correction_reason,
            confidence,
            reason,
            model_name,
            source_transcript,
            source_chunk_ids,
            status,
            reviewed_by,
            reviewed_at,
            spec_version,
            created_at,
            updated_at
        FROM topic_human_review
        WHERE source_transcript = :source_transcript
          AND normalized_topic = :normalized_topic
          AND spec_version = :spec_version
        ORDER BY
            CASE status
                WHEN 'corrected' THEN 0
                WHEN 'approved' THEN 1
                WHEN 'pending' THEN 2
                WHEN 'rejected' THEN 3
                ELSE 4
            END,
            id ASC
        LIMIT 1
        """
    )

    with _engine.connect() as connection:
        row = connection.execute(
            statement,
            {
                "source_transcript": transcript_name,
                "normalized_topic": normalized_topic,
                "spec_version": AQA_SPEC_VERSION,
            },
        ).mappings().one_or_none()

    return dict(row) if row is not None else None

def topic_review_list_for_transcript(
    transcript_name: str,
) -> list[dict[str, Any]]:
    """Return source-aware topic-review records for Streamlit JSON output."""

    if not POSTGRES_MEMORY_ENABLED or _engine is None:
        return []

    statement = text(
        """
        SELECT
            selected.id,
            selected.cache_key,
            selected.rough_topic,
            selected.decision,
            selected.mapped_concept_id,
            selected.candidate_concept_ids,
            selected.qdrant_candidates,
            selected.corrected_decision,
            selected.corrected_mapped_concept_id,
            selected.correction_reason,
            selected.confidence,
            selected.reason,
            selected.model,
            selected.source_transcript,
            selected.source_chunk_ids,
            selected.status,
            selected.reviewed_by,
            selected.reviewed_at,
            selected.spec_version,
            selected.created_at,
            selected.updated_at
        FROM (
            SELECT DISTINCT ON (normalized_topic)
                id,
                cache_key,
                normalized_topic,
                original_topic AS rough_topic,
                proposed_decision AS decision,
                proposed_mapped_concept_id AS mapped_concept_id,
                candidate_concept_ids,
                qdrant_candidates,
                corrected_decision,
                corrected_mapped_concept_id,
                correction_reason,
                confidence,
                reason,
                model_name AS model,
                source_transcript,
                source_chunk_ids,
                status,
                reviewed_by,
                reviewed_at,
                spec_version,
                created_at,
                updated_at
            FROM topic_human_review
            WHERE source_transcript = :source_transcript
              AND spec_version = :spec_version
            ORDER BY
                normalized_topic,
                CASE status
                    WHEN 'corrected' THEN 0
                    WHEN 'approved' THEN 1
                    WHEN 'pending' THEN 2
                    WHEN 'rejected' THEN 3
                    ELSE 4
                END,
                id ASC
        ) AS selected
        ORDER BY selected.id
        """
    )

    with _engine.connect() as connection:
        rows = connection.execute(
            statement,
            {
                "source_transcript": transcript_name,
                "spec_version": AQA_SPEC_VERSION,
            },
        ).mappings().all()

    results: list[dict[str, Any]] = []

    for row in rows:
        item = dict(row)
        status = str(item.get("status") or "pending").casefold()
        concept_id = (
            item.get("corrected_mapped_concept_id")
            if status == "corrected"
            else item.get("mapped_concept_id")
        )
        concept = CONCEPT_BY_ID.get(concept_id) if concept_id else None
        item["mapped_topic"] = concept.label if concept else None
        item["official_reference"] = (
            concept.official_reference if concept else None
        )
        item["chapter_reference"] = (
            concept.chapter_reference if concept else None
        )
        item["resolution_source"] = (
            "memory" if status in {"approved", "corrected"} else "llm"
        )

        for key, value in list(item.items()):
            if isinstance(value, datetime):
                item[key] = value.isoformat()

        results.append(item)

    return results

def topic_review_set_status(
    record_id: int,
    status: str,
    *,
    reviewed_by: str | None = None,
    corrected_decision: str | None = None,
    corrected_mapped_concept_id: str | None = None,
    correction_reason: str | None = None,
) -> dict[str, Any]:
    """Approve, correct, or reject one topic-review record."""

    normalised_status = str(status).strip().casefold()
    if normalised_status not in {"approved", "corrected", "rejected"}:
        raise ValueError(
            "Status must be 'approved', 'corrected', or 'rejected'."
        )

    if not POSTGRES_MEMORY_ENABLED or _engine is None:
        raise RuntimeError(
            "PostgreSQL is required for persistent topic human review."
        )

    final_decision = None
    final_concept_id = None
    final_reason = None
    if normalised_status == "corrected":
        final_decision = str(corrected_decision or "").strip().casefold()
        final_reason = str(correction_reason or "").strip()
        if final_decision not in {"mapped", "resolved_by_module3", "out_of_syllabus"}:
            raise ValueError("Invalid corrected decision.")
        if not final_reason:
            raise ValueError("Correction reason is required.")
        if final_decision != "out_of_syllabus":
            final_concept_id = str(corrected_mapped_concept_id or "").strip()
            if not final_concept_id:
                raise ValueError("Corrected mapped concept ID is required.")

    statement = text(
        """
        UPDATE topic_human_review
        SET
            status = :status,
            corrected_decision = :corrected_decision,
            corrected_mapped_concept_id = :corrected_mapped_concept_id,
            correction_reason = :correction_reason,
            reviewed_by = :reviewed_by,
            reviewed_at = NOW(),
            updated_at = NOW()
        WHERE id = :record_id
        RETURNING
            id,
            cache_key,
            original_topic,
            proposed_decision,
            proposed_mapped_concept_id,
            corrected_decision,
            corrected_mapped_concept_id,
            correction_reason,
            confidence,
            status,
            spec_version,
            reviewed_at
        """
    )

    with _engine.begin() as connection:
        row = connection.execute(
            statement,
            {
                "record_id": int(record_id),
                "status": normalised_status,
                "corrected_decision": final_decision,
                "corrected_mapped_concept_id": final_concept_id,
                "correction_reason": final_reason,
                "reviewed_by": reviewed_by,
            },
        ).mappings().one_or_none()

    if row is None:
        raise KeyError(f"Topic review record {record_id} was not found.")

    result = dict(row)
    if isinstance(result.get("reviewed_at"), datetime):
        result["reviewed_at"] = result["reviewed_at"].isoformat()

    result["promoted_to_mapping_memory"] = (
        normalised_status in {"approved", "corrected"}
    )
    return result

print("Groq model:", GROQ_MODEL)
print("Qdrant shortlist:", LLM_QDRANT_TOP_K)
print("Auto-map threshold:", LLM_AUTO_MAP_CONFIDENCE)
print("Groq configured:", groq_client is not None)
print(
    "Omitted-item retry limit:",
    GROQ_OMITTED_ITEM_MAX_RETRIES,
)
print(
    "Memory backend:",
    "PostgreSQL reviewer-approved memory"
    if POSTGRES_MEMORY_ENABLED
    else "Qdrant fallback (no trusted local JSON reuse)",
)
print("Local Module 3 shortcut enabled:", MODULE4_ALLOW_LOCAL_MODULE3_RESOLUTION)
print("Topic approval trigger:", "enabled" if POSTGRES_MEMORY_ENABLED else "unavailable")


## 13. Every-unmapped-topic Module 4 input collection


In [ ]:
@dataclass(frozen=True)
class UnmappedTopicInput:
    rough_topic: str
    domain: str
    score: float
    evidence: str
    matched_aliases: list[str]
    detection_method: str
    source_chunk_ids: list[int]
    source_text: str
    requires_topic_label: bool = False


@dataclass(frozen=True)
class LLMResolvedTopic:
    rough_topic: str
    decision: str
    mapped_concept_id: str | None
    mapped_topic: str | None
    domain: str | None
    official_reference: str | None
    chapter_reference: str | None
    confidence: float
    reason: str
    source_chunk_ids: list[int]
    qdrant_candidates: list[dict[str, Any]]
    model: str
    resolution_source: str = "llm"
    review_id: int | None = None
    review_status: str | None = None
    memory_cache_key: str | None = None


GENERIC_UNMAPPED_TOPIC_LABELS = {
    "unmapped computer science content",
    "unmapped cs content",
    "computer science content",
    "unknown computer science topic",
    "unknown cs topic",
}


def _normalise_unmapped_label(value: str) -> str:
    value = re.sub(r"[^a-z0-9]+", " ", str(value).casefold())
    return re.sub(r"\s+", " ", value).strip()


def _is_generic_unmapped_label(value: str) -> bool:
    return (
        _normalise_unmapped_label(value)
        in GENERIC_UNMAPPED_TOPIC_LABELS
    )


def _alias_occurrences(
    *,
    text: str,
    alias: str,
) -> int:
    normalised_text = _normalise_unmapped_label(text)
    normalised_alias = _normalise_unmapped_label(alias)

    if not normalised_alias:
        return 0

    return len(
        re.findall(
            rf"(?<![a-z0-9]){re.escape(normalised_alias)}"
            rf"(?![a-z0-9])",
            normalised_text,
        )
    )


def _derive_specific_unmapped_family(
    *,
    evidence: str,
    source_text: str,
) -> tuple[str, str, list[str]] | None:
    """
    Derive a known specific off-syllabus family from generic semantic residue.

    Evidence is weighted more strongly than the surrounding chunk. If no
    specific family is supported, the item is marked ``needs_review`` and is
    not sent to Groq or saved as an approvable mapping.
    """

    ranked_matches: list[
        tuple[int, int, int, UnmappedConceptFamily, list[str]]
    ] = []

    for family in UNMAPPED_CONCEPT_FAMILIES:
        evidence_aliases = [
            alias
            for alias in family.aliases
            if _alias_occurrences(
                text=evidence,
                alias=alias,
            )
            > 0
        ]
        context_aliases = [
            alias
            for alias in family.aliases
            if alias not in evidence_aliases
            and _alias_occurrences(
                text=source_text,
                alias=alias,
            )
            > 0
        ]

        matched_aliases = [
            *evidence_aliases,
            *context_aliases,
        ]
        distinct_hits = len(matched_aliases)
        total_hits = sum(
            _alias_occurrences(
                text=evidence,
                alias=alias,
            )
            * 2
            + _alias_occurrences(
                text=source_text,
                alias=alias,
            )
            for alias in matched_aliases
        )

        if (
            distinct_hits < family.minimum_distinct_aliases
            or total_hits < family.minimum_total_hits
        ):
            continue

        # At least one direct evidence match is required unless the chunk has
        # two or more distinct, explicit aliases for the same family.
        if not evidence_aliases and distinct_hits < 2:
            continue

        ranked_matches.append(
            (
                len(evidence_aliases),
                total_hits,
                max(
                    (
                        len(_normalise_unmapped_label(alias))
                        for alias in matched_aliases
                    ),
                    default=0,
                ),
                family,
                matched_aliases,
            )
        )

    if not ranked_matches:
        return None

    ranked_matches.sort(
        key=lambda item: item[:3],
        reverse=True,
    )
    _, _, _, family, matched_aliases = ranked_matches[0]

    return (
        family.rough_topic,
        family.domain,
        matched_aliases,
    )


def _topic_label_override_for_chunk(
    topic_label_overrides: dict[str, Any] | None,
    chunk_id: int,
) -> dict[str, Any] | None:
    if not isinstance(topic_label_overrides, dict):
        return None

    raw = topic_label_overrides.get(str(chunk_id))
    if not isinstance(raw, dict):
        raw = topic_label_overrides.get(chunk_id)
    return raw if isinstance(raw, dict) else None


def collect_unmapped_inputs(
    module3_result: Module3Result,
    chunks: list[dict[str, Any]],
    topic_label_overrides: dict[str, Any] | None = None,
    transcript_name: str | None = None,
) -> list[UnmappedTopicInput]:
    """
    Collect every genuine unmapped-CS signal for memory-first resolution.

    A generic semantic label is never made directly approvable. It is first
    converted to a supported specific rough-topic family. When that cannot be
    done safely, it becomes a non-approvable ``needs_review`` result.

        specific unmapped signal -> PostgreSQL memory lookup
                                 -> Qdrant shortlist + one Groq batch
                                 -> pending human-review record
                                 -> approved mapping memory

        generic semantic residue -> derive specific family
                                 -> otherwise needs_review only
    """

    chunk_text_by_id = {
        int(chunk["chunk_id"]): str(chunk["text"])
        for chunk in chunks
    }

    collected: list[UnmappedTopicInput] = []
    seen: set[tuple[str, int]] = set()

    for chunk_result in module3_result.chunk_results:
        signals = list(chunk_result.unmapped_cs_signals)

        for signal in signals:
            rough_topic = signal.rough_topic
            domain = signal.domain
            matched_aliases = list(signal.matched_aliases)
            detection_method = signal.detection_method
            requires_topic_label = False
            source_text = chunk_text_by_id.get(
                chunk_result.chunk_id,
                "",
            )

            if _is_generic_unmapped_label(rough_topic):
                derived = _derive_specific_unmapped_family(
                    evidence=signal.evidence,
                    source_text=source_text,
                )

                if derived is None:
                    override = _topic_label_override_for_chunk(
                        topic_label_overrides,
                        chunk_result.chunk_id,
                    )
                    override_action = str(
                        (override or {}).get("action") or ""
                    ).strip().casefold()

                    if override_action == "ignore":
                        # Current-run reviewer guidance is authoritative.
                        continue

                    if override_action == "label":
                        human_label = str(
                            (override or {}).get("rough_topic") or ""
                        ).strip()
                        if human_label:
                            rough_topic = human_label
                            matched_aliases = []
                            detection_method = "human_topic_label"
                            requires_topic_label = False
                        else:
                            requires_topic_label = True
                    else:
                        # No run-local override exists. Consult persistent
                        # reviewer-validated evidence memory before asking the
                        # human again on this or a future transcript.
                        label_memory = topic_label_memory_lookup(
                            evidence_text=source_text or signal.evidence,
                            transcript_name=transcript_name,
                            source_chunk_ids=[chunk_result.chunk_id],
                        )

                        if label_memory is None:
                            requires_topic_label = True
                        elif label_memory.get("action") == "ignore":
                            # A strongly matching prior human decision says
                            # this mixed residual adds no new lesson topic.
                            continue
                        else:
                            remembered_label = str(
                                label_memory.get("assigned_rough_topic") or ""
                            ).strip()
                            if remembered_label:
                                rough_topic = remembered_label
                                matched_aliases = []
                                detection_method = "topic_label_memory"
                                requires_topic_label = False
                            else:
                                requires_topic_label = True
                else:
                    (
                        rough_topic,
                        domain,
                        matched_aliases,
                    ) = derived

            key = (
                rough_topic.casefold(),
                chunk_result.chunk_id,
            )

            if key in seen:
                continue

            seen.add(key)

            collected.append(
                UnmappedTopicInput(
                    rough_topic=rough_topic,
                    domain=domain,
                    score=float(signal.score),
                    evidence=signal.evidence,
                    matched_aliases=matched_aliases,
                    detection_method=detection_method,
                    source_chunk_ids=[chunk_result.chunk_id],
                    source_text=source_text,
                    requires_topic_label=requires_topic_label,
                )
            )

        # Preserve the existing borderline-official fallback route. This only
        # applies when Module 3 asked for fallback but produced no unmapped
        # signal.
        if (
            not signals
            and chunk_result.requires_llm_fallback
            and chunk_result.rejected_candidates
        ):
            candidate = max(
                chunk_result.rejected_candidates,
                key=lambda item: item.cs_relevance_score,
            )
            key = (
                candidate.topic.casefold(),
                chunk_result.chunk_id,
            )

            if key not in seen:
                seen.add(key)
                collected.append(
                    UnmappedTopicInput(
                        rough_topic=candidate.topic,
                        domain=candidate.domain,
                        score=float(candidate.cs_relevance_score),
                        evidence=(
                            candidate.evidence[0]
                            if candidate.evidence
                            else chunk_text_by_id.get(
                                chunk_result.chunk_id,
                                "",
                            )
                        ),
                        matched_aliases=list(candidate.matched_aliases),
                        detection_method=(
                            "semantic"
                            if candidate.extraction_method == "embedding"
                            else "lexical_semantic"
                        ),
                        source_chunk_ids=[chunk_result.chunk_id],
                        source_text=chunk_text_by_id.get(
                            chunk_result.chunk_id,
                            "",
                        ),
                        requires_topic_label=False,
                    )
                )

    return collected


## 14. Qdrant shortlist, Groq validation, and safe candidate restrictions


In [ ]:
def retrieve_llm_candidates(
    item: UnmappedTopicInput,
) -> list[dict[str, Any]]:
    retrieval_text = (
        f"Unmapped CS topic: {item.rough_topic}\n"
        f"Domain: {item.domain}\n"
        f"Evidence: {item.evidence}\n"
        f"Lesson context: {item.source_text}"
    )

    query_vector = embed_texts(
        [retrieval_text],
        TOPIC_EMBEDDING_MODEL,
        32,
    )

    result_sets = qdrant_store.search_by_vectors(
        query_vector,
        top_k=LLM_QDRANT_TOP_K,
    )

    matches = result_sets[0] if result_sets else []
    candidates = []

    for match in matches:
        concept = CONCEPT_BY_ID.get(match.concept_id)

        if concept is None:
            continue

        candidates.append(
            {
                "concept_id": concept.concept_id,
                "label": concept.label,
                "domain": concept.domain,
                "official_reference": concept.official_reference,
                "chapter_reference": concept.chapter_reference,
                "official_title": concept.official_title,
                "qdrant_score": round(float(match.score), 4),
            }
        )

    return candidates


SYSTEM_PROMPT = '''
You map unresolved computer-science lesson topics to the AQA GCSE Computer
Science syllabus.

Rules:
1. Select only a concept_id from the supplied candidate list.
2. Never invent a concept, chapter, title, or reference.
3. Map only when the lesson evidence genuinely teaches the candidate concept.
4. If none fit the syllabus, return out_of_syllabus.
5. If evidence is insufficient or ambiguous, return needs_review.
6. Return JSON only.

Required JSON:
{
  "decision": "mapped | out_of_syllabus | needs_review",
  "mapped_concept_id": "candidate ID or null",
  "confidence": 0.0,
  "reason": "brief evidence-based explanation"
}
'''.strip()


def call_groq_mapper(
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
) -> dict[str, Any]:
    if groq_client is None:
        raise EnvironmentError(
            "GROQ_API_KEY is missing from .env."
        )

    user_prompt = f'''
UNMAPPED TOPIC:
{item.rough_topic}

DETECTED DOMAIN:
{item.domain}

MODULE 3 SCORE:
{item.score}

EVIDENCE:
{item.evidence}

SOURCE CHUNK:
{item.source_text}

ALLOWED OFFICIAL CANDIDATES:
{json.dumps(candidates, indent=2, ensure_ascii=False)}
'''.strip()

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=250,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    content = response.choices[0].message.content

    if not content:
        raise RuntimeError("Groq returned an empty response.")

    return json.loads(content)


def validate_llm_resolution(
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    raw: dict[str, Any],
) -> LLMResolvedTopic:
    valid_decisions = {
        "mapped",
        "out_of_syllabus",
        "needs_review",
    }

    decision = str(
        raw.get("decision", "needs_review")
    ).strip().lower()

    if decision not in valid_decisions:
        decision = "needs_review"

    try:
        confidence = float(raw.get("confidence", 0.0))
    except (TypeError, ValueError):
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))
    reason = str(raw.get("reason", "No reason supplied.")).strip()

    selected_id = raw.get("mapped_concept_id")
    selected_id = (
        str(selected_id).strip()
        if selected_id is not None
        else None
    )

    allowed_ids = {
        candidate["concept_id"]
        for candidate in candidates
    }

    selected = (
        CONCEPT_BY_ID.get(selected_id)
        if selected_id
        else None
    )

    if decision == "mapped":
        if (
            selected is None
            or selected_id not in allowed_ids
        ):
            decision = "needs_review"
            selected = None
            reason = (
                "The LLM selected an invalid or non-shortlisted "
                "concept. " + reason
            )

        elif confidence < LLM_AUTO_MAP_CONFIDENCE:
            decision = "needs_review"
            reason = (
                f"Confidence {confidence:.2f} is below the "
                f"{LLM_AUTO_MAP_CONFIDENCE:.2f} threshold. "
                + reason
            )

    else:
        selected = None

    return LLMResolvedTopic(
        rough_topic=item.rough_topic,
        decision=decision,
        mapped_concept_id=(
            selected.concept_id
            if selected
            else None
        ),
        mapped_topic=(
            selected.label
            if selected
            else None
        ),
        domain=(
            selected.domain
            if selected
            else item.domain
        ),
        official_reference=(
            selected.official_reference
            if selected
            else None
        ),
        chapter_reference=(
            selected.chapter_reference
            if selected
            else None
        ),
        confidence=round(confidence, 4),
        reason=reason,
        source_chunk_ids=item.source_chunk_ids,
        qdrant_candidates=candidates,
        model=GROQ_MODEL,
    )


def resolve_unmapped_item(
    item: UnmappedTopicInput,
) -> LLMResolvedTopic:
    candidates = retrieve_llm_candidates(item)

    if not candidates:
        return LLMResolvedTopic(
            rough_topic=item.rough_topic,
            decision="needs_review",
            mapped_concept_id=None,
            mapped_topic=None,
            domain=item.domain,
            official_reference=None,
            chapter_reference=None,
            confidence=0.0,
            reason="Qdrant returned no official candidates.",
            source_chunk_ids=item.source_chunk_ids,
            qdrant_candidates=[],
            model=GROQ_MODEL,
        )

    raw = call_groq_mapper(item, candidates)

    return validate_llm_resolution(
        item,
        candidates,
        raw,
    )


## 15. Memory-first PostgreSQL resolution, batched Groq, and human-review persistence

If an otherwise valid Groq batch omits one or more `item_id` values, only those missing
entries are retried. Valid results from the original batch are preserved. Every new
proposal is written to `topic_human_review`; it is not reusable memory until approved.


In [ ]:
def _normalise_memory_text(value: str) -> str:
    value = unicodedata.normalize(
        "NFKC",
        str(value),
    ).casefold()

    return re.sub(
        r"\s+",
        " ",
        value,
    ).strip()


def _sha256(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def _compact_context(
    item: UnmappedTopicInput,
) -> str:
    evidence = re.sub(
        r"\s+",
        " ",
        item.evidence,
    ).strip()

    source = re.sub(
        r"\s+",
        " ",
        item.source_text,
    ).strip()

    combined = (
        f"Evidence: {evidence}\n"
        f"Context: {source}"
    )

    if len(combined) <= MODULE4_CONTEXT_LIMIT:
        return combined

    return (
        combined[:MODULE4_CONTEXT_LIMIT].rstrip()
        + "..."
    )


def _evidence_hash_for_item(
    item: UnmappedTopicInput,
) -> str:
    return _sha256(
        _normalise_memory_text(
            f"{item.evidence}\n{item.source_text}"
        )
    )


def _memory_identity(
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    module3_ids: list[str],
) -> tuple[str, str]:
    evidence_hash = _evidence_hash_for_item(item)

    payload = {
        "topic": _normalise_memory_text(
            item.rough_topic
        ),
        "evidence_hash": evidence_hash,
        "candidate_ids": sorted(
            candidate["concept_id"]
            for candidate in candidates
        ),
        "module3_ids": sorted(module3_ids),
        "model": GROQ_MODEL,
        "prompt_version": MODULE4_PROMPT_VERSION,
    }

    cache_key = _sha256(
        json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False,
            separators=(",", ":"),
        )
    )

    return cache_key, evidence_hash


def _mapped_result(
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    concept_id: str,
    confidence: float,
    reason: str,
    resolution_source: str = "llm",
) -> LLMResolvedTopic:
    concept = CONCEPT_BY_ID[concept_id]

    return LLMResolvedTopic(
        rough_topic=item.rough_topic,
        decision="mapped",
        mapped_concept_id=concept.concept_id,
        mapped_topic=concept.label,
        domain=concept.domain,
        official_reference=concept.official_reference,
        chapter_reference=concept.chapter_reference,
        confidence=round(
            max(0.0, min(1.0, confidence)),
            4,
        ),
        reason=reason,
        source_chunk_ids=item.source_chunk_ids,
        qdrant_candidates=candidates,
        model=GROQ_MODEL,
        resolution_source=resolution_source,
    )



def _memory_record_to_result(
    *,
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    record: Any,
) -> LLMResolvedTopic | None:
    cache_key = str(record.cache_key)

    if record.decision == "out_of_syllabus":
        return LLMResolvedTopic(
            rough_topic=item.rough_topic,
            decision="out_of_syllabus",
            mapped_concept_id=None,
            mapped_topic=None,
            domain=item.domain,
            official_reference=None,
            chapter_reference=None,
            confidence=round(float(record.confidence), 4),
            reason="[MAPPING MEMORY] " + str(record.reason),
            source_chunk_ids=item.source_chunk_ids,
            qdrant_candidates=candidates,
            model=str(record.model_name),
            resolution_source="memory",
            review_status="approved",
            memory_cache_key=cache_key,
        )

    if not record.mapped_concept_id:
        return None

    return replace(
        _mapped_result(
            item=item,
            candidates=candidates,
            concept_id=str(record.mapped_concept_id),
            confidence=float(record.confidence),
            reason="[MAPPING MEMORY] " + str(record.reason),
            resolution_source="memory",
        ),
        review_status="approved",
        memory_cache_key=cache_key,
    )


def _existing_topic_review_to_result(
    *,
    item: UnmappedTopicInput,
    record: dict[str, Any],
) -> LLMResolvedTopic:
    """
    Convert an existing transcript/topic review row into the current run's
    result without creating another proposal or making another Groq call.
    """

    status = str(record.get("status") or "pending").casefold()
    decision = str(
        record.get("proposed_decision") or "needs_review"
    ).casefold()
    mapped_concept_id = record.get(
        "proposed_mapped_concept_id"
    )
    review_id = int(record["id"])
    cache_key = str(record["cache_key"])
    model_name = str(record.get("model_name") or GROQ_MODEL)
    confidence = round(
        max(0.0, min(1.0, float(record.get("confidence") or 0.0))),
        4,
    )
    stored_reason = str(
        record.get("reason") or "Existing topic-review proposal."
    )

    if status == "rejected":
        return LLMResolvedTopic(
            rough_topic=item.rough_topic,
            decision="needs_review",
            mapped_concept_id=None,
            mapped_topic=None,
            domain=item.domain,
            official_reference=None,
            chapter_reference=None,
            confidence=confidence,
            reason=(
                "[EXISTING REJECTED TOPIC REVIEW] "
                + stored_reason
                + " The same transcript/topic was not sent to Groq again."
            ),
            source_chunk_ids=item.source_chunk_ids,
            qdrant_candidates=[],
            model=model_name,
            resolution_source="deterministic",
            review_id=review_id,
            review_status=status,
            memory_cache_key=cache_key,
        )

    resolution_source = (
        "memory"
        if status == "approved"
        else "llm"
    )
    prefix = (
        "[EXISTING APPROVED TOPIC REVIEW] "
        if status == "approved"
        else "[EXISTING PENDING TOPIC REVIEW] "
    )

    if (
        decision == "mapped"
        and mapped_concept_id in CONCEPT_BY_ID
    ):
        return replace(
            _mapped_result(
                item=item,
                candidates=[],
                concept_id=str(mapped_concept_id),
                confidence=confidence,
                reason=prefix + stored_reason,
                resolution_source=resolution_source,
            ),
            model=model_name,
            review_id=review_id,
            review_status=status,
            memory_cache_key=cache_key,
        )

    if decision == "out_of_syllabus":
        return LLMResolvedTopic(
            rough_topic=item.rough_topic,
            decision="out_of_syllabus",
            mapped_concept_id=None,
            mapped_topic=None,
            domain=item.domain,
            official_reference=None,
            chapter_reference=None,
            confidence=confidence,
            reason=prefix + stored_reason,
            source_chunk_ids=item.source_chunk_ids,
            qdrant_candidates=[],
            model=model_name,
            resolution_source=resolution_source,
            review_id=review_id,
            review_status=status,
            memory_cache_key=cache_key,
        )

    return LLMResolvedTopic(
        rough_topic=item.rough_topic,
        decision="needs_review",
        mapped_concept_id=None,
        mapped_topic=None,
        domain=item.domain,
        official_reference=None,
        chapter_reference=None,
        confidence=confidence,
        reason=prefix + stored_reason,
        source_chunk_ids=item.source_chunk_ids,
        qdrant_candidates=[],
        model=model_name,
        resolution_source=resolution_source,
        review_id=review_id,
        review_status=status,
        memory_cache_key=cache_key,
    )


def _find_agent1_code_root_for_mapping_memory() -> Path | None:
    """Locate Agent_1/app from the final Streamlit notebook location."""

    candidates = [
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
        PROJECT_ROOT.parent.parent,
    ]

    for candidate in candidates:
        if (candidate / "app" / "db").is_dir():
            return candidate

    return None


def _lookup_memory_first(
    *,
    item: UnmappedTopicInput,
    transcript_name: str,
) -> LLMResolvedTopic | None:
    """
    Safely reuse only compatible reviewer-approved PostgreSQL memory.

    A failure in the memory layer is fail-open toward the existing Qdrant
    path: it never crashes or replaces the working Module 3/Qdrant logic.
    """

    if not POSTGRES_MEMORY_ENABLED or _SessionFactory is None:
        return None

    code_root = _find_agent1_code_root_for_mapping_memory()
    if code_root is None:
        print(
            "Topic mapping memory service not found beside the Streamlit "
            "frontend; continuing with existing Qdrant flow."
        )
        return None

    if str(code_root) not in sys.path:
        sys.path.insert(0, str(code_root))

    try:
        # Register all referenced tables in the Agent 1 SQLAlchemy metadata
        # before the audit-log repository is used.
        from app.db.models.topic_human_review import TopicHumanReview as _TopicHumanReview
        from app.db.repositories.topic_mapping_decision_log_repository import (
            TopicMappingDecisionLogRepository,
        )
        from app.db.repositories.topic_mapping_memory_repository import (
            TopicMappingMemoryRepository,
        )
        from app.services.topic_mapping_memory_service import (
            TopicMappingMemoryService,
        )
    except Exception as error:
        print(
            "Topic mapping memory service import failed; continuing with "
            f"existing Qdrant flow: {type(error).__name__}: {error}"
        )
        return None

    normalized_topic = _normalise_memory_text(item.rough_topic)
    new_evidence = _compact_context(item)

    try:
        with _SessionFactory() as session:
            service = TopicMappingMemoryService(
                TopicMappingMemoryRepository(session),
                TopicMappingDecisionLogRepository(session),
                embedding_function=embed_texts,
            )

            lookup = service.evaluate_and_record(
                normalized_topic=normalized_topic,
                new_evidence=new_evidence,
                spec_version=AQA_SPEC_VERSION,
                pipeline_run_id=None,
                cache_key=None,
                source_transcript=transcript_name,
                source_chunk_ids=item.source_chunk_ids,
            )
            session.commit()

            if not lookup.is_hit or lookup.matched_memory is None:
                return None

            return _memory_record_to_result(
                item=item,
                candidates=[],
                record=lookup.matched_memory,
            )
    except Exception as error:
        print(
            "Reviewer-approved memory compatibility check failed; "
            "continuing with existing Qdrant flow: "
            f"{type(error).__name__}: {error}"
        )
        return None


def _lookup_memory(
    *,
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    cache_key: str,
) -> LLMResolvedTopic | None:
    """Exact-cache compatibility lookup after Qdrant shortlist creation."""

    record = memory_get_and_mark_used(cache_key)

    if record is None:
        return None

    return _memory_record_to_result(
        item=item,
        candidates=candidates,
        record=record,
    )


def _save_memory(
    *,
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    cache_key: str,
    evidence_hash: str,
    module3_ids: list[str],
    result: LLMResolvedTopic,
    transcript_name: str,
    decision_override: str | None = None,
) -> bool:
    if result.decision not in {"mapped", "out_of_syllabus"}:
        return False

    decision = decision_override or result.decision
    mapped_concept_id = (
        result.mapped_concept_id
        if decision in {"mapped", "resolved_by_module3"}
        else None
    )

    memory_upsert_mapping(
        cache_key=cache_key,
        normalized_topic=_normalise_memory_text(item.rough_topic),
        original_topic=item.rough_topic,
        evidence_hash=evidence_hash,
        evidence_text=_compact_context(item),
        candidate_concept_ids=[candidate["concept_id"] for candidate in candidates],
        module3_concept_ids=module3_ids,
        decision=decision,
        mapped_concept_id=mapped_concept_id,
        confidence=float(result.confidence),
        reason=result.reason,
        model_name=GROQ_MODEL,
        prompt_version=MODULE4_PROMPT_VERSION,
        source_transcript=transcript_name,
        source_chunk_ids=item.source_chunk_ids,
        validation_status="validated",
    )
    return True




def _save_topic_review(
    *,
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    cache_key: str,
    evidence_hash: str,
    module3_ids: list[str],
    result: LLMResolvedTopic,
    transcript_name: str,
) -> tuple[int | None, str]:
    """
    Save a first-time resolver proposal to ``topic_human_review``.

    The proposal is deliberately not reusable mapping memory yet. PostgreSQL
    promotes it only after the user selects Approve.
    """

    return topic_review_upsert(
        cache_key=cache_key,
        normalized_topic=_normalise_memory_text(item.rough_topic),
        original_topic=item.rough_topic,
        evidence_hash=evidence_hash,
        evidence_text=_compact_context(item),
        candidate_concept_ids=[
            candidate["concept_id"]
            for candidate in candidates
        ],
        qdrant_candidates=candidates,
        module3_concept_ids=module3_ids,
        proposed_decision=result.decision,
        proposed_mapped_concept_id=result.mapped_concept_id,
        confidence=float(result.confidence),
        reason=result.reason,
        model_name=result.model or GROQ_MODEL,
        prompt_version=MODULE4_PROMPT_VERSION,
        source_transcript=transcript_name,
        source_chunk_ids=item.source_chunk_ids,
    )


def _resolve_from_module3(
    *,
    item: UnmappedTopicInput,
    candidates: list[dict[str, Any]],
    module3_id_set: set[str],
) -> LLMResolvedTopic | None:
    if not candidates:
        return None

    top = candidates[0]
    top_score = float(top["qdrant_score"])

    second_score = (
        float(candidates[1]["qdrant_score"])
        if len(candidates) > 1
        else 0.0
    )

    margin = top_score - second_score
    concept_id = str(top["concept_id"])

    if concept_id not in module3_id_set:
        return None

    if top_score < MODULE3_LOCAL_MIN_SCORE:
        return None

    if margin < MODULE3_LOCAL_MIN_MARGIN:
        return None

    return _mapped_result(
        item=item,
        candidates=candidates,
        concept_id=concept_id,
        confidence=top_score,
        reason=(
            "[RESOLVED BY MODULE 3] The strongest Qdrant "
            "candidate was already retained by Module 3 "
            f"(score={top_score:.4f}, margin={margin:.4f})."
        ),
        resolution_source="module3",
    )


BATCH_MAPPING_SYSTEM_PROMPT = (
    "You map unresolved computer-science lesson topics to the official "
    "AQA GCSE Computer Science 8525 specification. You receive one array "
    "containing all unresolved topics from a single transcript. Treat every "
    "item independently and return exactly one result for every item_id in "
    "one JSON object containing a results array. Select only a concept ID "
    "from that item's allowed_candidates and never invent a concept ID. "
    "Use decision='mapped' only when the lesson evidence directly matches "
    "the examinable content of that official candidate; broad chapter-level "
    "similarity is not enough. Use decision='out_of_syllabus' when the topic "
    "is computer science but is not explicitly required by AQA 8525. For an "
    "out_of_syllabus result, mapped_concept_id must be null and the reason "
    "should mention the closest related AQA area when one exists. Use "
    "decision='needs_review' when the evidence is insufficient or genuinely "
    "ambiguous. ArrayList, list collection, dynamic-array, or resizable-array "
    "evidence may map to AQA 3.2.6 arrays only when it is being used as an "
    "array-equivalent structure for storing, indexing, or traversing values. "
    "Do not force object-oriented constructs such as classes, constructors, "
    "object initialisation, encapsulation, inheritance, polymorphism, or "
    "private/public access modifiers into general programming topics. Those "
    "constructs are out of syllabus unless the evidence independently and "
    "directly teaches an explicit AQA concept such as local-variable scope, "
    "subroutines, variable declaration, assignment, arrays, or records. "
    "Return JSON only in this exact shape: "
    "{\"results\":[{\"item_id\":\"u1\","
    "\"decision\":\"mapped | out_of_syllabus | needs_review\","
    "\"mapped_concept_id\":\"allowed ID or null\","
    "\"confidence\":0.0,\"reason\":\"brief evidence-based explanation\"}]}."
)


def _call_groq_batch(
    pending: list[dict[str, Any]],
) -> dict[str, Any]:
    if not pending:
        return {"results": []}

    if groq_client is None:
        raise EnvironmentError(
            "GROQ_API_KEY is missing from .env."
        )

    request_items = []

    for entry in pending:
        item = entry["item"]
        candidates = entry["candidates"]

        request_items.append(
            {
                "item_id": entry["item_id"],
                "rough_topic": item.rough_topic,
                "domain": item.domain,
                "module3_score": item.score,
                "evidence": _compact_context(item),
                "allowed_candidates": [
                    {
                        "id": candidate["concept_id"],
                        "topic": candidate["label"],
                        "reference": candidate[
                            "official_reference"
                        ],
                        "score": candidate[
                            "qdrant_score"
                        ],
                    }
                    for candidate in candidates
                ],
            }
        )

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=min(
            1800,
            max(300, 180 * len(pending)),
        ),
        response_format={
            "type": "json_object",
        },
        messages=[
            {
                "role": "system",
                "content": BATCH_MAPPING_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": json.dumps(
                    {"items": request_items},
                    indent=2,
                    ensure_ascii=False,
                ),
            },
        ],
    )

    content = response.choices[0].message.content

    if not content:
        raise RuntimeError(
            "Groq returned an empty response."
        )

    return json.loads(content)



def _index_groq_batch_results(
    raw_batch: dict[str, Any],
    *,
    allowed_item_ids: set[str] | None = None,
) -> dict[str, dict[str, Any]]:
    """
    Index valid Groq batch results by item ID.

    Unknown IDs are ignored when an allowed-ID set is supplied. Duplicate
    IDs use the last valid object, matching normal dictionary semantics.
    """

    raw_results = raw_batch.get(
        "results",
        [],
    )

    if not isinstance(raw_results, list):
        raise ValueError(
            "Groq results must be a list."
        )

    indexed: dict[str, dict[str, Any]] = {}

    for raw in raw_results:
        if not isinstance(raw, dict):
            continue

        raw_item_id = raw.get("item_id")

        if raw_item_id is None:
            continue

        item_id = str(raw_item_id)

        if (
            allowed_item_ids is not None
            and item_id not in allowed_item_ids
        ):
            continue

        indexed[item_id] = raw

    return indexed


def _retry_omitted_groq_items(
    *,
    pending: list[dict[str, Any]],
    raw_by_id: dict[str, dict[str, Any]],
    max_retries: int,
    call_batch: Any = None,
) -> tuple[
    dict[str, dict[str, Any]],
    list[dict[str, Any]],
    int,
    int,
    str | None,
]:
    """
    Retry only entries omitted from a Groq batch response.

    Returns:
        updated result index,
        entries still omitted,
        retry API-call count,
        total item attempts across retries,
        optional retry error message.

    A retry failure does not discard valid results from the original batch.
    """

    if max_retries < 0:
        raise ValueError(
            "max_retries cannot be negative."
        )

    call_batch = call_batch or _call_groq_batch

    pending_by_id = {
        str(entry["item_id"]): entry
        for entry in pending
    }

    missing_entries = [
        entry
        for item_id, entry in pending_by_id.items()
        if item_id not in raw_by_id
    ]

    retry_calls = 0
    retried_item_attempts = 0
    retry_error: str | None = None

    for _ in range(max_retries):
        if not missing_entries:
            break

        retry_calls += 1
        retried_item_attempts += len(
            missing_entries
        )

        retry_ids = {
            str(entry["item_id"])
            for entry in missing_entries
        }

        try:
            retry_raw = call_batch(
                missing_entries
            )

            retry_index = (
                _index_groq_batch_results(
                    retry_raw,
                    allowed_item_ids=retry_ids,
                )
            )

            raw_by_id.update(
                retry_index
            )

        except Exception as error:
            retry_error = (
                f"{type(error).__name__}: {error}"
            )
            break

        missing_entries = [
            entry
            for entry in missing_entries
            if str(entry["item_id"])
            not in raw_by_id
        ]

    return (
        raw_by_id,
        missing_entries,
        retry_calls,
        retried_item_attempts,
        retry_error,
    )

def process_module4_optimized(
    *,
    transcript_name: str,
    module3_result: Module3Result,
    chunks_for_module3: list[dict[str, Any]],
    topic_label_overrides: dict[str, Any] | None = None,
) -> tuple[
    list[UnmappedTopicInput],
    list[LLMResolvedTopic],
    dict[str, int],
]:
    """
    Resolve every genuine unmapped topic through the restored architecture.

    Order for every item:
      1. reviewer-approved PostgreSQL memory compatibility lookup;
      2. existing review lookup by transcript + normalized rough topic;
      3. on both misses, existing Qdrant official shortlist construction;
      4. one Groq batch validates all genuinely new transcript topics;
      5. save one proposal to topic_human_review;
      6. PostgreSQL trigger promotes it to topic_mapping_memory only after
         human approval.

    Official extraction, ranking, merging, and primary/supporting assignment
    remain unchanged.
    """

    unmapped_inputs = collect_unmapped_inputs(
        module3_result,
        chunks_for_module3,
        topic_label_overrides=topic_label_overrides,
        transcript_name=transcript_name,
    )

    module3_ids = sorted(
        {
            topic.concept_id
            for topic in module3_result.merged_topics
        }
    )
    module3_id_set = set(module3_ids)

    stats = {
        "unmapped_inputs": len(unmapped_inputs),
        "memory_hits": 0,
        "existing_review_hits": 0,
        "generic_needs_review": 0,
        "duplicate_existing_topic_suppressed": 0,
        "resolved_by_module3": 0,
        "groq_calls": 0,
        "groq_topics": 0,
        "groq_retry_calls": 0,
        "groq_retried_item_attempts": 0,
        "groq_items_still_omitted": 0,
        "memory_writes": 0,
        "review_writes": 0,
    }

    prepared: list[dict[str, Any]] = []

    for index, item in enumerate(
        unmapped_inputs,
        start=1,
    ):
        evidence_hash = _evidence_hash_for_item(item)

        if item.requires_topic_label:
            generic_result = LLMResolvedTopic(
                rough_topic=item.rough_topic,
                decision="needs_review",
                mapped_concept_id=None,
                mapped_topic=None,
                domain=item.domain,
                official_reference=None,
                chapter_reference=None,
                confidence=0.0,
                reason=(
                    "The semantic residual was too generic to create a safe "
                    "topic mapping. A specific rough topic could not be "
                    "derived from the evidence. It was not sent to Groq and "
                    "was not saved as an approvable database proposal."
                ),
                source_chunk_ids=item.source_chunk_ids,
                qdrant_candidates=[],
                model=GROQ_MODEL,
                resolution_source="deterministic",
                review_id=None,
                review_status="needs_topic_label",
                memory_cache_key=None,
            )

            prepared.append(
                {
                    "item_id": f"u{index}",
                    "item": item,
                    "candidates": [],
                    "cache_key": None,
                    "evidence_hash": evidence_hash,
                    "memory_result": None,
                    "existing_review_result": None,
                    "pre_resolved_result": generic_result,
                }
            )
            continue

        # True memory-first lookup: a reusable approved decision avoids both
        # Qdrant retrieval and a Groq call.
        memory_result = _lookup_memory_first(
            item=item,
            transcript_name=transcript_name,
        )

        if memory_result is not None:
            prepared.append(
                {
                    "item_id": f"u{index}",
                    "item": item,
                    "candidates": [],
                    "cache_key": memory_result.memory_cache_key,
                    "evidence_hash": evidence_hash,
                    "memory_result": memory_result,
                    "existing_review_result": None,
                    "pre_resolved_result": None,
                }
            )
            continue

        existing_review = topic_review_get_for_transcript_topic(
            transcript_name=transcript_name,
            normalized_topic=_normalise_memory_text(
                item.rough_topic
            ),
        )

        if existing_review is not None:
            existing_review_result = (
                _existing_topic_review_to_result(
                    item=item,
                    record=existing_review,
                )
            )

            prepared.append(
                {
                    "item_id": f"u{index}",
                    "item": item,
                    "candidates": [],
                    "cache_key": str(
                        existing_review["cache_key"]
                    ),
                    "evidence_hash": evidence_hash,
                    "memory_result": None,
                    "existing_review_result": (
                        existing_review_result
                    ),
                    "pre_resolved_result": None,
                }
            )
            continue

        candidates = retrieve_llm_candidates(item)
        cache_key, evidence_hash = _memory_identity(
            item,
            candidates,
            module3_ids,
        )

        prepared.append(
            {
                "item_id": f"u{index}",
                "item": item,
                "candidates": candidates,
                "cache_key": cache_key,
                "evidence_hash": evidence_hash,
                "memory_result": None,
                "existing_review_result": None,
                "pre_resolved_result": None,
            }
        )

    resolved: dict[str, LLMResolvedTopic] = {}
    pending_for_groq: list[dict[str, Any]] = []

    def save_pending_review(
        entry: dict[str, Any],
        result: LLMResolvedTopic,
    ) -> LLMResolvedTopic:
        review_id, review_status = _save_topic_review(
            item=entry["item"],
            candidates=entry["candidates"],
            cache_key=entry["cache_key"],
            evidence_hash=entry["evidence_hash"],
            module3_ids=module3_ids,
            result=result,
            transcript_name=transcript_name,
        )

        if review_id is not None:
            stats["review_writes"] += 1

        return replace(
            result,
            review_id=review_id,
            review_status=review_status,
            memory_cache_key=entry["cache_key"],
        )

    for entry in prepared:
        item = entry["item"]
        candidates = entry["candidates"]

        memory_result = entry.get("memory_result")
        existing_review_result = entry.get(
            "existing_review_result"
        )
        pre_resolved_result = entry.get(
            "pre_resolved_result"
        )

        if pre_resolved_result is not None:
            resolved[entry["item_id"]] = pre_resolved_result
            stats["generic_needs_review"] += 1
            continue

        if existing_review_result is not None:
            resolved[entry["item_id"]] = (
                existing_review_result
            )
            stats["existing_review_hits"] += 1
            continue

        if memory_result is not None:
            resolved[entry["item_id"]] = memory_result
            stats["memory_hits"] += 1
            continue

        if not candidates:
            result = LLMResolvedTopic(
                rough_topic=item.rough_topic,
                decision="needs_review",
                mapped_concept_id=None,
                mapped_topic=None,
                domain=item.domain,
                official_reference=None,
                chapter_reference=None,
                confidence=0.0,
                reason="Qdrant returned no candidates.",
                source_chunk_ids=item.source_chunk_ids,
                qdrant_candidates=[],
                model=GROQ_MODEL,
                resolution_source="deterministic",
            )
            resolved[entry["item_id"]] = save_pending_review(
                entry,
                result,
            )
            continue

        # Kept as an explicit compatibility option only. The restored default
        # architecture sends a memory miss to Qdrant + Groq.
        if MODULE4_ALLOW_LOCAL_MODULE3_RESOLUTION:
            module3_result_local = _resolve_from_module3(
                item=item,
                candidates=candidates,
                module3_id_set=module3_id_set,
            )

            if module3_result_local is not None:
                module3_result_local = replace(
                    module3_result_local,
                    review_status="validated",
                    memory_cache_key=entry["cache_key"],
                )
                resolved[entry["item_id"]] = module3_result_local
                stats["resolved_by_module3"] += 1

                # Updated architecture: only a human Approved/Corrected
                # review may enter topic_mapping_memory.

                continue

        pending_for_groq.append(entry)

    if pending_for_groq and groq_client is None:
        for entry in pending_for_groq:
            result = LLMResolvedTopic(
                rough_topic=entry["item"].rough_topic,
                decision="needs_review",
                mapped_concept_id=None,
                mapped_topic=None,
                domain=entry["item"].domain,
                official_reference=None,
                chapter_reference=None,
                confidence=0.0,
                reason=(
                    "GROQ_API_KEY is not configured. The topic remains "
                    "pending human review; no unvalidated mapping was added "
                    "to reusable memory."
                ),
                source_chunk_ids=entry["item"].source_chunk_ids,
                qdrant_candidates=entry["candidates"],
                model=GROQ_MODEL,
                resolution_source="deterministic",
            )
            resolved[entry["item_id"]] = save_pending_review(
                entry,
                result,
            )

    elif pending_for_groq:
        stats["groq_calls"] = 1
        stats["groq_topics"] = len(pending_for_groq)

        raw_batch = _call_groq_batch(pending_for_groq)

        pending_ids = {
            str(entry["item_id"])
            for entry in pending_for_groq
        }

        raw_by_id = _index_groq_batch_results(
            raw_batch,
            allowed_item_ids=pending_ids,
        )

        # Strict one-call mode: do not issue item-level or targeted retries.
        # Any omitted item remains a pending human-review record.
        still_omitted = [
            entry
            for entry in pending_for_groq
            if str(entry["item_id"]) not in raw_by_id
        ]
        retry_error = None

        stats["groq_retry_calls"] = 0
        stats["groq_retried_item_attempts"] = 0
        stats["groq_items_still_omitted"] = len(still_omitted)

        for entry in pending_for_groq:
            raw = raw_by_id.get(entry["item_id"])

            if raw is None:
                result = LLMResolvedTopic(
                    rough_topic=entry["item"].rough_topic,
                    decision="needs_review",
                    mapped_concept_id=None,
                    mapped_topic=None,
                    domain=entry["item"].domain,
                    official_reference=None,
                    chapter_reference=None,
                    confidence=0.0,
                    reason=(
                        "Groq omitted this item from the single transcript "
                        "batch response. It remains pending human review; "
                        "no second Groq request was made."
                    ),
                    source_chunk_ids=entry["item"].source_chunk_ids,
                    qdrant_candidates=entry["candidates"],
                    model=GROQ_MODEL,
                    resolution_source="llm",
                )
            else:
                result = validate_llm_resolution(
                    entry["item"],
                    entry["candidates"],
                    raw,
                )

            # A fallback proposal must not create another human-review row
            # for an official concept already retained in merged_topics.
            if (
                result.decision == "mapped"
                and result.mapped_concept_id in module3_id_set
            ):
                existing_concept = CONCEPT_BY_ID[
                    str(result.mapped_concept_id)
                ]
                resolved[entry["item_id"]] = replace(
                    result,
                    mapped_topic=existing_concept.topic,
                    domain=existing_concept.domain,
                    official_reference=(
                        existing_concept.official_reference
                    ),
                    chapter_reference=(
                        existing_concept.chapter_reference
                    ),
                    reason=(
                        "[DUPLICATE REVIEW SUPPRESSED] "
                        "Groq selected an official concept that Module 3 "
                        "had already retained in merged_topics. No new "
                        "topic_human_review row was created. "
                        + result.reason
                    ),
                    resolution_source="module3",
                    review_id=None,
                    review_status="already_detected",
                    memory_cache_key=None,
                )
                stats[
                    "duplicate_existing_topic_suppressed"
                ] += 1
                continue

            # Only genuinely new Groq decisions become review proposals.
            resolved[entry["item_id"]] = save_pending_review(
                entry,
                result,
            )

    ordered_results = [
        resolved[entry["item_id"]]
        for entry in prepared
    ]

    return (
        unmapped_inputs,
        ordered_results,
        stats,
    )


def _retained_topic_evidence(
    topic: MergedTopic,
    chunks_for_module3: list[dict[str, Any]],
) -> str:
    """Return the actual Module 2 evidence behind one retained topic."""

    chunk_text_by_id = {
        int(chunk["chunk_id"]): str(chunk.get("text", "")).strip()
        for chunk in chunks_for_module3
    }
    texts = [
        chunk_text_by_id.get(int(chunk_id), "")
        for chunk_id in topic.source_chunk_ids
    ]
    evidence = "\n\n".join(text for text in texts if text).strip()
    if evidence:
        return evidence
    return "\n".join(str(item).strip() for item in topic.evidence if str(item).strip())


def _apply_reviewer_memory_to_retained_topics(
    *,
    module3_result: Module3Result,
    chunks_for_module3: list[dict[str, Any]],
    transcript_name: str,
) -> Module3Result:
    """
    Apply reviewer-approved mapping memory to already-retained official topics.

    Why this exists:
    Module 3 can deterministically retain an official concept before the
    unresolved-topic Qdrant/Groq stage. A later human correction must therefore
    be consulted on future runs *before final retained topics are emitted*, not
    only for unresolved rough topics.

    Safety remains identical to the normal mapping-memory path:
      - reviewer-approved rows only
      - current AQA spec version only
      - same normalized rough topic
      - strong evidence similarity
      - human-correction reason compatibility when present

    A memory miss leaves the existing Module 3 topic untouched. Any memory-layer
    error is fail-open toward the existing working result.
    """

    if not POSTGRES_MEMORY_ENABLED or _SessionFactory is None:
        return module3_result

    code_root = _find_agent1_code_root_for_mapping_memory()
    if code_root is None:
        return module3_result

    if str(code_root) not in sys.path:
        sys.path.insert(0, str(code_root))

    try:
        from app.db.models.topic_human_review import TopicHumanReview as _TopicHumanReview
        from app.db.repositories.topic_mapping_decision_log_repository import (
            TopicMappingDecisionLogRepository,
        )
        from app.db.repositories.topic_mapping_memory_repository import (
            TopicMappingMemoryRepository,
        )
        from app.services.topic_mapping_memory_service import (
            TopicMappingMemoryService,
        )
    except Exception as error:
        print(
            "Retained-topic memory import failed; keeping existing Module 3 "
            f"topics unchanged: {type(error).__name__}: {error}"
        )
        return module3_result

    corrected_topics: list[MergedTopic] = []

    try:
        with _SessionFactory() as session:
            service = TopicMappingMemoryService(
                TopicMappingMemoryRepository(session),
                TopicMappingDecisionLogRepository(session),
                embedding_function=embed_texts,
            )

            for topic in module3_result.merged_topics:
                normalized_topic = _normalise_memory_text(topic.topic)
                evidence_text = _retained_topic_evidence(
                    topic,
                    chunks_for_module3,
                )

                lookup = service.evaluate_and_record(
                    normalized_topic=normalized_topic,
                    new_evidence=evidence_text,
                    spec_version=AQA_SPEC_VERSION,
                    pipeline_run_id=None,
                    cache_key=None,
                    source_transcript=transcript_name,
                    source_chunk_ids=topic.source_chunk_ids,
                )

                if not lookup.is_hit or lookup.matched_memory is None:
                    corrected_topics.append(topic)
                    continue

                memory = lookup.matched_memory

                if memory.decision == "out_of_syllabus":
                    print(
                        "Reviewer memory suppressed retained topic:",
                        topic.topic,
                        "-> out_of_syllabus",
                    )
                    continue

                target_id = str(memory.mapped_concept_id or "").strip()
                target = CONCEPT_BY_ID.get(target_id)
                if memory.decision != "mapped" or target is None:
                    print(
                        "Reviewer memory hit could not be applied to retained "
                        f"topic {topic.topic!r}; keeping existing mapping."
                    )
                    corrected_topics.append(topic)
                    continue

                if target_id == topic.concept_id:
                    corrected_topics.append(topic)
                    continue

                updated = topic.model_copy(
                    update={
                        "concept_id": target.concept_id,
                        "topic": target.label,
                        "domain": target.domain,
                        "official_reference": target.official_reference,
                        "chapter_reference": target.chapter_reference,
                        "official_title": target.official_title,
                        "paper": target.paper,
                        "source_pages": list(target.source_pages),
                    }
                )
                corrected_topics.append(updated)
                print(
                    "Reviewer memory corrected retained topic:",
                    topic.topic,
                    topic.official_reference,
                    "->",
                    updated.topic,
                    updated.official_reference,
                    f"(memory_id={memory.id})",
                )

            session.commit()

    except Exception as error:
        print(
            "Retained-topic memory compatibility check failed; keeping "
            "existing Module 3 topics unchanged: "
            f"{type(error).__name__}: {error}"
        )
        return module3_result

    return module3_result.model_copy(
        update={"merged_topics": corrected_topics}
    )



In [ ]:
# Regression test: retry only Groq batch items that were omitted

_retry_pending = [
    {"item_id": "u1"},
    {"item_id": "u2"},
    {"item_id": "u3"},
]

_retry_initial = {
    "u1": {
        "item_id": "u1",
        "decision": "needs_review",
    },
    "u3": {
        "item_id": "u3",
        "decision": "needs_review",
    },
}

_retry_calls_seen: list[list[str]] = []


def _fake_retry_batch(
    entries: list[dict[str, Any]],
) -> dict[str, Any]:
    item_ids = [
        str(entry["item_id"])
        for entry in entries
    ]

    _retry_calls_seen.append(
        item_ids
    )

    return {
        "results": [
            {
                "item_id": "u2",
                "decision": "needs_review",
                "mapped_concept_id": None,
                "confidence": 0.0,
                "reason": "Recovered on targeted retry.",
            }
        ]
    }


(
    _retry_merged,
    _retry_still_missing,
    _retry_call_count,
    _retry_item_attempts,
    _retry_error,
) = _retry_omitted_groq_items(
    pending=_retry_pending,
    raw_by_id=dict(_retry_initial),
    max_retries=1,
    call_batch=_fake_retry_batch,
)

assert _retry_calls_seen == [["u2"]]
assert "u2" in _retry_merged
assert _retry_still_missing == []
assert _retry_call_count == 1
assert _retry_item_attempts == 1
assert _retry_error is None

print(
    "Targeted omitted-item Groq retry regression test passed."
)


## 16. Discover and validate Module 2 JSON inputs

The notebook scans only `OUTPUT/<transcript>/02_chunking.json`. It never reads the original DOCX files and never regenerates `01_preprocessing.pdf` or `02_chunking.pdf`.


In [ ]:

def discover_module2_inputs() -> list[tuple[str, Path]]:
    if not OUTPUT_DIR.is_dir():
        raise FileNotFoundError(f"OUTPUT folder does not exist: {OUTPUT_DIR}")

    if FRONTEND_SINGLE_FILE_MODE:
        json_path = (
            OUTPUT_DIR
            / FRONTEND_TRANSCRIPT_NAME
            / "02_chunking.json"
        )
        return (
            [(FRONTEND_TRANSCRIPT_NAME, json_path)]
            if json_path.is_file()
            else []
        )

    inputs: list[tuple[str, Path]] = []

    for transcript_folder in sorted(
        (path for path in OUTPUT_DIR.iterdir() if path.is_dir()),
        key=lambda path: path.name.casefold(),
    ):
        if transcript_folder.name.casefold() in {
            name.casefold() for name in EXCLUDED_TRANSCRIPT_FOLDERS
        }:
            continue

        json_path = transcript_folder / "02_chunking.json"
        if json_path.is_file():
            inputs.append((transcript_folder.name, json_path))

    return inputs


def load_module2_chunks(json_path: Path) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    try:
        payload = json.loads(json_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid Module 2 JSON: {json_path}: {error}") from error

    if not isinstance(payload, dict):
        raise TypeError(f"Module 2 JSON root must be an object: {json_path}")

    chunks = payload.get("chunks")
    if not isinstance(chunks, list) or not chunks:
        raise ValueError(f"Module 2 JSON contains no chunks: {json_path}")

    validated: list[dict[str, Any]] = []
    seen_ids: set[int] = set()

    for position, raw in enumerate(chunks, start=1):
        if not isinstance(raw, dict):
            raise TypeError(f"Chunk {position} is not an object in {json_path}")

        chunk_id = int(raw.get("chunk_id", position))
        text = str(raw.get("text", "")).strip()
        if chunk_id < 1 or chunk_id in seen_ids:
            raise ValueError(f"Invalid or duplicate chunk_id {chunk_id} in {json_path}")
        if not text:
            raise ValueError(f"Chunk {chunk_id} has empty text in {json_path}")

        seen_ids.add(chunk_id)
        clean = dict(raw)
        clean["chunk_id"] = chunk_id
        clean["text"] = text
        clean["word_count"] = int(raw.get("word_count") or len(re.findall(r"\S+", text)))
        clean["overlap_word_count"] = int(raw.get("overlap_word_count") or 0)
        validated.append(clean)

    expected_ids = list(range(1, len(validated) + 1))
    actual_ids = [chunk["chunk_id"] for chunk in validated]
    if actual_ids != expected_ids:
        raise ValueError(
            f"Chunk IDs must be sequential in {json_path}. "
            f"Expected {expected_ids}; received {actual_ids}."
        )

    metadata = {key: value for key, value in payload.items() if key != "chunks"}
    return validated, metadata


MODULE2_INPUTS = discover_module2_inputs()

print("Module 2 JSON files found:", len(MODULE2_INPUTS))
for transcript_name, json_path in MODULE2_INPUTS:
    print("-", transcript_name, "->", json_path.name)

if len(MODULE2_INPUTS) != EXPECTED_TRANSCRIPT_COUNT:
    raise RuntimeError(
        f"Expected exactly {EXPECTED_TRANSCRIPT_COUNT} Module 2 JSON files "
        f"after exclusions, but found {len(MODULE2_INPUTS)}."
    )


## 17. PDF styling helpers


In [ ]:

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)

def to_plain_data(value: Any) -> Any:
    if hasattr(value, "model_dump"):
        return value.model_dump()

    if hasattr(value, "dict"):
        return value.dict()

    if hasattr(value, "__dataclass_fields__"):
        return asdict(value)

    if isinstance(value, list):
        return [to_plain_data(item) for item in value]

    if isinstance(value, tuple):
        return [to_plain_data(item) for item in value]

    if isinstance(value, dict):
        return {
            str(key): to_plain_data(item)
            for key, item in value.items()
        }

    return value


PDF_STYLES = getSampleStyleSheet()

PDF_STYLES.add(
    ParagraphStyle(
        name="AgentTitle",
        parent=PDF_STYLES["Title"],
        fontName="Helvetica-Bold",
        fontSize=17,
        leading=21,
        alignment=TA_CENTER,
        spaceAfter=14,
    )
)

PDF_STYLES.add(
    ParagraphStyle(
        name="AgentHeading",
        parent=PDF_STYLES["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=13,
        leading=16,
        spaceBefore=9,
        spaceAfter=6,
    )
)

PDF_STYLES.add(
    ParagraphStyle(
        name="AgentSubheading",
        parent=PDF_STYLES["Heading3"],
        fontName="Helvetica-Bold",
        fontSize=11,
        leading=14,
        spaceBefore=7,
        spaceAfter=4,
    )
)

PDF_STYLES.add(
    ParagraphStyle(
        name="AgentBody",
        parent=PDF_STYLES["BodyText"],
        fontName="Helvetica",
        fontSize=9,
        leading=13,
        spaceAfter=6,
    )
)

PDF_STYLES.add(
    ParagraphStyle(
        name="AgentSmall",
        parent=PDF_STYLES["BodyText"],
        fontName="Helvetica",
        fontSize=8,
        leading=11,
        spaceAfter=3,
    )
)


def safe_pdf_text(value: Any) -> str:
    text = "" if value is None else str(value)

    replacements = {
        "\u2013": "-",
        "\u2014": "-",
        "\u2192": "->",
        "\u2022": "-",
        "\u00a0": " ",
    }

    for source, replacement in replacements.items():
        text = text.replace(source, replacement)

    text = re.sub(r"\s+", " ", text).strip()
    return html.escape(text)


def add_page_number(canvas, document) -> None:
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.drawRightString(
        A4[0] - 16 * mm,
        10 * mm,
        f"Page {canvas.getPageNumber()}",
    )
    canvas.restoreState()


def build_pdf(
    output_path: Path,
    title: str,
    story: list[Any],
) -> None:
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    document = SimpleDocTemplate(
        str(output_path),
        pagesize=A4,
        rightMargin=16 * mm,
        leftMargin=16 * mm,
        topMargin=16 * mm,
        bottomMargin=16 * mm,
        title=title,
        author="Agent 1",
    )

    full_story = [
        Paragraph(
            safe_pdf_text(title),
            PDF_STYLES["AgentTitle"],
        ),
        Spacer(1, 4),
        *story,
    ]

    document.build(
        full_story,
        onFirstPage=add_page_number,
        onLaterPages=add_page_number,
    )


def add_pdf_heading(
    story: list[Any],
    text: str,
    level: int = 1,
) -> None:
    style_name = (
        "AgentHeading"
        if level == 1
        else "AgentSubheading"
    )

    story.append(
        Paragraph(
            safe_pdf_text(text),
            PDF_STYLES[style_name],
        )
    )


def add_pdf_text(
    story: list[Any],
    text: Any,
    small: bool = False,
) -> None:
    style_name = (
        "AgentSmall"
        if small
        else "AgentBody"
    )

    cleaned = str(text or "").strip()

    if not cleaned:
        story.append(
            Paragraph(
                "None",
                PDF_STYLES[style_name],
            )
        )
        return

    blocks = re.split(r"\n\s*\n", cleaned)

    for block in blocks:
        block = block.strip()
        if block:
            story.append(
                Paragraph(
                    safe_pdf_text(block),
                    PDF_STYLES[style_name],
                )
            )


def add_pdf_key_value_table(
    story: list[Any],
    rows: list[tuple[str, Any]],
) -> None:
    if not rows:
        return

    data = []

    for key, value in rows:
        data.append(
            [
                Paragraph(
                    f"<b>{safe_pdf_text(key)}</b>",
                    PDF_STYLES["AgentSmall"],
                ),
                Paragraph(
                    safe_pdf_text(value),
                    PDF_STYLES["AgentSmall"],
                ),
            ]
        )

    table = Table(
        data,
        colWidths=[
            52 * mm,
            122 * mm,
        ],
        repeatRows=0,
        hAlign="LEFT",
    )

    table.setStyle(
        TableStyle(
            [
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("GRID", (0, 0), (-1, -1), 0.25, colors.lightgrey),
                ("BACKGROUND", (0, 0), (0, -1), colors.whitesmoke),
                ("LEFTPADDING", (0, 0), (-1, -1), 5),
                ("RIGHTPADDING", (0, 0), (-1, -1), 5),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ]
        )
    )

    story.append(table)
    story.append(Spacer(1, 7))


## 18. `03_topics_readable.pdf`


In [ ]:
def create_topics_pdf(
    transcript_name: str,
    module3_result: Module3Result,
    output_path: Path,
) -> None:
    story: list[Any] = []

    add_pdf_heading(
        story,
        "Module 3 summary",
    )

    add_pdf_key_value_table(
        story,
        [
            ("Embedding model", module3_result.embedding_model),
            (
                "Candidate keep threshold",
                module3_result.candidate_keep_threshold,
            ),
            ("Total chunks", module3_result.total_chunks),
            (
                "CS-relevant chunks",
                module3_result.cs_relevant_chunks,
            ),
            (
                "Non-CS/no-new-topic chunks",
                module3_result.non_cs_chunks,
            ),
            (
                "Official-topic chunks",
                module3_result.official_topic_chunks,
            ),
            (
                "Mixed official + unmapped chunks",
                module3_result.mixed_official_unmapped_chunks,
            ),
            (
                "Unmapped-CS chunks",
                module3_result.unmapped_cs_chunks,
            ),
            (
                "Continuation chunks",
                module3_result.continuation_chunks,
            ),
            ("No-topic chunks", module3_result.no_topic_chunks),
            (
                "LLM fallback chunks",
                module3_result.llm_fallback_chunk_ids,
            ),
        ],
    )

    primary_topics = [
        topic
        for topic in module3_result.merged_topics
        if topic.topic_role == "primary"
    ]

    supporting_topics = [
        topic
        for topic in module3_result.merged_topics
        if topic.topic_role == "supporting"
    ]

    add_pdf_heading(story, "Primary topics")

    if primary_topics:
        for topic in primary_topics:
            add_pdf_text(
                story,
                f"- {topic.topic}",
            )
    else:
        add_pdf_text(story, "None")

    add_pdf_heading(story, "Supporting topics")

    if supporting_topics:
        for topic in supporting_topics:
            add_pdf_text(
                story,
                f"- {topic.topic}",
            )
    else:
        add_pdf_text(story, "None")

    add_pdf_heading(
        story,
        "Unmapped topics routed to memory/Qdrant/Groq resolution",
    )

    unmapped_names: list[str] = []
    seen_names: set[str] = set()

    for chunk_result in module3_result.chunk_results:
        for signal in chunk_result.unmapped_cs_signals:
            key = signal.rough_topic.casefold()

            if key not in seen_names:
                seen_names.add(key)
                unmapped_names.append(
                    signal.rough_topic
                )

    if unmapped_names:
        for name in unmapped_names:
            add_pdf_text(
                story,
                f"- {name}",
            )
    else:
        add_pdf_text(story, "None")

    add_pdf_heading(
        story,
        "Detailed merged official topics",
    )

    if not module3_result.merged_topics:
        add_pdf_text(
            story,
            "No merged official topics.",
        )

    for topic in module3_result.merged_topics:
        add_pdf_heading(
            story,
            topic.topic,
            level=2,
        )

        add_pdf_key_value_table(
            story,
            [
                ("Concept ID", topic.concept_id),
                ("Domain", topic.domain),
                ("Role", topic.topic_role),
                (
                    "Official reference",
                    topic.official_reference,
                ),
                (
                    "Chapter reference",
                    topic.chapter_reference,
                ),
                ("Confidence", topic.confidence),
                ("Ranking score", topic.ranking_score),
                ("Coverage score", topic.coverage_score),
                ("Source chunks", topic.source_chunk_ids),
                ("Support spans", topic.support_span_count),
                (
                    "Mean semantic score",
                    topic.mean_semantic_score,
                ),
                (
                    "Mean keyword score",
                    topic.mean_keyword_score,
                ),
                (
                    "Mean salience score",
                    topic.mean_salience_score,
                ),
            ],
        )

        if topic.evidence:
            add_pdf_heading(
                story,
                "Evidence",
                level=2,
            )

            for evidence in topic.evidence:
                add_pdf_text(
                    story,
                    f"- {evidence}",
                    small=True,
                )

    story.append(PageBreak())

    add_pdf_heading(
        story,
        "Chunk-by-chunk results",
    )

    for chunk_result in module3_result.chunk_results:
        add_pdf_heading(
            story,
            f"Chunk {chunk_result.chunk_id}",
            level=2,
        )

        add_pdf_key_value_table(
            story,
            [
                (
                    "Classification",
                    chunk_result.classification,
                ),
                (
                    "Source words",
                    chunk_result.source_word_count,
                ),
                (
                    "CS relevant",
                    chunk_result.is_cs_relevant,
                ),
                (
                    "Creates new topic",
                    chunk_result.creates_new_topic,
                ),
                (
                    "CS relevance score",
                    chunk_result.cs_relevance_score,
                ),
                (
                    "Requires memory-first resolution",
                    chunk_result.requires_llm_fallback,
                ),
                (
                    "Notes",
                    chunk_result.notes or "None",
                ),
            ],
        )

        add_pdf_heading(
            story,
            "Retained topics",
            level=2,
        )

        if chunk_result.topic_candidates:
            for candidate in chunk_result.topic_candidates:
                add_pdf_text(
                    story,
                    (
                        f"- {candidate.topic} "
                        f"(confidence={candidate.confidence})"
                    ),
                    small=True,
                )
        else:
            add_pdf_text(story, "None")

        add_pdf_heading(
            story,
            "Unmapped signals",
            level=2,
        )

        if chunk_result.unmapped_cs_signals:
            for signal in chunk_result.unmapped_cs_signals:
                add_pdf_text(
                    story,
                    (
                        f"- {signal.rough_topic} "
                        f"(score={signal.score})"
                    ),
                    small=True,
                )
        else:
            add_pdf_text(story, "None")

    build_pdf(
        output_path,
        f"{transcript_name} - Module 3 Readable Topics",
        story,
    )


## 19. `04_llm_mapping.pdf`


In [ ]:
def create_llm_mapping_pdf(
    transcript_name: str,
    unmapped_inputs: list[UnmappedTopicInput],
    llm_results: list[LLMResolvedTopic],
    output_path: Path,
) -> None:
    """
    Create a source-aware Module 4 report.

    A result is labelled according to whether it came from:
    - an existing Module 3 topic;
    - PostgreSQL mapping memory;
    - a new Groq decision;
    - deterministic handling.
    """

    story: list[Any] = []

    source_labels = {
        "module3": "Existing Module 3 topic",
        "memory": "Mapping memory (PostgreSQL/local)",
        "llm": "Groq LLM",
        "deterministic": "Deterministic rule",
    }

    source_counts = {
        key: sum(
            result.resolution_source == key
            for result in llm_results
        )
        for key in source_labels
    }

    add_pdf_heading(
        story,
        "Module 4 Resolution Summary",
    )

    add_pdf_text(
        story,
        (
            "Every genuine CS-relevant unmapped signal first checks "
            "approved PostgreSQL mapping memory. On a memory miss, Qdrant "
            "provides the official shortlist and Groq proposes a resolution. "
            "New proposals remain pending until human approval promotes them "
            "to reusable mapping memory."
        ),
    )

    add_pdf_key_value_table(
        story,
        [
            ("Groq model", GROQ_MODEL),
            ("Embedding model", TOPIC_EMBEDDING_MODEL),
            ("Qdrant shortlist size", LLM_QDRANT_TOP_K),
            ("Unmapped inputs", len(unmapped_inputs)),
            (
                "Resolved by existing Module 3",
                source_counts["module3"],
            ),
            (
                "Resolved from mapping memory",
                source_counts["memory"],
            ),
            (
                "Resolved by Groq",
                source_counts["llm"],
            ),
            (
                "Resolved deterministically",
                source_counts["deterministic"],
            ),
            (
                "Mapped",
                sum(
                    result.decision == "mapped"
                    for result in llm_results
                ),
            ),
            (
                "Out of syllabus",
                sum(
                    result.decision == "out_of_syllabus"
                    for result in llm_results
                ),
            ),
            (
                "Needs review",
                sum(
                    result.decision == "needs_review"
                    for result in llm_results
                ),
            ),
        ],
    )

    if not llm_results:
        add_pdf_heading(
            story,
            "Resolution Details",
        )
        add_pdf_text(
            story,
            "No Module 4 inputs were produced for this transcript.",
        )

    for index, result in enumerate(
        llm_results,
        start=1,
    ):
        add_pdf_heading(
            story,
            f"Resolution {index}: {result.rough_topic}",
        )

        resolution_source_label = source_labels.get(
            result.resolution_source,
            result.resolution_source,
        )

        confidence_label = (
            "Resolution confidence"
            if result.resolution_source != "llm"
            else "LLM confidence"
        )

        add_pdf_key_value_table(
            story,
            [
                (
                    "Resolution source",
                    resolution_source_label,
                ),
                (
                    "Human-review status",
                    result.review_status or "not required",
                ),
                (
                    "Human-review record ID",
                    result.review_id or "None",
                ),
                ("Final decision", result.decision),
                (
                    "Mapped official topic",
                    result.mapped_topic or "Not mapped",
                ),
                (
                    "Mapped concept ID",
                    result.mapped_concept_id or "None",
                ),
                (
                    "Official reference",
                    result.official_reference or "None",
                ),
                (
                    "Chapter reference",
                    result.chapter_reference or "None",
                ),
                (
                    confidence_label,
                    result.confidence,
                ),
                (
                    "Source chunks",
                    result.source_chunk_ids,
                ),
                ("Reason", result.reason),
            ],
        )

        add_pdf_heading(
            story,
            "Qdrant Candidates",
            level=2,
        )

        if result.qdrant_candidates:
            table_data = [
                [
                    Paragraph(
                        "<b>Rank</b>",
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        "<b>Topic</b>",
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        "<b>Concept ID</b>",
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        "<b>Reference</b>",
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        "<b>Score</b>",
                        PDF_STYLES["AgentSmall"],
                    ),
                ]
            ]

            for rank, candidate in enumerate(
                result.qdrant_candidates,
                start=1,
            ):
                table_data.append(
                    [
                        Paragraph(
                            safe_pdf_text(rank),
                            PDF_STYLES["AgentSmall"],
                        ),
                        Paragraph(
                            safe_pdf_text(
                                candidate.get("label")
                            ),
                            PDF_STYLES["AgentSmall"],
                        ),
                        Paragraph(
                            safe_pdf_text(
                                candidate.get("concept_id")
                            ),
                            PDF_STYLES["AgentSmall"],
                        ),
                        Paragraph(
                            safe_pdf_text(
                                candidate.get(
                                    "official_reference"
                                )
                            ),
                            PDF_STYLES["AgentSmall"],
                        ),
                        Paragraph(
                            safe_pdf_text(
                                candidate.get("qdrant_score")
                            ),
                            PDF_STYLES["AgentSmall"],
                        ),
                    ]
                )

            table = Table(
                table_data,
                colWidths=[
                    12 * mm,
                    48 * mm,
                    58 * mm,
                    25 * mm,
                    20 * mm,
                ],
                repeatRows=1,
            )

            table.setStyle(
                TableStyle(
                    [
                        (
                            "BACKGROUND",
                            (0, 0),
                            (-1, 0),
                            colors.whitesmoke,
                        ),
                        (
                            "GRID",
                            (0, 0),
                            (-1, -1),
                            0.25,
                            colors.lightgrey,
                        ),
                        (
                            "VALIGN",
                            (0, 0),
                            (-1, -1),
                            "TOP",
                        ),
                        (
                            "LEFTPADDING",
                            (0, 0),
                            (-1, -1),
                            3,
                        ),
                        (
                            "RIGHTPADDING",
                            (0, 0),
                            (-1, -1),
                            3,
                        ),
                    ]
                )
            )

            story.append(table)
        else:
            add_pdf_text(
                story,
                "No Qdrant candidates were retained.",
            )

    build_pdf(
        output_path,
        f"{transcript_name} - Module 4 Resolution",
        story,
    )


## 20. `05_final_topic_summary.pdf`


In [ ]:
def create_final_topic_summary_pdf(
    transcript_name: str,
    module3_result: Module3Result,
    unmapped_inputs: list[UnmappedTopicInput],
    llm_results: list[LLMResolvedTopic],
    output_path: Path,
) -> None:
    """
    Create a simple lesson-level summary with accurate resolution sources.
    """

    story: list[Any] = []

    primary_topics = [
        topic
        for topic in module3_result.merged_topics
        if topic.topic_role == "primary"
    ]

    supporting_topics = [
        topic
        for topic in module3_result.merged_topics
        if topic.topic_role == "supporting"
    ]

    mapped_results = [
        result
        for result in llm_results
        if result.decision == "mapped"
    ]

    module3_resolved_results = [
        result
        for result in mapped_results
        if result.resolution_source == "module3"
    ]

    memory_mapped_results = [
        result
        for result in mapped_results
        if result.resolution_source == "memory"
    ]

    groq_mapped_results = [
        result
        for result in mapped_results
        if result.resolution_source == "llm"
    ]

    deterministic_mapped_results = [
        result
        for result in mapped_results
        if result.resolution_source == "deterministic"
    ]

    out_of_syllabus_results = [
        result
        for result in llm_results
        if result.decision == "out_of_syllabus"
    ]

    needs_review_results = [
        result
        for result in llm_results
        if result.decision == "needs_review"
    ]

    pending_review_results = [
        result
        for result in llm_results
        if result.review_status == "pending"
    ]

    approved_or_validated_mapped_results = [
        result
        for result in mapped_results
        if (
            result.resolution_source in {"memory", "module3"}
            or result.review_status in {"approved", "validated"}
        )
    ]

    source_labels = {
        "module3": "Resolved by existing Module 3",
        "memory": "Mapping memory (PostgreSQL/local)",
        "llm": "Mapped by Groq LLM",
        "deterministic": "Deterministic rule",
    }

    add_pdf_heading(
        story,
        "Lesson Topic Summary",
    )

    add_pdf_key_value_table(
        story,
        [
            ("Primary topics", len(primary_topics)),
            ("Supporting topics", len(supporting_topics)),
            (
                "Topics sent to Module 4",
                len(unmapped_inputs),
            ),
            (
                "Pending human-review proposals",
                len(pending_review_results),
            ),
            (
                "Resolved by existing Module 3",
                len(module3_resolved_results),
            ),
            (
                "Mapped from mapping memory",
                len(memory_mapped_results),
            ),
            (
                "Mapped by Groq LLM",
                len(groq_mapped_results),
            ),
            (
                "Mapped deterministically",
                len(deterministic_mapped_results),
            ),
            (
                "Out of syllabus",
                len(out_of_syllabus_results),
            ),
            (
                "Needs review",
                len(needs_review_results),
            ),
            (
                "Module 3 embedding model",
                module3_result.embedding_model,
            ),
            ("Module 4 model", GROQ_MODEL),
        ],
    )

    def add_official_topic_section(
        heading: str,
        topics: list[Any],
        empty_message: str,
    ) -> None:
        add_pdf_heading(
            story,
            heading,
        )

        if not topics:
            add_pdf_text(
                story,
                empty_message,
            )
            return

        for topic in topics:
            add_pdf_heading(
                story,
                topic.topic,
                level=2,
            )

            add_pdf_key_value_table(
                story,
                [
                    (
                        "Official reference",
                        topic.official_reference,
                    ),
                    (
                        "Chapter reference",
                        topic.chapter_reference,
                    ),
                    ("Confidence", topic.confidence),
                    (
                        "Ranking score",
                        topic.ranking_score,
                    ),
                    (
                        "Source chunks",
                        topic.source_chunk_ids,
                    ),
                ],
            )

    add_official_topic_section(
        "Primary Topics",
        primary_topics,
        "No primary topics were identified.",
    )

    add_official_topic_section(
        "Supporting Topics",
        supporting_topics,
        "No supporting topics were identified.",
    )

    add_pdf_heading(
        story,
        "Topics Sent to Module 4",
    )

    if unmapped_inputs:
        for item in unmapped_inputs:
            add_pdf_heading(
                story,
                item.rough_topic,
                level=2,
            )

            add_pdf_key_value_table(
                story,
                [
                    ("Detected domain", item.domain),
                    ("Module 3 score", item.score),
                    (
                        "Detection method",
                        item.detection_method,
                    ),
                    (
                        "Source chunks",
                        item.source_chunk_ids,
                    ),
                ],
            )
    else:
        add_pdf_text(
            story,
            "No Module 3 topic required LLM fallback.",
        )

    add_pdf_heading(
        story,
        "Resolved Topic Mappings",
    )

    if mapped_results:
        for result in mapped_results:
            add_pdf_heading(
                story,
                result.rough_topic,
                level=2,
            )

            source_label = source_labels.get(
                result.resolution_source,
                result.resolution_source,
            )

            confidence_label = (
                "LLM confidence"
                if result.resolution_source == "llm"
                else "Resolution confidence"
            )

            add_pdf_key_value_table(
                story,
                [
                    (
                        "Resolution source",
                        source_label,
                    ),
                    (
                        "Human-review status",
                        result.review_status or "not required",
                    ),
                    (
                        "Human-review record ID",
                        result.review_id or "None",
                    ),
                    (
                        "Original unmapped topic",
                        result.rough_topic,
                    ),
                    (
                        "Mapped official topic",
                        result.mapped_topic,
                    ),
                    (
                        "Mapped concept ID",
                        result.mapped_concept_id,
                    ),
                    (
                        "Official reference",
                        result.official_reference,
                    ),
                    (
                        "Chapter reference",
                        result.chapter_reference,
                    ),
                    (
                        confidence_label,
                        result.confidence,
                    ),
                    (
                        "Source chunks",
                        result.source_chunk_ids,
                    ),
                    ("Reason", result.reason),
                ],
            )
    else:
        add_pdf_text(
            story,
            "No Module 4 topic was mapped to an official concept.",
        )

    add_pdf_heading(
        story,
        "Out-of-Syllabus Topics",
    )

    if out_of_syllabus_results:
        for result in out_of_syllabus_results:
            add_pdf_heading(
                story,
                result.rough_topic,
                level=2,
            )

            add_pdf_key_value_table(
                story,
                [
                    (
                        "Resolution source",
                        source_labels.get(
                            result.resolution_source,
                            result.resolution_source,
                        ),
                    ),
                    (
                        "Detected domain",
                        result.domain,
                    ),
                    (
                        "Resolution confidence",
                        result.confidence,
                    ),
                    (
                        "Source chunks",
                        result.source_chunk_ids,
                    ),
                    ("Reason", result.reason),
                ],
            )
    else:
        add_pdf_text(
            story,
            "No topics were confirmed as outside the AQA syllabus.",
        )

    add_pdf_heading(
        story,
        "Needs Human Review",
    )

    if needs_review_results:
        for result in needs_review_results:
            add_pdf_heading(
                story,
                result.rough_topic,
                level=2,
            )

            add_pdf_key_value_table(
                story,
                [
                    (
                        "Resolution source",
                        source_labels.get(
                            result.resolution_source,
                            result.resolution_source,
                        ),
                    ),
                    (
                        "Detected domain",
                        result.domain,
                    ),
                    (
                        "Resolution confidence",
                        result.confidence,
                    ),
                    (
                        "Source chunks",
                        result.source_chunk_ids,
                    ),
                    ("Reason", result.reason),
                ],
            )
    else:
        add_pdf_text(
            story,
            "No topics require human review.",
        )

    add_pdf_heading(
        story,
        "Final Consolidated Official Topics",
    )

    final_topic_rows: list[
        tuple[str, str, Any]
    ] = []

    seen_concept_ids: set[str] = set()

    for topic in primary_topics:
        if topic.concept_id not in seen_concept_ids:
            seen_concept_ids.add(topic.concept_id)
            final_topic_rows.append(
                (
                    "Primary",
                    topic.topic,
                    topic.official_reference,
                )
            )

    for topic in supporting_topics:
        if topic.concept_id not in seen_concept_ids:
            seen_concept_ids.add(topic.concept_id)
            final_topic_rows.append(
                (
                    "Supporting",
                    topic.topic,
                    topic.official_reference,
                )
            )

    for result in approved_or_validated_mapped_results:
        if (
            result.mapped_concept_id
            and result.mapped_concept_id
            not in seen_concept_ids
        ):
            seen_concept_ids.add(
                result.mapped_concept_id
            )

            final_topic_rows.append(
                (
                    source_labels.get(
                        result.resolution_source,
                        result.resolution_source,
                    ),
                    (
                        result.mapped_topic
                        or "Unknown topic"
                    ),
                    result.official_reference,
                )
            )

    if final_topic_rows:
        table_data = [
            [
                Paragraph(
                    "<b>Source</b>",
                    PDF_STYLES["AgentSmall"],
                ),
                Paragraph(
                    "<b>Final official topic</b>",
                    PDF_STYLES["AgentSmall"],
                ),
                Paragraph(
                    "<b>Official reference</b>",
                    PDF_STYLES["AgentSmall"],
                ),
            ]
        ]

        for source, topic_name, reference in final_topic_rows:
            table_data.append(
                [
                    Paragraph(
                        safe_pdf_text(source),
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        safe_pdf_text(topic_name),
                        PDF_STYLES["AgentSmall"],
                    ),
                    Paragraph(
                        safe_pdf_text(reference),
                        PDF_STYLES["AgentSmall"],
                    ),
                ]
            )

        table = Table(
            table_data,
            colWidths=[
                45 * mm,
                95 * mm,
                30 * mm,
            ],
            repeatRows=1,
        )

        table.setStyle(
            TableStyle(
                [
                    (
                        "BACKGROUND",
                        (0, 0),
                        (-1, 0),
                        colors.whitesmoke,
                    ),
                    (
                        "GRID",
                        (0, 0),
                        (-1, -1),
                        0.25,
                        colors.lightgrey,
                    ),
                    (
                        "VALIGN",
                        (0, 0),
                        (-1, -1),
                        "TOP",
                    ),
                    (
                        "LEFTPADDING",
                        (0, 0),
                        (-1, -1),
                        4,
                    ),
                    (
                        "RIGHTPADDING",
                        (0, 0),
                        (-1, -1),
                        4,
                    ),
                    (
                        "TOPPADDING",
                        (0, 0),
                        (-1, -1),
                        4,
                    ),
                    (
                        "BOTTOMPADDING",
                        (0, 0),
                        (-1, -1),
                        4,
                    ),
                ]
            )
        )

        story.append(table)
    else:
        add_pdf_text(
            story,
            "No final official topics were identified.",
        )

    build_pdf(
        output_path,
        f"{transcript_name} - Final Topic Summary",
        story,
    )


## 21. Existing Module 3 regression tests

These tests were migrated from `scripts/test_module_3_regressions.py`. They use a deterministic offline Qdrant stand-in, while production processing still uses MiniLM and the real Qdrant collection.


In [ ]:
import hashlib
from collections.abc import Sequence

def lexical_test_embeddings(
    texts: Sequence[str],
    model_name: str,
    batch_size: int,
) -> np.ndarray:
    """
    Lightweight deterministic test embedding.

    Production still uses MiniLM. The regression suite avoids model loading
    so it remains fast and repeatable.
    """

    dimension = 512
    rows: list[np.ndarray] = []

    for text in texts:
        vector = np.zeros(dimension, dtype=np.float32)
        normalized = re.sub(r"[^a-z0-9]+", " ", text.lower())
        tokens = normalized.split()

        features = list(tokens)
        features.extend(
            " ".join(tokens[index:index + 2])
            for index in range(max(0, len(tokens) - 1))
        )
        features.extend(
            " ".join(tokens[index:index + 3])
            for index in range(max(0, len(tokens) - 2))
        )

        for feature in features:
            digest = hashlib.md5(feature.encode("utf-8")).digest()
            index = int.from_bytes(digest[:4], "little") % dimension
            vector[index] += 1.0

        # Broad semantic anchors make the fake embedding useful for the
        # unmapped-CS detector while remaining deterministic and offline.
        semantic_groups = (
            {
                "programming", "code", "variable", "function", "method",
                "class", "object", "constructor", "attribute", "private",
                "public", "array", "loop",
            },
            {
                "algorithm", "search", "sorting", "sort", "trace",
                "efficiency", "decomposition", "abstraction",
            },
            {
                "binary", "hexadecimal", "bit", "byte", "encoding",
                "bitmap", "sound", "compression",
            },
            {
                "hardware", "software", "cpu", "memory", "storage",
                "operating", "system", "translator",
            },
            {
                "network", "protocol", "router", "switch", "security",
                "malware", "firewall", "encryption",
            },
            {
                "database", "sql", "table", "record", "field", "query",
            },
        )

        token_set = set(tokens)
        for group_index, group in enumerate(semantic_groups):
            overlap = len(token_set & group)
            if overlap:
                vector[group_index] += 4.0 * overlap

        norm = float(np.linalg.norm(vector))
        if norm > 0.0:
            vector /= norm

        rows.append(vector)

    return np.vstack(rows)



class DeterministicTestSyllabusStore:
    """Offline test double that emulates Qdrant cosine retrieval."""

    def __init__(self) -> None:
        self._concepts = list(syllabus_concepts)
        self._ids = [concept.concept_id for concept in self._concepts]
        self._vectors = lexical_test_embeddings(
            [concept.embedding_text for concept in self._concepts],
            TOPIC_EMBEDDING_MODEL,
            32,
        )
        self._vector_by_id = {
            concept_id: vector
            for concept_id, vector in zip(self._ids, self._vectors, strict=True)
        }

    def search_by_vectors(
        self,
        query_vectors: np.ndarray,
        *,
        top_k: int | None = None,
    ) -> list[list[SemanticConceptMatch]]:
        limit = top_k or 20
        similarities = query_vectors @ self._vectors.T
        results: list[list[SemanticConceptMatch]] = []
        for row in similarities:
            indices = np.argsort(row)[::-1][:limit]
            results.append([
                SemanticConceptMatch(
                    concept_id=self._ids[int(index)],
                    score=float(row[int(index)]),
                    payload={"concept_id": self._ids[int(index)]},
                )
                for index in indices
            ])
        return results

    def retrieve_concept_vectors(
        self,
        concept_ids: list[str],
    ) -> dict[str, np.ndarray]:
        return {
            concept_id: self._vector_by_id[concept_id]
            for concept_id in concept_ids
            if concept_id in self._vector_by_id
        }


def build_test_pipeline() -> Module3TopicPipeline:
    test_store = DeterministicTestSyllabusStore()
    extractor = TopicCandidateExtractor(
        embedding_function=lexical_test_embeddings,
        qdrant_store=test_store,
    )
    unmapped_detector = CSUnmappedDetector(
        embedding_function=lexical_test_embeddings
    )

    return Module3TopicPipeline(
        extractor=extractor,
        relevance_filter=CSRelevanceFilter(),
        unmapped_detector=unmapped_detector,
        merger=TopicMerger(),
    )


def test_pure_social_chunk_is_rejected() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 29,
                "overlap_word_count": 0,
                "text": (
                    "How was your holiday? The weather was cold and I "
                    "stayed in London. My family travelled but I stayed "
                    "at home."
                ),
            }
        ]
    )

    chunk = result.chunk_results[0]
    assert chunk.classification == "no_topic"
    assert not chunk.topic_candidates
    assert not chunk.has_unmapped_cs_content


def test_ambiguous_single_word_does_not_create_bits_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 25,
                "overlap_word_count": 0,
                "text": (
                    "Wait a little bit and let me put this on the paper. "
                    "We will continue in a bit."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_3_3_bits_bytes" not in ids


def test_incidental_integer_does_not_create_data_types_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 35,
                "overlap_word_count": 0,
                "text": (
                    "The code declares integer k equals array zero, then "
                    "uses k while explaining how the values are swapped."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_2_1_data_types" not in ids


def test_mixed_chunk_keeps_official_aqa_topics() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 70,
                "overlap_word_count": 0,
                "text": (
                    "We briefly talked about the holiday. Now look at the "
                    "array. The while loop uses K as the array index and "
                    "continues while K is less than array length. The array "
                    "is checked again in the next sentence."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_2_2_iteration" in ids
    assert "aqa_3_2_6_arrays" in ids


def test_broad_sorting_topic_is_available_without_forcing_bubble_sort() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 55,
                "overlap_word_count": 0,
                "text": (
                    "We have sorting algorithms in the syllabus. This "
                    "example shows sorting by swapping values. We compare "
                    "and swap values so they are arranged in order."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_1_4_sorting_algorithms" in ids


def test_same_evidence_does_not_create_duplicate_subroutine_labels() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 45,
                "overlap_word_count": 0,
                "text": (
                    "The function receives an array as input. This function "
                    "is called once and then returns the result."
                ),
            }
        ]
    )

    subroutine_ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
        if "subroutine" in candidate.topic.lower()
        or "function" in candidate.topic.lower()
    }
    assert len(subroutine_ids) <= 1


def test_overlap_continuation_does_not_create_new_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "The array is checked using a while loop. The array "
                    "index increases while the condition remains true. "
                    "The array is then checked again."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 45,
                "overlap_word_count": 18,
                "text": (
                    "That result is not possible because the earlier "
                    "statement can only execute twice. The answer is two."
                ),
            },
        ]
    )

    second = result.chunk_results[1]
    assert second.classification == "continuation_no_new_topic"
    assert not second.creates_new_topic
    assert not second.is_cs_relevant


def test_unmapped_cs_content_is_flagged_without_fake_aqa_mapping() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 95,
                "overlap_word_count": 0,
                "text": (
                    "The constructor creates the object and initialises its "
                    "attributes. A second constructor accepts a size. The "
                    "private attributes cannot be accessed outside the "
                    "class, while public methods provide controlled access."
                ),
            }
        ]
    )

    chunk = result.chunk_results[0]
    assert chunk.classification in {
        "cs_related_unmapped",
        "mixed_official_and_unmapped",
    }
    assert chunk.has_unmapped_cs_content
    assert chunk.requires_llm_fallback


def test_adjacent_chunks_do_not_inflate_merged_confidence() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 60,
                "overlap_word_count": 0,
                "text": (
                    "The array index is less than the array length. The "
                    "array index is checked again."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 60,
                "overlap_word_count": 10,
                "text": (
                    "The array continues and the array index remains less "
                    "than the array length."
                ),
            },
            {
                "chunk_id": 3,
                "word_count": 60,
                "overlap_word_count": 10,
                "text": (
                    "The same array explanation continues with its array "
                    "index and array length."
                ),
            },
        ]
    )

    arrays = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_2_6_arrays"
    )
    assert arrays.support_span_count == 1
    assert arrays.confidence <= 0.95


def test_catalogue_topics_keep_official_metadata() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 50,
                "overlap_word_count": 0,
                "text": (
                    "Binary search checks the middle item and discards half "
                    "of the sorted data."
                ),
            }
        ]
    )

    binary = next(
        candidate
        for candidate in result.chunk_results[0].topic_candidates
        if candidate.concept_id == "aqa_3_1_3_binary_search"
    )
    assert binary.official_reference == "3.1.3"
    assert binary.chapter_reference == "3.1"
    assert binary.official_title == "Searching algorithms"



def test_arraylist_compound_is_not_mapped_as_official_array() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 45,
                "overlap_word_count": 0,
                "text": (
                    "An ArrayList is a resizable list. The ArrayList can "
                    "grow while the program runs."
                ),
            }
        ]
    )

    chunk = result.chunk_results[0]
    ids = {
        candidate.concept_id
        for candidate in chunk.topic_candidates
    }

    assert "aqa_3_2_6_arrays" not in ids
    assert chunk.has_unmapped_cs_content
    assert any(
        signal.rough_topic == "Dynamic arrays and list collections"
        for signal in chunk.unmapped_cs_signals
    )


def test_real_arrays_and_arraylists_are_classified_separately() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 75,
                "overlap_word_count": 0,
                "text": (
                    "We finished one-dimensional arrays and practised array "
                    "index and array length questions. Next we introduce "
                    "ArrayLists, which are resizable list structures."
                ),
            }
        ]
    )

    chunk = result.chunk_results[0]
    ids = {
        candidate.concept_id
        for candidate in chunk.topic_candidates
    }

    assert "aqa_3_2_6_arrays" in ids
    assert chunk.classification == "mixed_official_and_unmapped"
    assert any(
        signal.rough_topic == "Dynamic arrays and list collections"
        for signal in chunk.unmapped_cs_signals
    )


def test_generic_code_tracing_language_maps_to_official_trace_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 80,
                "overlap_word_count": 0,
                "text": (
                    "Follow the code step by step and track variable values. "
                    "Count statement executions and determine how many times "
                    "the loop runs before the condition becomes false."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_1_algorithm_purpose_trace" in ids


def test_unmapped_signals_return_specific_rough_topics() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 95,
                "overlap_word_count": 0,
                "text": (
                    "The constructor initialises the object. Private "
                    "attributes are not accessible outside the class, and "
                    "public methods provide controlled access."
                ),
            }
        ]
    )

    rough_topics = {
        signal.rough_topic
        for signal in result.chunk_results[0].unmapped_cs_signals
    }

    assert "Object construction and initialisation" in rough_topics
    assert "Encapsulation and access modifiers" in rough_topics


def test_final_ranking_suppresses_keyword_dominated_supporting_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 100,
                "overlap_word_count": 0,
                "text": (
                    "The array is traversed while the loop runs. Follow the "
                    "code step by step and track variable values. The index "
                    "is less than the array length."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 100,
                "overlap_word_count": 0,
                "text": (
                    "Continue the dry run and count statement executions. "
                    "Track variable values through the array while the loop "
                    "runs. One value is greater than another."
                ),
            },
            {
                "chunk_id": 3,
                "word_count": 80,
                "overlap_word_count": 0,
                "text": (
                    "The comparison uses less than and equal to, but the main "
                    "task is to trace the code and count how many times the "
                    "loop runs."
                ),
            },
        ]
    )

    tracing = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_1_1_algorithm_purpose_trace"
    )

    merged_ids = {
        topic.concept_id
        for topic in result.merged_topics
    }

    assert tracing.topic_role == "primary"
    assert "aqa_3_2_4_relational_operations" not in merged_ids



def test_asr_style_execution_counts_map_to_tracing() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 95,
                "overlap_word_count": 0,
                "text": (
                    "The statement is executed four times and the loop ran "
                    "nine times. The index is updated after each pass. We "
                    "must decide whether that execution is possible."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_1_algorithm_purpose_trace" in ids


def test_overlapped_same_tracing_topic_is_continuation() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 100,
                "overlap_word_count": 0,
                "text": (
                    "The statement is executed four times. Follow the code "
                    "and track how the index is updated after every pass."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 70,
                "overlap_word_count": 18,
                "text": (
                    "The loop ran twice, so the statement cannot execute "
                    "five times. That execution count is impossible."
                ),
            },
        ]
    )

    second = result.chunk_results[1]
    assert second.classification == "continuation_no_new_topic"
    assert not second.creates_new_topic
    assert any(
        candidate.concept_id == "aqa_3_1_1_algorithm_purpose_trace"
        for candidate in second.topic_candidates
    )


def test_strong_lexical_unmapped_signal_enters_memory_first_resolution() -> None:
    pipeline = build_test_pipeline()

    signals = [
        UnmappedCSSignal(
            rough_topic="Object construction and initialisation",
            domain="Programming and software development",
            score=0.79,
            evidence="The constructor initialises the object.",
            matched_aliases=["constructor"],
            detection_method="lexical_semantic",
        )
    ]

    assert pipeline._unmapped_requires_llm_fallback(signals)


def test_semantic_only_unmapped_signal_uses_llm() -> None:
    pipeline = build_test_pipeline()

    signals = [
        UnmappedCSSignal(
            rough_topic="Unmapped Computer Science content",
            domain="Programming and software development",
            score=0.74,
            evidence="This technical behaviour is discussed indirectly.",
            matched_aliases=[],
            detection_method="semantic",
        )
    ]

    assert pipeline._unmapped_requires_llm_fallback(signals)


def test_same_evidence_close_unmapped_topics_use_llm() -> None:
    pipeline = build_test_pipeline()

    evidence = "The object is created through a special class routine."
    signals = [
        UnmappedCSSignal(
            rough_topic="Object construction and initialisation",
            domain="Programming and software development",
            score=0.76,
            evidence=evidence,
            matched_aliases=["object creation"],
            detection_method="lexical_semantic",
        ),
        UnmappedCSSignal(
            rough_topic="Class lifecycle management",
            domain="Programming and software development",
            score=0.74,
            evidence=evidence,
            matched_aliases=["class routine"],
            detection_method="lexical_semantic",
        ),
    ]

    assert pipeline._unmapped_requires_llm_fallback(signals)


def test_binary_search_does_not_map_to_number_bases() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 70,
                "overlap_word_count": 0,
                "text": (
                    "Binary search checks the middle value of a sorted list. "
                    "If the target is smaller, it discards the upper half. "
                    "If the target is larger, it discards the lower half."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_3_binary_search" in ids
    assert "aqa_3_3_1_number_bases" not in ids


def test_binary_number_context_does_not_map_to_binary_search() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 70,
                "overlap_word_count": 0,
                "text": (
                    "Binary is base two, decimal is base ten and "
                    "hexadecimal is base sixteen. Each number base uses "
                    "place values, and values can be converted between them."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_3_1_number_bases" in ids
    assert "aqa_3_1_3_binary_search" not in ids


def test_mixed_binary_context_keeps_both_official_topics() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 115,
                "overlap_word_count": 0,
                "text": (
                    "First we revise number bases. Binary is base two, "
                    "decimal is base ten and hexadecimal is base sixteen. "
                    "We then move to searching algorithms. Binary search "
                    "checks the middle item of sorted data and discards "
                    "either the lower half or the upper half."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_3_1_number_bases" in ids
    assert "aqa_3_1_3_binary_search" in ids


def test_incidental_sorting_mention_is_rejected() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 35,
                "overlap_word_count": 0,
                "text": (
                    "We also have sorting algorithms in our syllabus, by "
                    "the way. We will cover them in another lesson."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_4_sorting_algorithms" not in ids


def test_explained_sorting_topic_is_retained() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 80,
                "overlap_word_count": 0,
                "text": (
                    "A sorting algorithm arranges values into order. For "
                    "example, we compare two adjacent values and swap them "
                    "when they are in the wrong order. Let's trace the list "
                    "step by step and explain why each swap happens."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_4_sorting_algorithms" in ids


def test_relational_operator_phrases_need_operator_context() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 70,
                "overlap_word_count": 0,
                "text": (
                    "The while loop continues while k is less than array "
                    "length. We trace the loop and update the array index "
                    "after every pass."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_2_4_relational_operations" not in ids


def test_relational_operators_survive_explicit_teaching_context() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 85,
                "overlap_word_count": 0,
                "text": (
                    "A relational operator compares two values. Less than, "
                    "greater than and equal to are comparison operators. For "
                    "example, x less than y creates a Boolean expression. "
                    "Let's decide whether each condition evaluates to true "
                    "or false."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_2_4_relational_operations" in ids


def test_shared_evidence_prefers_stronger_main_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 95,
                "overlap_word_count": 0,
                "text": (
                    "We trace this while loop step by step. The loop checks "
                    "whether k is less than the array length, then updates k. "
                    "How many times does the statement execute? Explain why "
                    "the loop stops and track every variable value."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }

    assert "aqa_3_1_1_algorithm_purpose_trace" in ids
    assert "aqa_3_2_4_relational_operations" not in ids


def test_high_coverage_topic_is_primary_and_single_chunk_topic_supporting() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "Linear search checks each item in order. For example, "
                    "we start at the first value and compare it with the "
                    "target."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "Continue the linear search by checking the next item. "
                    "Explain why the algorithm stops when the target is "
                    "found."
                ),
            },
            {
                "chunk_id": 3,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "Trace the linear search step by step through this list. "
                    "Count the comparisons made before finding the target."
                ),
            },
            {
                "chunk_id": 4,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "Now compare another linear search example and explain "
                    "how an unsuccessful search reaches the end of the list."
                ),
            },
            {
                "chunk_id": 5,
                "word_count": 75,
                "overlap_word_count": 0,
                "text": (
                    "A for loop repeats a fixed number of times. For example, "
                    "the loop prints each value once."
                ),
            },
        ]
    )

    linear = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_1_3_linear_search"
    )
    iteration = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_2_2_iteration"
    )

    assert linear.coverage_score > iteration.coverage_score
    assert linear.topic_role == "primary"
    assert iteration.topic_role == "supporting"


def test_generic_multi_topic_recap_does_not_create_primary_topics() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 45,
                "overlap_word_count": 0,
                "text": (
                    "Today we covered linear search, binary search, bubble "
                    "sort and arrays. Those were the topics in this lesson."
                ),
            }
        ]
    )

    assert not any(
        topic.topic_role == "primary"
        for topic in result.merged_topics
    )


def test_substantive_topic_survives_separate_generic_recap() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 90,
                "overlap_word_count": 0,
                "text": (
                    "Linear search checks each value in order until the target "
                    "is found. For example, we trace the list step by step and "
                    "count every comparison."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 45,
                "overlap_word_count": 0,
                "text": (
                    "To recap, we covered linear search, arrays, sorting and "
                    "iteration."
                ),
            },
        ]
    )

    linear = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_1_3_linear_search"
    )

    assert linear.topic_role == "primary"

    other_ids = {
        topic.concept_id
        for topic in result.merged_topics
        if topic.concept_id != "aqa_3_1_3_linear_search"
    }

    assert "aqa_3_2_6_arrays" not in other_ids
    assert "aqa_3_2_2_iteration" not in other_ids


def test_brief_comparison_only_binary_search_is_rejected() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 115,
                "overlap_word_count": 0,
                "text": (
                    "Linear search starts at the beginning of the data set "
                    "and checks each item in turn until the target is found. "
                    "It does not require the data to be in order, unlike "
                    "another type of search, binary search. For example, "
                    "we trace a list step by step, compare each value with "
                    "the target, update the index and stop when found is "
                    "true or the end of the list is reached."
                ),
            }
        ]
    )

    retained_ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    rejected_ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].rejected_candidates
    }
    merged_ids = {
        topic.concept_id
        for topic in result.merged_topics
    }

    assert "aqa_3_1_3_linear_search" in retained_ids
    assert "aqa_3_1_3_binary_search" not in retained_ids
    assert "aqa_3_1_3_binary_search" in rejected_ids
    assert "aqa_3_1_3_binary_search" not in merged_ids


def test_detailed_merge_sort_comparison_is_retained() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 180,
                "overlap_word_count": 0,
                "text": (
                    "Bubble sort compares adjacent items and swaps them when "
                    "they are out of order. We trace several passes until no "
                    "more swaps are needed. Now compare merge sort and bubble "
                    "sort. Merge sort compares items from separate lists to "
                    "create new sorted lists. Merge sort is usually quicker "
                    "and is suitable for large data sets. Merge sort is more "
                    "difficult to program and its memory footprint can grow "
                    "while it executes. Bubble sort is slower but easier to "
                    "implement and has a known memory footprint."
                ),
            }
        ]
    )

    retained_ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    merged_ids = {
        topic.concept_id
        for topic in result.merged_topics
    }

    assert "aqa_3_1_4_bubble_sort" in retained_ids
    assert "aqa_3_1_4_merge_sort" in retained_ids
    assert "aqa_3_1_4_bubble_sort" in merged_ids
    assert "aqa_3_1_4_merge_sort" in merged_ids


def test_array_row_column_context_does_not_create_database_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 120,
                "overlap_word_count": 0,
                "text": (
                    "A two dimensional array is visualised as a table. "
                    "One index selects the row and another index selects "
                    "the column. We access each array element using its "
                    "two indexes and trace several examples."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_2_6_arrays" in ids
    assert "aqa_3_7_1_database_structure" not in ids


def test_database_row_column_context_keeps_database_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 110,
                "overlap_word_count": 0,
                "text": (
                    "A relational database stores data in a database table. "
                    "Each row is a record and each column is a database "
                    "field. The primary key uniquely identifies each record."
                ),
            }
        ]
    )

    ids = {
        candidate.concept_id
        for candidate in result.chunk_results[0].topic_candidates
    }
    assert "aqa_3_7_1_database_structure" in ids


def test_continuation_preserves_existing_topic_evidence() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 120,
                "overlap_word_count": 0,
                "text": (
                    "Algorithm efficiency compares algorithms that solve "
                    "the same problem. One algorithm may be faster because "
                    "it executes fewer instructions."
                ),
            },
            {
                "chunk_id": 2,
                "word_count": 110,
                "overlap_word_count": 24,
                "text": (
                    "The same problem is solved again, but this version is "
                    "more efficient because the calculation runs once rather "
                    "than repeating inside a loop."
                ),
            },
        ]
    )

    second = result.chunk_results[1]
    assert second.classification == "continuation_no_new_topic"
    assert not second.creates_new_topic
    assert any(
        candidate.concept_id == "aqa_3_1_2_efficiency"
        for candidate in second.topic_candidates
    )
    efficiency = next(
        topic
        for topic in result.merged_topics
        if topic.concept_id == "aqa_3_1_2_efficiency"
    )
    assert 2 in efficiency.source_chunk_ids


def test_higher_dimensional_arrays_are_extended_not_fake_official_topic() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 130,
                "overlap_word_count": 0,
                "text": (
                    "GCSE covers one dimensional and two dimensional arrays. "
                    "Beyond the specification, a three dimensional array "
                    "uses three indexes and a four dimensional array uses "
                    "four indexes. These higher dimensional arrays continue "
                    "the same indexing idea."
                ),
            }
        ]
    )

    signals = result.chunk_results[0].unmapped_cs_signals
    assert any(
        signal.rough_topic == "Higher-dimensional arrays"
        for signal in signals
    )


def test_complexity_topics_are_preserved_as_extended_content() -> None:
    result = build_test_pipeline().process_chunks(
        [
            {
                "chunk_id": 1,
                "word_count": 170,
                "overlap_word_count": 0,
                "text": (
                    "Algorithm efficiency at GCSE asks which solution is "
                    "quicker. Beyond GCSE, time complexity describes how "
                    "running time grows, while space complexity describes "
                    "the amount of memory required. Big O notation classifies "
                    "constant time and linear time. Intractable problems do "
                    "not run in a practical amount of time as input grows."
                ),
            }
        ]
    )

    rough_topics = {
        signal.rough_topic
        for signal in result.chunk_results[0].unmapped_cs_signals
    }
    assert "Time complexity" in rough_topics
    assert "Space complexity" in rough_topics
    assert "Big O notation" in rough_topics
    assert "Tractable and intractable problems" in rough_topics




def test_module4_collector_includes_every_unmapped_signal() -> None:
    signal = UnmappedCSSignal(
        rough_topic="Object construction and initialisation",
        domain="Programming and software development",
        score=0.79,
        evidence="The constructor initialises the object.",
        matched_aliases=["constructor"],
        detection_method="lexical_semantic",
    )
    chunk = ChunkTopicResult(
        chunk_id=1,
        source_word_count=20,
        classification="cs_related_unmapped",
        is_cs_relevant=True,
        creates_new_topic=True,
        cs_relevance_score=0.79,
        has_unmapped_cs_content=True,
        unmapped_cs_signals=[signal],
        requires_llm_fallback=False,
    )
    result = Module3Result(
        chunk_results=[chunk],
        merged_topics=[],
        total_chunks=1,
        cs_relevant_chunks=1,
        non_cs_chunks=0,
        unmapped_cs_chunks=1,
        embedding_model=TOPIC_EMBEDDING_MODEL,
        candidate_keep_threshold=0.46,
    )
    inputs = collect_unmapped_inputs(
        result,
        [{"chunk_id": 1, "text": "The constructor initialises the object."}],
    )
    assert len(inputs) == 1
    assert inputs[0].rough_topic == "Object construction and initialisation"
    assert inputs[0].source_chunk_ids == [1]


def test_module4_collector_includes_flagged_ambiguous_topic() -> None:
    signal = UnmappedCSSignal(
        rough_topic="Unmapped Computer Science content",
        domain="Programming and software development",
        score=0.70,
        evidence="A technical behaviour is discussed indirectly.",
        matched_aliases=[],
        detection_method="semantic",
    )
    chunk = ChunkTopicResult(
        chunk_id=1,
        source_word_count=20,
        classification="cs_related_unmapped",
        is_cs_relevant=True,
        creates_new_topic=True,
        cs_relevance_score=0.70,
        has_unmapped_cs_content=True,
        unmapped_cs_signals=[signal],
        requires_llm_fallback=True,
    )
    result = Module3Result(
        chunk_results=[chunk],
        merged_topics=[],
        total_chunks=1,
        cs_relevant_chunks=1,
        non_cs_chunks=0,
        unmapped_cs_chunks=1,
        llm_fallback_chunk_ids=[1],
        embedding_model=TOPIC_EMBEDDING_MODEL,
        candidate_keep_threshold=0.46,
    )
    inputs = collect_unmapped_inputs(
        result,
        [{"chunk_id": 1, "text": "A technical behaviour is discussed indirectly."}],
    )
    assert len(inputs) == 1
    assert inputs[0].source_chunk_ids == [1]



def test_llm_resolved_topic_exposes_review_metadata() -> None:
    result = LLMResolvedTopic(
        rough_topic="Object construction and initialisation",
        decision="mapped",
        mapped_concept_id="aqa_3_2_6_arrays",
        mapped_topic="One- and two-dimensional arrays",
        domain="Programming",
        official_reference="3.2.6",
        chapter_reference="3.2",
        confidence=0.82,
        reason="Test proposal.",
        source_chunk_ids=[1],
        qdrant_candidates=[],
        model="test-model",
    )

    assert result.review_id is None
    assert result.review_status is None
    assert result.memory_cache_key is None




import inspect


def test_module4_uses_strict_single_groq_batch_mode() -> None:
    assert GROQ_OMITTED_ITEM_MAX_RETRIES == 0

    process_source = inspect.getsource(
        process_module4_optimized
    )

    assert (
        process_source.count(
            "_call_groq_batch(pending_for_groq)"
        )
        == 1
    )
    assert "_retry_omitted_groq_items(" not in process_source


def test_batch_prompt_requires_one_result_per_item() -> None:
    prompt = BATCH_MAPPING_SYSTEM_PROMPT.casefold()

    assert "exactly one result for every item_id" in prompt
    assert "one json object containing a results array" in prompt
    assert "never invent a concept id" in prompt


def test_batch_prompt_does_not_force_oop_into_aqa_topics() -> None:
    prompt = BATCH_MAPPING_SYSTEM_PROMPT.casefold()

    assert "constructors" in prompt
    assert "encapsulation" in prompt
    assert "private/public access modifiers" in prompt
    assert "out of syllabus" in prompt


def test_batch_prompt_allows_array_equivalent_mapping() -> None:
    prompt = BATCH_MAPPING_SYSTEM_PROMPT.casefold()

    assert "array-equivalent structure" in prompt
    assert "aqa 3.2.6 arrays" in prompt


def test_review_identity_is_transcript_plus_topic() -> None:
    source = inspect.getsource(topic_review_upsert)

    assert "source_transcript = :source_transcript" in source
    assert "normalized_topic = :normalized_topic" in source
    assert "ON CONFLICT DO NOTHING" in source


def test_existing_review_is_checked_before_qdrant() -> None:
    source = inspect.getsource(process_module4_optimized)

    review_position = source.index(
        "topic_review_get_for_transcript_topic"
    )
    qdrant_position = source.index(
        "retrieve_llm_candidates"
    )

    assert review_position < qdrant_position
    assert "existing_review_hits" in source


def test_same_transcript_review_unique_index_exists() -> None:
    source = inspect.getsource(
        ensure_topic_review_schema_and_trigger
    )

    assert "uq_topic_human_review_transcript_topic" in source
    assert "source_transcript" in source
    assert "normalized_topic" in source


def test_generic_unmapped_label_is_not_directly_approvable() -> None:
    item = UnmappedTopicInput(
        rough_topic="Unmapped Computer Science content",
        domain="Networks and cyber security",
        score=0.72,
        evidence="A technical sentence without a specific family label.",
        matched_aliases=[],
        detection_method="semantic",
        source_chunk_ids=[3],
        source_text="A technical sentence without a specific family label.",
        requires_topic_label=True,
    )

    assert item.requires_topic_label
    assert _is_generic_unmapped_label(item.rough_topic)


def test_generic_constructor_evidence_derives_specific_family() -> None:
    derived = _derive_specific_unmapped_family(
        evidence=(
            "The constructor initialises the object and creates "
            "a new class instance."
        ),
        source_text="The class has two constructors.",
    )

    assert derived is not None
    assert derived[0] == "Object construction and initialisation"


def test_process_module4_suppresses_existing_merged_topic_review() -> None:
    source = inspect.getsource(process_module4_optimized)

    assert "duplicate_existing_topic_suppressed" in source
    assert "result.mapped_concept_id in module3_id_set" in source
    assert "[DUPLICATE REVIEW SUPPRESSED]" in source


def test_generic_residual_skips_groq_and_database_review() -> None:
    source = inspect.getsource(process_module4_optimized)

    generic_position = source.index("item.requires_topic_label")
    memory_position = source.index("_lookup_memory_first")

    assert generic_position < memory_position
    assert "was not sent to Groq" in source
    assert "pre_resolved_result" in source

def run_module3_regression_tests() -> pd.DataFrame:
    tests = sorted(
        (value for name, value in globals().items()
         if name.startswith("test_") and callable(value)),
        key=lambda function: function.__name__,
    )
    rows: list[dict[str, str]] = []
    for test in tests:
        try:
            test()
            rows.append({"test": test.__name__, "status": "passed"})
        except Exception as error:
            rows.append({
                "test": test.__name__,
                "status": "failed",
                "error": f"{type(error).__name__}: {error}",
            })
    frame = pd.DataFrame(rows)
    failed = frame[frame["status"] == "failed"]
    display(frame)
    if not failed.empty:
        raise AssertionError(
            f"{len(failed)} Module 3 regression test(s) failed."
        )
    print(f"All {len(frame)} Module 3 regression tests passed.")
    return frame


In [ ]:

module3_test_results = (
    run_module3_regression_tests()
    if RUN_REGRESSION_TESTS
    else pd.DataFrame()
)


## 22. Run Module 3 on all Module 2 JSON files

This cell preserves the existing Module 3 extraction, ranking, merging, and
primary/supporting role logic. It writes the three Module 3/4 PDFs and, in frontend mode,
the additive `03_topic_mapping.json` containing source-aware LLM results and
`topic_review_items`. Module 1 and Module 2 outputs are protected from modification.


In [ ]:

if RUN_MODULE3_BATCH:
    if qdrant_store is None:
        raise RuntimeError("Qdrant must be prepared before the production batch.")

    MODULE3_PRODUCTION_EMBEDDING_FUNCTION = globals().get(
        "MODULE3_PRODUCTION_EMBEDDING_FUNCTION",
        embed_texts,
    )
    topic_pipeline = Module3TopicPipeline(
        extractor=TopicCandidateExtractor(
            embedding_function=MODULE3_PRODUCTION_EMBEDDING_FUNCTION,
            qdrant_store=qdrant_store,
        ),
        unmapped_detector=CSUnmappedDetector(
            embedding_function=MODULE3_PRODUCTION_EMBEDDING_FUNCTION,
        ),
    )

    batch_summary_rows: list[dict[str, Any]] = []
    protected_file_times: dict[Path, int] = {}

    for transcript_name, json_path in MODULE2_INPUTS:
        transcript_output = json_path.parent
        for protected_name in ("01_preprocessing.pdf", "02_chunking.pdf"):
            protected_path = transcript_output / protected_name
            if protected_path.exists():
                protected_file_times[protected_path] = protected_path.stat().st_mtime_ns

    for transcript_index, (transcript_name, json_path) in enumerate(
        MODULE2_INPUTS,
        start=1,
    ):
        started = time.perf_counter()
        transcript_output = json_path.parent

        topic_label_overrides: dict[str, Any] = {}
        topic_label_override_path = (
            transcript_output / "topic_label_overrides.json"
        )
        if topic_label_override_path.is_file():
            try:
                raw_override_payload = json.loads(
                    topic_label_override_path.read_text(encoding="utf-8")
                )
                raw_overrides = raw_override_payload.get("overrides", {})
                if isinstance(raw_overrides, dict):
                    topic_label_overrides = raw_overrides
            except (OSError, json.JSONDecodeError) as override_error:
                print(
                    "WARNING: could not load topic-label overrides:",
                    type(override_error).__name__,
                    override_error,
                )

        print("=" * 100)
        print(f"[{transcript_index}/{len(MODULE2_INPUTS)}] {transcript_name}")
        print("Input:", json_path)

        try:
            chunks_for_module3, module2_metadata = load_module2_chunks(json_path)
            module3_result = topic_pipeline.process_chunks(chunks_for_module3)

            # Human-corrected mapping memory must also override topics that
            # Module 3 retained deterministically. This is deliberately
            # post-extraction: the existing extractor/ranker is unchanged.
            module3_result = _apply_reviewer_memory_to_retained_topics(
                module3_result=module3_result,
                chunks_for_module3=chunks_for_module3,
                transcript_name=transcript_name,
            )

            expected_fallback_ids = set(module3_result.llm_fallback_chunk_ids)
            unmapped_inputs, llm_results, optimization_stats = process_module4_optimized(
                transcript_name=transcript_name,
                module3_result=module3_result,
                chunks_for_module3=chunks_for_module3,
                topic_label_overrides=topic_label_overrides,
            )
            topic_review_items = topic_review_list_for_transcript(
                transcript_name
            )
            collected_fallback_ids = {
                chunk_id
                for item in unmapped_inputs
                for chunk_id in item.source_chunk_ids
            }

            if not collected_fallback_ids.issubset(expected_fallback_ids):
                raise AssertionError(
                    "Module 4 collected a chunk that Module 3 did not flag "
                    "for LLM fallback."
                )

            output_paths = {
                "topics": transcript_output / "03_topics_readable.pdf",
                "llm": transcript_output / "04_llm_mapping.pdf",
                "summary": transcript_output / "05_final_topic_summary.pdf",
            }

            if OVERWRITE_MODULE3_PDFS:
                for output_path in output_paths.values():
                    output_path.unlink(missing_ok=True)

            create_topics_pdf(
                transcript_name=transcript_name,
                module3_result=module3_result,
                output_path=output_paths["topics"],
            )
            create_llm_mapping_pdf(
                transcript_name=transcript_name,
                unmapped_inputs=unmapped_inputs,
                llm_results=llm_results,
                output_path=output_paths["llm"],
            )
            create_final_topic_summary_pdf(
                transcript_name=transcript_name,
                module3_result=module3_result,
                unmapped_inputs=unmapped_inputs,
                llm_results=llm_results,
                output_path=output_paths["summary"],
            )

            if FRONTEND_SINGLE_FILE_MODE:
                # Additive JSON used by Streamlit to render topic tables.
                # It serialises the existing in-memory results without
                # changing extraction, mapping, ranking, or LLM decisions.
                topic_json_path = (
                    transcript_output
                    / "03_topic_mapping.json"
                )
                topic_json_payload = {
                    "schema_version": "1.0",
                    "module": "agent_1_module_3_topic_mapping",
                    "transcript": transcript_name,
                    "module2_metadata": module2_metadata,
                    "module3_result": to_plain_data(module3_result),
                    "unmapped_inputs": to_plain_data(unmapped_inputs),
                    "llm_results": to_plain_data(llm_results),
                    "topic_review_items": to_plain_data(
                        topic_review_items
                    ),
                    "topic_label_overrides": to_plain_data(
                        topic_label_overrides
                    ),
                    "optimization_stats": to_plain_data(
                        optimization_stats
                    ),
                }
                topic_json_path.write_text(
                    json.dumps(
                        topic_json_payload,
                        indent=2,
                        ensure_ascii=False,
                    ),
                    encoding="utf-8",
                )
                output_paths["json"] = topic_json_path

            missing_outputs = [
                str(path) for path in output_paths.values()
                if not path.is_file() or path.stat().st_size == 0
            ]
            if missing_outputs:
                raise RuntimeError(f"Missing/empty PDF outputs: {missing_outputs}")

            elapsed = time.perf_counter() - started
            batch_summary_rows.append({
                "transcript": transcript_name,
                "status": "completed",
                "module2_json": str(json_path),
                "module2_embedding_model": module2_metadata.get("embedding_model"),
                "chunks": len(chunks_for_module3),
                "module3_topics": len(module3_result.merged_topics),
                "llm_fallback_chunks": len(expected_fallback_ids),
                **optimization_stats,
                "mapped_total": sum(result.decision == "mapped" for result in llm_results),
                "mapped_by_groq": sum(result.decision == "mapped" and result.resolution_source == "llm" for result in llm_results),
                "mapped_from_memory": sum(result.decision == "mapped" and result.resolution_source == "memory" for result in llm_results),
                "mapped_from_module3": sum(result.decision == "mapped" and result.resolution_source == "module3" for result in llm_results),
                "pending_topic_reviews": sum(item.get("status") == "pending" for item in topic_review_items),
                "approved_topic_reviews": sum(item.get("status") == "approved" for item in topic_review_items),
                "rejected_topic_reviews": sum(item.get("status") == "rejected" for item in topic_review_items),
                "out_of_syllabus": sum(result.decision == "out_of_syllabus" for result in llm_results),
                "needs_review": sum(result.decision == "needs_review" for result in llm_results),
                "seconds": round(elapsed, 3),
                "output_folder": str(transcript_output),
            })
            print("Completed:", transcript_output)

        except Exception as error:
            elapsed = time.perf_counter() - started
            batch_summary_rows.append({
                "transcript": transcript_name,
                "status": "failed",
                "module2_json": str(json_path),
                "chunks": 0,
                "module3_topics": 0,
                "llm_fallback_chunks": 0,
                "unmapped_inputs": 0,
                "memory_hits": 0,
                "resolved_by_module3": 0,
                "groq_calls": 0,
                "groq_topics": 0,
                "groq_retry_calls": 0,
                "groq_retried_item_attempts": 0,
                "groq_items_still_omitted": 0,
                "memory_writes": 0,
                "review_writes": 0,
                "seconds": round(elapsed, 3),
                "output_folder": str(transcript_output),
                "error": f"{type(error).__name__}: {error}",
            })
            print("FAILED:", type(error).__name__, error)

    batch_summary_df = pd.DataFrame(batch_summary_rows)
    display(batch_summary_df)

    failed_rows = batch_summary_df[batch_summary_df["status"] != "completed"]
    if not failed_rows.empty:
        raise RuntimeError(
            f"{len(failed_rows)} transcript(s) failed Module 3 processing."
        )

    for protected_path, original_mtime in protected_file_times.items():
        if not protected_path.exists():
            raise AssertionError(f"Protected file was removed: {protected_path}")
        if protected_path.stat().st_mtime_ns != original_mtime:
            raise AssertionError(
                f"Module 1/2 output was modified unexpectedly: {protected_path}"
            )

    print("Verified: Module 1 and Module 2 PDFs were not recreated or modified.")
else:
    batch_summary_df = pd.DataFrame()
    print("Production batch skipped for offline notebook validation.")


## 23. Save Module 3 batch summary and verify all outputs


In [ ]:

if RUN_MODULE3_BATCH:
    summary_csv = OUTPUT_DIR / "module3_batch_summary.csv"
    summary_json = OUTPUT_DIR / "module3_batch_summary.json"

    batch_summary_df.to_csv(summary_csv, index=False)
    summary_json.write_text(
        json.dumps(
            {
                "input_type": "OUTPUT/<transcript>/02_chunking.json",
                "expected_transcripts": EXPECTED_TRANSCRIPT_COUNT,
                "excluded_folders": sorted(EXCLUDED_TRANSCRIPT_FOLDERS),
                "outputs_per_transcript": [
                    "03_topics_readable.pdf",
                    "04_llm_mapping.pdf",
                    "05_final_topic_summary.pdf",
                ],
                "rows": batch_summary_rows,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    verification_rows = []
    for transcript_name, json_path in MODULE2_INPUTS:
        folder = json_path.parent
        row = {"transcript": transcript_name, "02_chunking.json": json_path.is_file()}
        for name in (
            "03_topics_readable.pdf",
            "04_llm_mapping.pdf",
            "05_final_topic_summary.pdf",
        ):
            row[name] = (folder / name).is_file() and (folder / name).stat().st_size > 0

        if FRONTEND_SINGLE_FILE_MODE:
            json_output = folder / "03_topic_mapping.json"
            row["03_topic_mapping.json"] = (
                json_output.is_file()
                and json_output.stat().st_size > 0
            )

        verification_rows.append(row)

    verification_df = pd.DataFrame(verification_rows)
    display(verification_df)

    output_columns = [
        "03_topics_readable.pdf",
        "04_llm_mapping.pdf",
        "05_final_topic_summary.pdf",
    ]
    if FRONTEND_SINGLE_FILE_MODE:
        output_columns.append("03_topic_mapping.json")

    if len(verification_df) != EXPECTED_TRANSCRIPT_COUNT:
        if FRONTEND_SINGLE_FILE_MODE:
            raise AssertionError(
                "Frontend verification did not cover the uploaded transcript."
            )
        raise AssertionError("Verification did not cover all 12 transcripts.")

    if not verification_df[output_columns].all().all():
        if FRONTEND_SINGLE_FILE_MODE:
            raise AssertionError(
                "At least one frontend Module 3 output is missing."
            )
        raise AssertionError("At least one Module 3 PDF output is missing.")

    if FRONTEND_SINGLE_FILE_MODE:
        frontend_manifest = {
            "module": "module_3",
            "status": (
                "completed"
                if (
                    not batch_summary_df.empty
                    and (
                        batch_summary_df["status"]
                        == "completed"
                    ).all()
                )
                else "failed"
            ),
            "transcript_name": FRONTEND_TRANSCRIPT_NAME,
            "output_root": str(OUTPUT_DIR),
            "rows": batch_summary_rows,
        }
        (
            OUTPUT_DIR
            / "module3_frontend_manifest.json"
        ).write_text(
            json.dumps(
                frontend_manifest,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        print("Frontend Module 3 transcript was processed.")
    else:
        print("All 12 Module 2 JSON inputs were processed.")

    print("Summary CSV:", summary_csv)
    print("Summary JSON:", summary_json)
else:
    print("Output verification skipped with the production batch.")


# Final production decision

```text
INPUT
OUTPUT/<transcript>/02_chunking.json

MODULE 3 — UNCHANGED CORE LOGIC
MiniLM + Qdrant official AQA retrieval
+ lexical/context/salience/evidence-quality rules
+ non-CS, continuation, and unmapped-CS handling
+ existing topic merging and primary/supporting assignment

RESTORED RESOLUTION ARCHITECTURE
Every genuine unmapped CS topic
-> Qdrant official shortlist
-> PostgreSQL topic_mapping_memory lookup
-> memory miss: exactly one transcript-level Groq batch call
-> one JSON results array validated item-by-item
-> proposals saved to topic_human_review (pending)
-> user selects Approve
-> PostgreSQL trigger upserts human-corrected result into topic_mapping_memory
-> next matching run returns resolution_source='memory'

OUTPUT
OUTPUT/<transcript>/03_topics_readable.pdf
OUTPUT/<transcript>/04_llm_mapping.pdf
OUTPUT/<transcript>/05_final_topic_summary.pdf
OUTPUT/<transcript>/03_topic_mapping.json  (frontend mode)
```

The notebook remains self-contained. Existing topic extraction, mapping candidates,
ranking, merging, evidence evaluation, regression coverage, and primary/supporting role
assignment are preserved. Only unmapped-topic routing and human-approved memory
persistence are restored.


## Strict AQA mapping boundary

- Dynamic/resizable list collections may map to **3.2.6 arrays** when the
  evidence uses them as an array-equivalent collection.
- Encapsulation, private/public access modifiers, classes, constructors, and
  object initialisation are **not explicit AQA 8525 examinable topics**.
  They remain `out_of_syllabus`, while the explanation records the closest
  related AQA programming area when useful.
- Broad chapter similarity never forces an official mapping.


## Repeated transcript-run deduplication

Human-review identity is now:

```text
(source_transcript, normalized_topic)
```

Therefore:

- running the same transcript repeatedly does not insert another review row;
- an existing pending, approved, or rejected row is reused before Qdrant/Groq;
- a genuinely new rough topic in the same transcript creates a new row;
- a different transcript may create its own review row;
- existing accidental duplicates are cleaned on schema initialisation;
- an approved row is retained first during cleanup, otherwise the earliest row
  is retained.


## Generic semantic-residual guard

- `Unmapped Computer Science content` is never sent directly for approval.
- The evidence is first matched against a known specific unmapped family.
- If no safe specific family can be derived, the result is
  `needs_review` with no Groq call and no `topic_human_review` row.
- If Groq proposes an official concept already present in `merged_topics`,
  the duplicate proposal is suppressed and no new review row is inserted.
